# Case Study Group 44 - Factory Audit Recommendation

**Course:** Introduction to Engineering Data Analytics  
**Semester:** Summer Semester 2026  
**Group number:** 44  
**Case study scope:** OEM1 Vehicle Type 11 and OEM2 Vehicle Type 21

# Table of Contents

1. [Project Information](#section-1)
2. [Business Context, Task, and Analytical Strategy](#section-2)
3. [Data Scope and Data Model](#section-3)
4. [Project Setup and Initial Raw-Data Inspection](#section-4)
5. [Exploratory Data Analysis](#section-5)
6. [Data Cleaning and Preparation](#section-6)
7. [Data Integration](#section-7)
8. [Creation and Validation of the Final Dataset](#section-8)
9. [Decision Criteria](#section-9)
10. [OEM Plant Comparison](#section-10)
11. [Component Defects at Supplier Level](#section-11)
12. [Audit Recommendation](#section-12)

<a id="section-1"></a>

# 1. Project Information

## 1.1 Group Members

| Name | Matr.-Nr. |
|---|---|
| Asmarani Syifa Syakira | `463533` |
| Yannick Lennartz Orth | `379719` |
| Patrick Rodriguez Granda | `652537` Humboldt Universität |
| Group member 4 | `[TO BE ADDED]` |
| Group member 5 | `[TO BE ADDED]` |

## 1.2 Purpose of This Notebook

The notebook serves as both the analytical workflow and the written case study report. It contains all code required to reproduce the analysis from the original source files, together with explanations of the analytical decisions, validation steps, visualisations, and interpretations of the results.

## 1.3 Project Deliverables

The final case study submission consists of the following files:

- `SoSe26_Case_Study_Group_44.ipynb`: executable analysis and report,
- `SoSe26_Case_Study_Group_44.html`: static export with all final outputs,
- `SoSe26_Case_Study_finalData_Group_44.csv`: final processed dataset used by the application, and
- `SoSe26_Case_Study_App_Group_44.py`: interactive web application.

All project paths are defined relative to the submission folder so that the complete workflow can be run on another computer without modifying computer-specific paths.

<a id="section-2"></a>

# 2. Business Context, Task, and Strategy

## 2.1 Business Context

The automotive group manufactures several vehicle types under the brands OEM1 and OEM2. Its supply chain comprises Tier-2 suppliers that manufacture individual parts, Tier-1 suppliers that assemble these parts into components, and OEM production plants that install the components in finished vehicles. A quality issue recorded at any of these levels can therefore affect the quality status of a completed vehicle.

We must decide where the next process audit should take place. The decision should reflect both the number of affected vehicles in the field and the defect frequency relative to the production volume of each plant.

## 2.2 Assigned Task Group 44

> Analyse the failure data for OEM1 Vehicle Type 11 and OEM2 Vehicle Type 21. Relate the absolute and relative defect frequencies of these vehicles to their production plants and recommend the plant in which the next process audit should be conducted. Identify additional insights that can support the audit.

## 2.3 Objective

The primary objective is to identify the OEM production plant with the greatest audit priority. The main decision criterion is the relative frequency of defective vehicles because it relates observed defects to the number of vehicles produced at each plant. Absolute defect counts are analysed alongside the relative rates mainly to represent customer impact.

The analysis also examines defect sources at component and individual-part level. These supplier-level results are used to refine the proposed audit scope, but they do not replace the OEM plant comparison required by the task.

## 2.4 Defect Definition

Three related defect flags are distinguished:

- **Vehicle defect:** the defect flag recorded directly in the vehicle production table.
- **Component defect:** a defect flag recorded for at least one component installed in the vehicle.
- **Individual-part defect:** a defect flag recorded for at least one part contained in an installed component.

A vehicle is classified as defective when at least one of these indicators confirms a defect. This definition sets the rule that defects propagate through the supply chain from individual parts to components and from components to the finished vehicle. Missing defect information remains explicitly unknown during data preparation and is examined before the final rates are interpreted.

## 2.5 Questions

The analysis is organised around five questions:

1. How many defective vehicles are associated with each OEM production plant?
2. What proportion of the vehicles produced at each plant is effectively defective?
3. Does the plant ranking change when only direct vehicle defects are considered instead of the complete supply-chain definition?
4. Which component roles, component variants, and Tier-1 supplier plants contribute the most to vehicle defects, and how often does the aggregated part status affect the component result?
5. How do the observed defect patterns change over time, and which limitations must be considered when translating them into an audit recommendation?

## 2.6 Strategy

The workflow follows the logic of the business case. First, the relevant vehicle, component, part, mapping, and plant tables are identified. Second, the raw data are imported and examined before major transformations. Third, recurring quality problems are cleaned and the supply-chain tables are integrated using validated keys. Then, one final dataset is created on a unique vehicle level using the defined defect-propagation rule. Finally, the OEM plants are compared using absolute counts and relative rates.

<a id="section-3"></a>

# 3. Available Data

## 3.1 Scope of the Analysis

The required analysis consists of **OEM1 Vehicle Type 11** and **OEM2 Vehicle Type 21**. Other vehicle types, registrations, and logistics-delay tables are outside the scope because they are not required to answer the assigned plant-audit question. Geodata are retained for the OEM plants. For Tier-1 suppliers, the plant and city are retained for the component analysis. Tier-2 plant numbers are used to validate the part records but are not copied to the final Vehicle-Level dataset.

The unit of analysis for the final OEM comparison is one finished vehicle. Supplier tables contain lower-level observations and are integrated only to determine whether an installed component or individual part changes the effective defect status of that vehicle.

## 3.2 Data-Selection Strategy

Data selection starts with the two required vehicle master files and their vehicle-component mapping files. The component identifiers in these mappings determine the relevant body, transmission, seat, and engine variants. The component-part mappings then determine which individual-part types belong to those components. Finally, the OEM, Tier-1, and Tier-2 plant tables provide a common location reference for the three supply-chain levels.

The resulting inventory is reported below so that the import is reproducible.

The selected vehicle files lead to three OEM plant candidates for the audit:

| OEM plant | City | Vehicle type |
|---|---|---|
| O11 | Nürnberg | Type 11 |
| O12 | Bonn | Type 11 |
| O21 | Göttingen | Type 21 |

O11 and O12 can be compared within the same vehicle type. O21 produces Type 21, so its result can be used to set an audit priority but not to separate a plant effect from a vehicle-type effect.

## 3.3 Supply-Chain Data

The data has the following structure:

**Individual part -> component -> vehicle -> OEM production plant**

| Data level | Single unit (Row) | Primary key (Column) | Purpose |
|---|---|---|---|
| Vehicle | One finished Type 11 or Type 21 vehicle | `ID_Fahrzeug` | Defines the final unit of analysis, OEM plant, production date, and direct vehicle-defect status |
| Vehicle-component mapping | One vehicle with four component ID columns | `ID_Fahrzeug` | Connects each vehicle to body, transmission, seats, and engine |
| Component | One component | `ID_Komponente` | Provides component type, Tier-1 plant, and component-defect status |
| Component-part mapping | One component with several part ID columns | `ID_Komponente` | Connects a component to its parts |
| Individual part | One individual part | `ID_Einzelteil` | Provides the part-defect status used for the bottom up aggregation |
| Plant | One OEM, Tier-1, or Tier-2 production location | `Werksnummer` | Adds plant information or validates the production-plant key |

## 3.4 Relationships Between Data

| Relationship | Merge key | Expected relationship | Validation objective |
|---|---|---|---|
| Vehicle data -> wide vehicle-component mapping | `ID_Fahrzeug` | One-to-one | Every vehicle should have one mapping row with four component IDs |
| Each component ID column -> component data | `ID_Komponente` | Many-to-one | Every installed component should have one unique component record |
| Component-part assignments -> part data | `ID_Einzelteil` | Many-to-one | Every installed part ID should have one part-quality record |
| Aggregated part result -> component data | `ID_Komponente` | One-to-one | Every component should receive one aggregated part status |
| Vehicles and components -> plant data | `Werksnummer` | Many-to-one | Every production object should match the correct plant reference row |

## 3.5 Assumptions and Exclusions

- The structured object IDs (e.g. `ID_Fahrzeug`, `ID_Komponente`, etc.) are used to verify manufacturer and plant numbers when separately stored values are missing or inconsistent.
- Only component and part variants that occur in the mappings of Vehicle Types 11 and 21 are relevant to the analysis.
- Registration and logistics-delay data are excluded because the assigned task asks for production-plant comparisons based on vehicle defects, not registration or logistics performance.
- Supplier-level defect patterns provide additional information for the audit scope, but they are not the primary criterion for selecting the OEM plant.

## 3.6 Relevant Source Files

These are the source files required to reconstruct the complete vehicle-component-part quality chain for both vehicle types. These relationships have been found out by looking at the available data manually and creating an overview that can be found in the additional submission files.

### 3.6.1 Relevant Files for Vehicle Type 11

| Category | Filename |
|---|---|
| Vehicle data | `Fahrzeuge_OEM1_Typ11.csv` |
| Vehicle–component mapping | `Bestandteile_Fahrzeuge_OEM1_Typ11.csv` |
| OEM plant locations | `OEM_Werke_2017-07-04_TrR.csv` |
| Tier 1 plant locations | `Tier1_Werke_2017-07-11_v1.2_TrR.csv` |
| Tier 2 plant locations | `Tier2_Werke_2017-07-11_v1.2_TrR.csv` |
| Body component | `Komponente_K4.csv` |
| Body component–part mapping | `Bestandteile_Komponente_K4.csv` |
| Body part | `Einzelteil_T30.csv` |
| Body part | `Einzelteil_T31.txt` |
| Body part | `Einzelteil_T32.csv` |
| Transmission component | `Komponente_K3AG1.csv` |
| Transmission component | `Komponente_K3SG1.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3AG1.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3SG1.csv` |
| Transmission part | `Einzelteil_T21.csv` |
| Transmission part | `Einzelteil_T22.txt` |
| Transmission part | `Einzelteil_T23.csv` |
| Transmission part | `Einzelteil_T24.txt` |
| Transmission part | `Einzelteil_T25.csv` |
| Seat component | `Komponente_K2LE1.txt` |
| Seat component | `Komponente_K2ST1.txt` |
| Seat component–part mapping | `Bestandteile_Komponente_K2LE1.csv` |
| Seat component–part mapping | `Bestandteile_Komponente_K2ST1.csv` |
| Seat part | `Einzelteil_T11.txt` |
| Seat part | `Einzelteil_T12.csv` |
| Seat part | `Einzelteil_T13.csv` |
| Seat part | `Einzelteil_T14.csv` |
| Seat part | `Einzelteil_T15.csv` |
| Engine component | `Komponente_K1BE1.csv` |
| Engine component | `Komponente_K1DI1.csv` |
| Engine component–part mapping | `Bestandteile_Komponente_K1BE1.csv` |
| Engine component–part mapping | `Bestandteile_Komponente_K1DI1.csv` |
| Engine part | `Einzelteil_T01.txt` |
| Engine part | `Einzelteil_T02.txt` |
| Engine part | `Einzelteil_T03.txt` |
| Engine part | `Einzelteil_T04.csv` |
| Engine part | `Einzelteil_T05.csv` |
| Engine part | `Einzelteil_T06.csv` |

### 3.6.2 Relevant Files for Vehicle Type 21

| Category | Filename |
|---|---|
| Vehicle data | `Fahrzeuge_OEM2_Typ21.csv` |
| Vehicle–component mapping | `Bestandteile_Fahrzeuge_OEM2_Typ21.csv` |
| OEM plant locations | `OEM_Werke_2017-07-04_TrR.csv` |
| Tier 1 plant locations | `Tier1_Werke_2017-07-11_v1.2_TrR.csv` |
| Tier 2 plant locations | `Tier2_Werke_2017-07-11_v1.2_TrR.csv` |
| Body component | `Komponente_K6.csv` |
| Body component–part mapping | `Bestandteile_Komponente_K6.csv` |
| Body part | `Einzelteil_T34.txt` |
| Body part | `Einzelteil_T35.txt` |
| Body part | `Einzelteil_T36.txt` |
| Body part | `Einzelteil_T37.csv` |
| Transmission component | `Komponente_K3AG2.txt` |
| Transmission component | `Komponente_K3SG2.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3AG2.csv` |
| Transmission component–part mapping | `Bestandteile_Komponente_K3SG2.csv` |
| Transmission part | `Einzelteil_T21.csv` |
| Transmission part | `Einzelteil_T22.txt` |
| Transmission part | `Einzelteil_T24.txt` |
| Transmission part | `Einzelteil_T26.csv` |
| Transmission part | `Einzelteil_T27.txt` |
| Seat component | `Komponente_K2LE2.txt` |
| Seat component | `Komponente_K2ST2.csv` |
| Seat component–part mapping | `Bestandteile_Komponente_K2LE2.csv` |
| Seat component–part mapping | `Bestandteile_Komponente_K2ST2.csv` |
| Seat part | `Einzelteil_T16.txt` |
| Seat part | `Einzelteil_T17.csv` |
| Seat part | `Einzelteil_T18.csv` |
| Seat part | `Einzelteil_T19.csv` |
| Seat part | `Einzelteil_T20.txt` |
| Engine component | `Komponente_K1BE2.csv` |
| Engine component | `Komponente_K1DI2.txt` |
| Engine component–part mapping | `Bestandteile_Komponente_K1BE2.csv` |
| Engine component–part mapping | `Bestandteile_Komponente_K1DI2.csv` |
| Engine part | `Einzelteil_T01.txt` |
| Engine part | `Einzelteil_T02.txt` |
| Engine part | `Einzelteil_T07.txt` |
| Engine part | `Einzelteil_T08.csv` |
| Engine part | `Einzelteil_T09.txt` |
| Engine part | `Einzelteil_T10.csv` |

<a id="section-4"></a>

# 4. Project Setup and Initial Raw Data Inspection

The manual data-selection strategy, including why these files are needed and why the remaining files are excluded, is explained in Section 3.2. The required files are then listed in Section 3.6. Before any cleaning, we:

1. define all paths relative to the project folder,
2. check that every required file is actually present,
3. open a few representative raw files to see typical formatting problems.

The goal is to understand the raw data as it currently exists and to identify the import and cleaning steps needed later.


## 4.1 Imports

`pathlib` keeps file paths independent of the operating system. `pandas` is used for all tables. `csv`, `re`, and `mmap` are needed later for a few TXT files that do not use a normal delimiter or line break. `numpy` is used for everything numeric.


In [1]:
from pathlib import Path
import csv
import mmap
import re

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 100)


## 4.2 Project and Data Directories

All paths start from the current project folder.

The helper function `check_paths` is reused for every file group below. It shows whether a listed path exists without opening the file.


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIRECTORY = PROJECT_ROOT / "Data"

VEHICLE_DIRECTORY = DATA_DIRECTORY / "Fahrzeug"
COMPONENT_DIRECTORY = DATA_DIRECTORY / "Komponente"
PART_DIRECTORY = DATA_DIRECTORY / "Einzelteil"
GEODATA_DIRECTORY = DATA_DIRECTORY / "Geodaten"


def check_paths(path_dict, expect_file=True):
    """Show whether each listed path exists."""
    rows = []
    for name, path in path_dict.items():
        exists = path.is_file() if expect_file else path.is_dir()
        rows.append(
            {
                "File": name,
                "Relative Path": path.relative_to(PROJECT_ROOT).as_posix(),
                "Exists": "Yes" if exists else "No",
            }
        )
    return pd.DataFrame(rows)


data_directories = {
    "Vehicle data": VEHICLE_DIRECTORY,
    "Component data": COMPONENT_DIRECTORY,
    "Part data": PART_DIRECTORY,
    "Plant data": GEODATA_DIRECTORY,
}

check_paths(data_directories, expect_file=False)


,File,Relative Path,Exists
0,Vehicle data,Data/Fahrzeug,Yes
1,Component data,Data/Komponente,Yes
2,Part data,Data/Einzelteil,Yes
3,Plant data,Data/Geodaten,Yes


All four data folders are available. The same overview is used for every file group below.

## 4.3 Vehicle Files

Each vehicle type has one vehicle file and one mapping file that links the vehicle to its four installed components.


In [3]:
vehicle_paths = {
    "Vehicle Type 11": VEHICLE_DIRECTORY / "Fahrzeuge_OEM1_Typ11.csv",
    "Vehicle Type 11 mapping": VEHICLE_DIRECTORY / "Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    "Vehicle Type 21": VEHICLE_DIRECTORY / "Fahrzeuge_OEM2_Typ21.csv",
    "Vehicle Type 21 mapping": VEHICLE_DIRECTORY / "Bestandteile_Fahrzeuge_OEM2_Typ21.csv",
}

## 4.4 Component Files

For each component we need two files: the production file (quality and plant of the component) and the mapping file (which parts are installed in it).


In [4]:
component_paths = {
    "K4 production": COMPONENT_DIRECTORY / "Komponente_K4.csv",
    "K4 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K4.csv",
    "K3AG1 production": COMPONENT_DIRECTORY / "Komponente_K3AG1.csv",
    "K3AG1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3AG1.csv",
    "K3SG1 production": COMPONENT_DIRECTORY / "Komponente_K3SG1.csv",
    "K3SG1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3SG1.csv",
    "K2LE1 production": COMPONENT_DIRECTORY / "Komponente_K2LE1.txt",
    "K2LE1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2LE1.csv",
    "K2ST1 production": COMPONENT_DIRECTORY / "Komponente_K2ST1.txt",
    "K2ST1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2ST1.csv",
    "K1BE1 production": COMPONENT_DIRECTORY / "Komponente_K1BE1.csv",
    "K1BE1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1BE1.csv",
    "K1DI1 production": COMPONENT_DIRECTORY / "Komponente_K1DI1.csv",
    "K1DI1 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1DI1.csv",
    "K6 production": COMPONENT_DIRECTORY / "Komponente_K6.csv",
    "K6 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K6.csv",
    "K3AG2 production": COMPONENT_DIRECTORY / "Komponente_K3AG2.txt",
    "K3AG2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3AG2.csv",
    "K3SG2 production": COMPONENT_DIRECTORY / "Komponente_K3SG2.csv",
    "K3SG2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K3SG2.csv",
    "K2LE2 production": COMPONENT_DIRECTORY / "Komponente_K2LE2.txt",
    "K2LE2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2LE2.csv",
    "K2ST2 production": COMPONENT_DIRECTORY / "Komponente_K2ST2.csv",
    "K2ST2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K2ST2.csv",
    "K1BE2 production": COMPONENT_DIRECTORY / "Komponente_K1BE2.csv",
    "K1BE2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1BE2.csv",
    "K1DI2 production": COMPONENT_DIRECTORY / "Komponente_K1DI2.txt",
    "K1DI2 mapping": COMPONENT_DIRECTORY / "Bestandteile_Komponente_K1DI2.csv",
}

## 4.5 Part Files

Each part type is listed only once, even if both vehicle types use it. That avoids reading the same file twice later.


In [5]:
part_paths = {
    "T01": PART_DIRECTORY / "Einzelteil_T01.txt",
    "T02": PART_DIRECTORY / "Einzelteil_T02.txt",
    "T03": PART_DIRECTORY / "Einzelteil_T03.txt",
    "T04": PART_DIRECTORY / "Einzelteil_T04.csv",
    "T05": PART_DIRECTORY / "Einzelteil_T05.csv",
    "T06": PART_DIRECTORY / "Einzelteil_T06.csv",
    "T07": PART_DIRECTORY / "Einzelteil_T07.txt",
    "T08": PART_DIRECTORY / "Einzelteil_T08.csv",
    "T09": PART_DIRECTORY / "Einzelteil_T09.txt",
    "T10": PART_DIRECTORY / "Einzelteil_T10.csv",
    "T11": PART_DIRECTORY / "Einzelteil_T11.txt",
    "T12": PART_DIRECTORY / "Einzelteil_T12.csv",
    "T13": PART_DIRECTORY / "Einzelteil_T13.csv",
    "T14": PART_DIRECTORY / "Einzelteil_T14.csv",
    "T15": PART_DIRECTORY / "Einzelteil_T15.csv",
    "T16": PART_DIRECTORY / "Einzelteil_T16.txt",
    "T17": PART_DIRECTORY / "Einzelteil_T17.csv",
    "T18": PART_DIRECTORY / "Einzelteil_T18.csv",
    "T19": PART_DIRECTORY / "Einzelteil_T19.csv",
    "T20": PART_DIRECTORY / "Einzelteil_T20.txt",
    "T21": PART_DIRECTORY / "Einzelteil_T21.csv",
    "T22": PART_DIRECTORY / "Einzelteil_T22.txt",
    "T23": PART_DIRECTORY / "Einzelteil_T23.csv",
    "T24": PART_DIRECTORY / "Einzelteil_T24.txt",
    "T25": PART_DIRECTORY / "Einzelteil_T25.csv",
    "T26": PART_DIRECTORY / "Einzelteil_T26.csv",
    "T27": PART_DIRECTORY / "Einzelteil_T27.txt",
    "T30": PART_DIRECTORY / "Einzelteil_T30.csv",
    "T31": PART_DIRECTORY / "Einzelteil_T31.txt",
    "T32": PART_DIRECTORY / "Einzelteil_T32.csv",
    "T34": PART_DIRECTORY / "Einzelteil_T34.txt",
    "T35": PART_DIRECTORY / "Einzelteil_T35.txt",
    "T36": PART_DIRECTORY / "Einzelteil_T36.txt",
    "T37": PART_DIRECTORY / "Einzelteil_T37.csv",
}

## 4.6 Plant Files

The three plant files connect plant numbers to a location. OEM plants assemble vehicles, Tier 1 plants produce components, and Tier 2 plants produce parts.


In [6]:
plant_paths = {
    "OEM plants": GEODATA_DIRECTORY / "OEM_Werke_2017-07-04_TrR.csv",
    "Tier 1 plants": GEODATA_DIRECTORY / "Tier1_Werke_2017-07-11_v1.2_TrR.csv",
    "Tier 2 plants": GEODATA_DIRECTORY / "Tier2_Werke_2017-07-11_v1.2_TrR.csv",
}

## 4.7 Files Availability Check

The following table summarises the required and available files for each data group. If any listed file is missing, the notebook stops here.


In [7]:
source_path_groups = {
    "Vehicle files": vehicle_paths,
    "Component files": component_paths,
    "Part files": part_paths,
    "Plant files": plant_paths,
}

path_check_rows = []
missing_files = []

for group_name, paths in source_path_groups.items():
    group_missing = [
        path.relative_to(PROJECT_ROOT).as_posix()
        for path in paths.values()
        if not path.is_file()
    ]
    missing_files.extend(group_missing)
    path_check_rows.append(
        {
            "Data group": group_name,
            "Required files": len(paths),
            "Available files": len(paths) - len(group_missing),
            "Missing files": len(group_missing),
        }
    )

path_check_summary = pd.DataFrame(path_check_rows)
display(path_check_summary)

if missing_files:
    raise FileNotFoundError(f"Missing files: {missing_files}")

all_relevant_paths = {
    **vehicle_paths,
    **component_paths,
    **part_paths,
    **plant_paths,
}

print(f"All {len(all_relevant_paths)} required files are available.")


,Data group,Required files,Available files,Missing files
0,Vehicle files,4,4,0
1,Component files,28,28,0
2,Part files,34,34,0
3,Plant files,3,3,0


All 69 required files are available.


## 4.8 Selected Raw File Previews

The file inventory in Section 3 lists all relevant sources. Here, three representative files are inspected in more detail. The inspection also reports the shape, full column list, raw data types, missing values, and unique values in the inspected sample.

The examples show the main structures that have to be handled later: a vehicle file, a component file with duplicate columns, and a non-standard TXT file.

### 4.8.1 Vehicle Type 11

The first five rows are displayed, while the structure report uses a sample of 1,000 rows.

In [8]:
RAW_INSPECTION_ROWS = 1_000


def show_raw_structure(data):
    """Display a compact overview and one profile row per column."""
    overview = pd.DataFrame(
        [
            {
                "Rows inspected": len(data),
                "Number of columns": data.shape[1],
                "Column names": ", ".join(map(str, data.columns)),
            }
        ]
    )
    column_profile = pd.DataFrame(
        {
            "Column": data.columns,
            "Raw dtype": [str(dtype) for dtype in data.dtypes],
            "Missing values": [int(data[column].isna().sum()) for column in data],
            "Unique values": [int(data[column].nunique(dropna=True)) for column in data],
        }
    )
    display(overview)
    display(column_profile)


vehicle_type_11_raw = pd.read_csv(
    vehicle_paths["Vehicle Type 11"],
    sep=",",
    encoding="cp1252",
    nrows=RAW_INSPECTION_ROWS,
)
display(vehicle_type_11_raw.head())
show_raw_structure(vehicle_type_11_raw)

,Unnamed: 0,X1,ID_Fahrzeug,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,1,1,11-1-11-1,2008-11-18,1,11,0,NaN,0
1,2,2,11-1-11-2,2008-11-18,1,11,0,NaN,0
2,3,3,11-1-11-3,2008-11-19,1,11,0,NaN,0
3,4,4,11-1-11-4,2008-11-19,1,11,0,NaN,0
4,5,5,11-1-11-5,2008-11-19,1,11,0,NaN,0


,Rows inspected,Number of columns,Column names
0,1000,9,"Unnamed: 0, X1, ID_Fahrzeug, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum..."


,Column,Raw dtype,Missing values,Unique values
0,Unnamed: 0,int64,0,1000
1,X1,int64,0,1000
2,ID_Fahrzeug,str,0,1000
3,Produktionsdatum,str,0,7
4,Herstellernummer,int64,0,1
5,Werksnummer,int64,0,1
6,Fehlerhaft,int64,0,2
7,Fehlerhaft_Datum,str,891,5
8,Fehlerhaft_Fahrleistung,int64,0,6


The first two columns are an unnamed column and `X1`. They look like duplicate row counters and will need to be removed.

### 4.8.2 Component K4

Component K4 is an example of a file with duplicated column groups.


In [9]:
component_k4_raw = pd.read_csv(
    component_paths["K4 production"],
    sep=";",
    encoding="cp1252",
    nrows=RAW_INSPECTION_ROWS,
)
display(component_k4_raw.head())
show_raw_structure(component_k4_raw)

,Unnamed: 0,X1,ID_Karosserie.x,Produktionsdatum.x,Herstellernummer.x,Werksnummer.x,Fehlerhaft.x,Fehlerhaft_Datum.x,Fehlerhaft_Fahrleistung.x,ID_Karosserie.y,Produktionsdatum.y,Herstellernummer.y,Werksnummer.y,Fehlerhaft.y,Fehlerhaft_Datum.y,Fehlerhaft_Fahrleistung.y
0,1,1,K4-112-1121-3,2008-11-12,112,1121,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2,K4-112-1121-4,2008-11-12,112,1121,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,3,K4-112-1121-7,2008-11-12,112,1121,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,4,K4-112-1121-9,2008-11-12,112,1121,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,5,K4-112-1121-11,2008-11-12,112,1121,0,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Rows inspected,Number of columns,Column names
0,1000,16,"Unnamed: 0, X1, ID_Karosserie.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehl..."


,Column,Raw dtype,Missing values,Unique values
0,Unnamed: 0,int64,0,1000
1,X1,int64,0,1000
2,ID_Karosserie.x,str,0,1000
3,Produktionsdatum.x,str,0,5
4,Herstellernummer.x,int64,0,1
5,Werksnummer.x,int64,0,1
6,Fehlerhaft.x,int64,0,2
7,Fehlerhaft_Datum.x,str,899,4
8,Fehlerhaft_Fahrleistung.x,int64,0,5
9,ID_Karosserie.y,float64,1000,0


Many attributes appear once with the suffix `.x` and again with `.y`. In the displayed rows the `.y` columns appear to be empty. During cleaning, both versions have to be compared first before they are merged into a single column. Again there are an unnamed column and `X1` that need to be removed.

### 4.8.3 Special TXT format

Several component and part files are not regular CSVs. `K2LE1` is one example.


In [10]:
def show_raw_excerpt(file_path, byte_limit=500):
    """Read a short raw excerpt and make control characters visible."""
    with file_path.open("rb") as raw_file:
        raw_text = raw_file.read(byte_limit).decode("cp1252", errors="replace")

    # Replace invisible control characters with readable labels.
    raw_text = raw_text.replace("\x08", "<BACKSPACE>")
    raw_text = raw_text.replace("\x0b", "<VERTICAL_TAB>")
    raw_text = raw_text.replace("\r", "<CR>")
    raw_text = raw_text.replace("\n", "<LF>")
    return raw_text


show_raw_excerpt(component_paths["K2LE1 production"])


'"X1"II"ID_Sitze.x"II"Produktionsdatum.x"II"Herstellernummer.x"II"Werksnummer.x"II"Fehlerhaft.x"II"Fehlerhaft_Datum.x"II"Fehlerhaft_Fahrleistung.x"II"ID_Sitze.y"II"Produktionsdatum.y"II"Herstellernummer.y"II"Werksnummer.y"II"Fehlerhaft.y"II"Fehlerhaft_Datum.y"II"Fehlerhaft_Fahrleistung.y"<VERTICAL_TAB>"1"II1II"K2LE1-109-1091-2"II2008-11-12II"109"II1091II1II2010-10-18II37080IINAIINAIINAIINAIINAIINAIINA<VERTICAL_TAB>"2"II2II"K2LE1-109-1091-1"II2008-11-12II"109"II1091II0IINAII0IINAIINAIINAIINAIINAIINAIINA<VERTICAL_TAB>"3"II3II"K2LE1-109-'

Fields are separated by `II`, while records (rows) are separated by `<VERTICAL_TAB>`. A normal `read_csv` would not produce the correct columns and rows. These files need the dedicated reader introduced in Section 4.10. After that reader is defined, the parsed sample is inspected with the same structure report as the two CSV examples.

## 4.9 Import Configuration

The initial inspection showed that the files need different import rules. The following configuration records the relationships between component and part types as well as the separators used by the CSV and TXT files. Keeping these rules in one place makes the later EDA and cleaning steps easier to follow.

### 4.9.1 File Import Problems

| Problem | Affected Files | Required Cleaning Step |
|---|---|---|
| CSV files use different separators | **Comma-separated:** `Fahrzeuge_OEM1_Typ11.csv`, `Fahrzeuge_OEM2_Typ21.csv`, `Komponente_K1BE1.csv`, `Komponente_K1DI1.csv`, `Komponente_K3AG1.csv`, `Komponente_K3SG1.csv`, `Komponente_K3SG2.csv`, `Einzelteil_T05.csv`, `T06.csv`, `T08.csv`, `T19.csv`, `T25.csv`, `T30.csv`, `T37.csv`<br><br>**Semicolon-separated:** all other relevant CSV files | Load each file with the correct separator. Then check whether the columns were imported correctly. |
| Component TXT files use special separators | `Komponente_K2LE1.txt`: fields `II`, rows `\x0b`<br>`Komponente_K3AG2.txt`: fields `\`, rows `\r`<br>`Komponente_K2LE2.txt`: fields `\`, rows `\r`<br>`Komponente_K2ST1.txt`: fields `\|`, rows `\r`<br>`Komponente_K1DI2.txt`: fields `\`, rows `\t` | Store the field and row separator for each file and use the special TXT reader to import it. |
| Part TXT files use special row separators | `T02`: fields = 2+ spaces, rows = `\t`<br>`T03`: fields = `\|`, rows = `\x0b`<br>`T09`: fields = `\`, rows = `\x0b`<br>`T11`: fields = `\t`, rows = `\x0c`<br>`T16`: fields = ` \| \| `, rows = `\t`<br>`T24`: fields = 2+ spaces, rows = `\x0c`<br>`T27`: fields = ` \| \| `, rows = `\x07`<br>`T31`: fields = 2+ spaces, rows = `\x08` | First split the file with the row separator. Then split each row with the field separator. |
| Some part TXT files have no row separator | `T01`: fields = ` \| \| `<br>`T07`: fields = `\t`<br>`T20`: fields = ` \| \| `<br>`T22`: fields = `\t`<br>`T34`: fields = ` \| \| `<br>`T35`: fields = `\`<br>`T36`: fields = 2+ spaces | Find the start of each new row using the quoted row number and split the file at these positions. |
| TXT rows can have the wrong number of fields | All component and part TXT files with special separators | Compare the number of values in every row with the number of header columns. Stop the import if they do not match. |
| Source files use a different text encoding | All imported source files | Read the files with `cp1252` encoding and replace characters that cannot be decoded. |


### 4.9.2 Component, Part, and File-Format Rules

The dictionaries below record the file relationships and import rules identified during the initial file inspection:

- which component belongs to which role (body, transmission, seats, engine),
- which part types belong to which component,
- which files use a comma instead of a semicolon,
- which TXT files need a special field and record separator.

The configuration is reused by the EDA in Section 5 and by the cleaning steps in Section 6.


In [11]:
CLEAN_DIRECTORY = DATA_DIRECTORY / "cleaned"

vehicle_data_paths = {
    "Type 11": vehicle_paths["Vehicle Type 11"],
    "Type 21": vehicle_paths["Vehicle Type 21"],
}

vehicle_mapping_paths = {
    "Type 11": vehicle_paths["Vehicle Type 11 mapping"],
    "Type 21": vehicle_paths["Vehicle Type 21 mapping"],
}

component_configuration = {
    "K4": "Karosserie",
    "K3AG1": "Schaltung",
    "K3SG1": "Schaltung",
    "K2LE1": "Sitze",
    "K2ST1": "Sitze",
    "K1BE1": "Motor",
    "K1DI1": "Motor",
    "K6": "Karosserie",
    "K3AG2": "Schaltung",
    "K3SG2": "Schaltung",
    "K2LE2": "Sitze",
    "K2ST2": "Sitze",
    "K1BE2": "Motor",
    "K1DI2": "Motor",
}

component_part_types = {
    "K4": ["T30", "T31", "T32"],
    "K3AG1": ["T21", "T24", "T25"],
    "K3SG1": ["T21", "T22", "T23"],
    "K2LE1": ["T11", "T14", "T15"],
    "K2ST1": ["T11", "T12", "T13"],
    "K1BE1": ["T01", "T02", "T03", "T04"],
    "K1DI1": ["T01", "T02", "T05", "T06"],
    "K6": ["T34", "T35", "T36", "T37"],
    "K3AG2": ["T21", "T24", "T27"],
    "K3SG2": ["T21", "T22", "T26"],
    "K2LE2": ["T16", "T19", "T20"],
    "K2ST2": ["T16", "T17", "T18"],
    "K1BE2": ["T01", "T02", "T07", "T08"],
    "K1DI2": ["T01", "T02", "T09", "T10"],
}

comma_separated_files = {
    "Fahrzeuge_OEM1_Typ11.csv",
    "Fahrzeuge_OEM2_Typ21.csv",
    "Komponente_K1BE1.csv",
    "Komponente_K1DI1.csv",
    "Komponente_K3AG1.csv",
    "Komponente_K3SG1.csv",
    "Komponente_K3SG2.csv",
    "Einzelteil_T05.csv",
    "Einzelteil_T06.csv",
    "Einzelteil_T08.csv",
    "Einzelteil_T19.csv",
    "Einzelteil_T25.csv",
    "Einzelteil_T30.csv",
    "Einzelteil_T37.csv",
}

special_text_formats = {
    "Komponente_K1DI2.txt": ("\\", "\t"),
    "Komponente_K2LE1.txt": ("II", "\x0b"),
    "Komponente_K2LE2.txt": ("\\", "\r"),
    "Komponente_K2ST1.txt": ("|", "\r"),
    "Komponente_K3AG2.txt": ("\\", "\r"),
    "Einzelteil_T01.txt": (" | | ", None),
    "Einzelteil_T02.txt": ("  ", "\t"),
    "Einzelteil_T03.txt": ("|", "\x0b"),
    "Einzelteil_T07.txt": ("\t", None),
    "Einzelteil_T09.txt": ("\\", "\x0b"),
    "Einzelteil_T11.txt": ("\t", "\x0c"),
    "Einzelteil_T16.txt": (" | | ", "\t"),
    "Einzelteil_T20.txt": (" | | ", None),
    "Einzelteil_T22.txt": ("\t", None),
    "Einzelteil_T24.txt": ("  ", "\x0c"),
    "Einzelteil_T27.txt": (" | | ", "\x07"),
    "Einzelteil_T31.txt": ("  ", "\x08"),
    "Einzelteil_T34.txt": (" | | ", None),
    "Einzelteil_T35.txt": ("\\", None),
    "Einzelteil_T36.txt": ("  ", None),
}



## 4.10 Reusable Raw-File Readers

Most files can be read with `pd.read_csv`. The previews showed, however, that several TXT files use unusual field separators and control characters instead of normal line breaks.

`read_special_text_file` reconstructs those files:

- if a record separator is known (for example a vertical tab), the file is split on that character
- if there is no record separator, a new row starts at a quoted numeric index (`"0"`, `"1"`, ...)
- each record is then split into columns with the field separator identified before

`mmap` is used because some of these TXT files are large. It lets us search the file without first copying the whole content into a Python string.

`read_source_file` is the wrapper function used throughout the notebook. It can read a complete file for cleaning or a limited number of rows for the EDA. The columns are first kept as strings so that no information gets lost during initial reading of the files. This in turn means that we have to typecast each individual column correctly afterwards.


In [12]:
def split_special_record(record, field_separator):
    """Split one TXT record into a list of field values.

    The files use different separators, so one split method is not enough.
    After splitting we only strip leftover spaces and quotes.
    """
    # cp1252 because the source files are Windows exports.
    text = record.decode("cp1252", errors="replace").strip()

    if field_separator == "II":
        # Fixed "II" separator, e.g. K2LE1.
        values = text.split("II")
    elif field_separator == " | | ":
        # Number of spaces around the pipes is not always the same.
        values = re.split(r"\s*\|\s*\|\s*", text)
    elif field_separator == "  ":
        # Two or more spaces = new field. A single space can still be inside a value.
        values = re.split(r" {2,}", text)
    else:
        # Backslash, pipe, tab, etc.
        values = next(
            csv.reader(
                [text],
                delimiter=field_separator,
                quotechar='"',
                skipinitialspace=True,
            )
        )

    return [value.strip().strip('"') for value in values]


def special_row_boundary(field_separator):
    """Build a regex that finds the start of a new row.

    Some TXT files have no line breaks at all, just one long stream.
    In those files a new row starts at a quoted index like "0", "1", "2", ...

    We cannot search for every quoted number though: the same pattern can also
    appear in the middle of a row. So we only keep a match if it is NOT sitting
    directly after a field separator.

    We also do not require an ID right after the index. In later .x / .y blocks
    the first ID field can be NA, so that would miss real row starts.
    """
    # Bytes patterns, because we search in the mmap.
    if field_separator == " | | ":
        separator_pattern = rb"\s*\|\s*\|\s*"
        not_after_separator = rb"(?<! \| \| )"
    elif field_separator == "  ":
        separator_pattern = rb" {2,}"
        not_after_separator = rb"(?<!  )"
    elif field_separator == "\t":
        separator_pattern = rb"\t"
        not_after_separator = rb"(?<!\t)"
    elif field_separator == "\\":
        separator_pattern = rb"\\"
        not_after_separator = rb"(?<!\\)"
    elif field_separator == "|":
        separator_pattern = rb"\|"
        not_after_separator = rb"(?<!\|)"
    else:
        encoded_separator = re.escape(field_separator.encode("cp1252"))
        separator_pattern = encoded_separator
        not_after_separator = rb"(?<!" + encoded_separator + rb")"

    return re.compile(not_after_separator + rb'(?="\d+"' + separator_pattern + rb")")


def extract_special_records(mapped_file, field_separator, record_separator):
    """Cut the mapped file into one byte-slice per record (header + data rows).

    Two cases:
    1) The file has a record separator (vertical tab, CR, ...): split there.
    2) It does not: search for row starts with special_row_boundary().
    """
    if record_separator is not None:
        separator = record_separator.encode("cp1252")
        start = 0

        # Walk through the file and cut at every separator.
        while True:
            end = mapped_file.find(separator, start)
            if end == -1:
                # Last piece (no separator after it anymore).
                if start < len(mapped_file):
                    yield mapped_file[start:]
                break
            yield mapped_file[start:end]
            start = end + len(separator)

        return

    # No record separator: the whole file is one stream.
    # Each regex match is the start of a data row. Everything before the
    # first match is the header.
    boundary = special_row_boundary(field_separator)
    previous_start = 0
    first_match = True

    for match in boundary.finditer(mapped_file):
        if first_match:
            yield mapped_file[:match.start()]
            first_match = False
        else:
            yield mapped_file[previous_start:match.start()]
        previous_start = match.start()

    if first_match:
        raise ValueError("No data-row boundary could be found.")

    # Last row goes until the end of the file.
    yield mapped_file[previous_start:]


def read_special_text_file(file_path, row_limit=None):
    """Read one non-standard TXT file and return it as a DataFrame.

    Steps:
    1) look up the separators for this filename
    2) split the file into records
    3) split each record into fields
    4) stop at row_limit when a smaller EDA sample is requested
    5) check that every imported row has the same number of columns
    """
    field_separator, record_separator = special_text_formats[file_path.name]

    # mmap to view the file as bytes without copying it into one huge string.
    with file_path.open("rb") as source_file:
        with mmap.mmap(
            source_file.fileno(),
            length=0,
            access=mmap.ACCESS_READ,
        ) as mapped_file:
            records = extract_special_records(
                mapped_file,
                field_separator,
                record_separator,
            )
            # First record is column names, the rest are data rows.
            header = split_special_record(next(records), field_separator)
            rows = []
            for record in records:
                if not record.strip():
                    continue
                rows.append(split_special_record(record, field_separator))
                if row_limit is not None and len(rows) >= row_limit:
                    break

            # Closing the generator releases its reference to the mmap when
            # the EDA intentionally stops before the end of a large file.
            records.close()

    # Some files put a row index in the data, but not in the header.
    # Then the first data row is one field longer -> add a dummy column name.
    if rows and len(rows[0]) == len(header) + 1:
        header = ["_Zeilenindex", *header]

    # If a later row has a different width, the separators were probably wrong.
    invalid_rows = [
        row_number
        for row_number, row in enumerate(rows, start=1)
        if len(row) != len(header)
    ]
    if invalid_rows:
        raise ValueError(
            f"Unexpected field count in {file_path.name}; "
            f"first invalid data row: {invalid_rows[0]}"
        )

    return pd.DataFrame(rows, columns=header, dtype="string")


def read_source_file(file_path, row_limit=None):
    """Read a complete source file or a limited number of rows."""
    if file_path.name in special_text_formats:
        return read_special_text_file(file_path, row_limit=row_limit)

    separator = "," if file_path.name in comma_separated_files else ";"
    return pd.read_csv(
        file_path,
        sep=separator,
        encoding="cp1252",
        encoding_errors="replace",
        dtype="string",
        keep_default_na=False,
        low_memory=False,
        nrows=row_limit,
    )

# Check that the special reader creates the expected table structure.
special_txt_sample = read_source_file(
    component_paths["K2LE1 production"],
    row_limit=RAW_INSPECTION_ROWS,
)
display(special_txt_sample.head())
show_raw_structure(special_txt_sample)
del special_txt_sample


,_Zeilenindex,X1,ID_Sitze.x,Produktionsdatum.x,Herstellernummer.x,Werksnummer.x,Fehlerhaft.x,Fehlerhaft_Datum.x,Fehlerhaft_Fahrleistung.x,ID_Sitze.y,Produktionsdatum.y,Herstellernummer.y,Werksnummer.y,Fehlerhaft.y,Fehlerhaft_Datum.y,Fehlerhaft_Fahrleistung.y
0,1,1,K2LE1-109-1091-2,2008-11-12,109,1091,1,2010-10-18,37080,NA,NA,NA,NA,NA,NA,NA
1,2,2,K2LE1-109-1091-1,2008-11-12,109,1091,0,NA,0,NA,NA,NA,NA,NA,NA,NA
2,3,3,K2LE1-109-1091-12,2008-11-13,109,1091,0,NA,0,NA,NA,NA,NA,NA,NA,NA
3,4,4,K2LE1-109-1091-5,2008-11-13,109,1091,0,NA,0,NA,NA,NA,NA,NA,NA,NA
4,5,5,K2LE1-109-1091-40,2008-11-13,109,1091,0,NA,0,NA,NA,NA,NA,NA,NA,NA


,Rows inspected,Number of columns,Column names
0,1000,16,"_Zeilenindex, X1, ID_Sitze.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerh..."


,Column,Raw dtype,Missing values,Unique values
0,_Zeilenindex,string,0,1000
1,X1,string,0,1000
2,ID_Sitze.x,string,0,1000
3,Produktionsdatum.x,string,0,9
4,Herstellernummer.x,string,0,1
5,Werksnummer.x,string,0,1
6,Fehlerhaft.x,string,0,2
7,Fehlerhaft_Datum.x,string,0,10
8,Fehlerhaft_Fahrleistung.x,string,0,19
9,ID_Sitze.y,string,0,1


<a id="section-5"></a>

# 5. Exploratory Data Analysis (EDA)

## 5.1 Purpose and Scope

The purpose of this EDA is to get a first view of the raw data before cleaning starts. The section describes the data structure, inspects a set number of rows, and calculates basic summaries.

The audit decision is made on a vehicle level. Therefore, the two vehicle files are explored in more detail. No data cleaning or merging of tables takes place in this section.

## 5.2 Overview of the Raw Data Structure

The case study uses six groups of source data. The table shows what one row represents and which variables are important for the later analysis. Detailed cleaning rules follow in Section 6.

### 5.2.1 Data Levels

In [13]:
raw_data_overview = pd.DataFrame(
    [
        {
            "Data level": "Vehicle",
            "Files": len(vehicle_data_paths),
            "One row represents": "One produced vehicle",
            "Important variables": "Vehicle ID, OEM plant, production date, defect flag",
        },
        {
            "Data level": "Vehicle-component mapping",
            "Files": len(vehicle_mapping_paths),
            "One row represents": "Components installed in one vehicle",
            "Important variables": "Vehicle ID and component IDs",
        },
        {
            "Data level": "Component",
            "Files": sum(name.endswith("production") for name in component_paths),
            "One row represents": "One produced component",
            "Important variables": "Component ID, Tier 1 plant, defect flag",
        },
        {
            "Data level": "Component-part mapping",
            "Files": sum(name.endswith("mapping") for name in component_paths),
            "One row represents": "Parts installed in one component",
            "Important variables": "Component ID and part IDs",
        },
        {
            "Data level": "Part",
            "Files": len(part_paths),
            "One row represents": "One produced individual part",
            "Important variables": "Part ID, Tier 2 plant, defect flag",
        },
        {
            "Data level": "Plant",
            "Files": len(plant_paths),
            "One row represents": "One production plant",
            "Important variables": "Plant number, name, city, and coordinates",
        },
    ]
)

display(raw_data_overview)
print(f"Total number of relevant source files: {raw_data_overview['Files'].sum()}")

,Data level,Files,One row represents,Important variables
0,Vehicle,2,One produced vehicle,"Vehicle ID, OEM plant, production date, defect flag"
1,Vehicle-component mapping,2,Components installed in one vehicle,Vehicle ID and component IDs
2,Component,14,One produced component,"Component ID, Tier 1 plant, defect flag"
3,Component-part mapping,14,Parts installed in one component,Component ID and part IDs
4,Part,34,One produced individual part,"Part ID, Tier 2 plant, defect flag"
5,Plant,3,One production plant,"Plant number, name, city, and coordinates"


Total number of relevant source files: 69


### 5.2.2 Short Profile of All Source Files

The following table shows an overview of all relevant files.

For large files, the first 50,000 rows are inspected. Smaller files are read completely. CSV and TXT files are treated in the same way after `read_source_file` has converted them into a DataFrame.

All columns are initially imported as strings by `read_source_file`. This prevents unintended conversions, keeps IDs unchanged, and allows dates and numeric values to be converted explicitly during cleaning. Therefore, the table reports **initial import dtypes**, not the automatically inferred pandas types shown in the selected previews. Missing cells include empty strings and the markers `NA`, `N/A`, and `NULL`. Duplicate rows are exact row copies within the inspected data, and duplicate IDs are handled later during cleaning.

In [14]:
EDA_ROW_LIMIT = 50_000
RAW_MISSING_MARKERS = {"", "NA", "N/A", "NULL", "null"}

profile_sources = (
    [
        ("Vehicle", path)
        for path in vehicle_data_paths.values()
    ]
    + [
        ("Vehicle-component mapping", path)
        for path in vehicle_mapping_paths.values()
    ]
    + [
        ("Component", path)
        for name, path in component_paths.items()
        if name.endswith("production")
    ]
    + [
        ("Component-part mapping", path)
        for name, path in component_paths.items()
        if name.endswith("mapping")
    ]
    + [
        ("Part", path)
        for path in part_paths.values()
    ]
    + [
        ("Plant", path)
        for path in plant_paths.values()
    ]
)

file_profile_rows = []

for data_level, file_path in profile_sources:
    raw_data = read_source_file(
        file_path,
        row_limit=EDA_ROW_LIMIT,
    )

    missing_counts = []
    for column in raw_data.columns:
        values = raw_data[column].astype("string").str.strip()
        is_missing = values.isna() | values.isin(RAW_MISSING_MARKERS)
        missing_counts.append(int(is_missing.sum()))

    missing_cells = sum(missing_counts)
    inspected_cells = raw_data.shape[0] * raw_data.shape[1]

    dtype_counts = raw_data.dtypes.astype(str).value_counts()
    dtype_summary = ", ".join(
        f"{dtype}: {count}"
        for dtype, count in dtype_counts.items()
    )

    file_profile_rows.append(
        {
            "Data level": data_level,
            "File": file_path.name,
            "Format": file_path.suffix.replace(".", "").upper(),
            "Rows inspected": len(raw_data),
            "Columns": raw_data.shape[1],
            "Initial import dtypes": dtype_summary,
            "Missing cells": missing_cells,
            "Missing cells (%)": round(
                100 * missing_cells / inspected_cells,
                2,
            ) if inspected_cells else 0,
            "Completely empty columns": sum(
                count == len(raw_data)
                for count in missing_counts
            ),
            "Exact duplicate rows": int(raw_data.duplicated().sum()),
        }
    )

    del raw_data

compact_file_profile = pd.DataFrame(file_profile_rows)
display(compact_file_profile)

print(
    f"Profiled {len(compact_file_profile)} files "
    f"with up to {EDA_ROW_LIMIT:,} rows per file."
)

,Data level,File,Format,Rows inspected,Columns,Initial import dtypes,Missing cells,Missing cells (%),Completely empty columns,Exact duplicate rows
0,Vehicle,Fahrzeuge_OEM1_Typ11.csv,CSV,50000,9,string: 9,45097,10.02,0,0
1,Vehicle,Fahrzeuge_OEM2_Typ21.csv,CSV,50000,10,string: 10,45111,9.02,0,0
2,Vehicle-component mapping,Bestandteile_Fahrzeuge_OEM1_Typ11.csv,CSV,50000,6,string: 6,0,0.00,0,0
3,Vehicle-component mapping,Bestandteile_Fahrzeuge_OEM2_Typ21.csv,CSV,50000,6,string: 6,0,0.00,0,0
4,Component,Komponente_K4.csv,CSV,50000,16,string: 16,395072,49.38,7,0
5,Component,Komponente_K3AG1.csv,CSV,50000,23,string: 23,739915,64.34,14,0
6,Component,Komponente_K3SG1.csv,CSV,50000,16,string: 16,395001,49.38,7,0
7,Component,Komponente_K2LE1.txt,TXT,50000,16,string: 16,395510,49.44,7,0
8,Component,Komponente_K2ST1.txt,TXT,50000,9,string: 9,45015,10.00,0,0
9,Component,Komponente_K1BE1.csv,CSV,50000,10,string: 10,41187,8.24,0,0


Profiled 69 files with up to 50,000 rows per file.


## 5.3 Inspection of Relevant Files

The first 50,000 rows of both vehicle files are used for the initial exploration. This keeps the EDA quick while still showing the structure and recurring raw values. All columns are intentionally read as strings so that pandas does not silently change IDs or dates during import.

Only the columns needed for the plant-level audit are shown below. Type 21 stores its production date as a number of days from an origin date, which is already visible without converting it.

In [15]:
vehicle_eda_data = {
    vehicle_type: read_source_file(file_path, row_limit=EDA_ROW_LIMIT)
    for vehicle_type, file_path in vehicle_data_paths.items()
}

structure_rows = []
for vehicle_type, data in vehicle_eda_data.items():
    dtype_counts = data.dtypes.astype(str).value_counts()
    dtype_text = ", ".join(
        f"{dtype}: {count} columns"
        for dtype, count in dtype_counts.items()
    )
    structure_rows.append(
        {
            "Vehicle type": vehicle_type,
            "Rows inspected": len(data),
            "Columns": data.shape[1],
            "Raw data types": dtype_text,
        }
    )

display(pd.DataFrame(structure_rows))

preview_columns = {
    "Type 11": [
        "ID_Fahrzeug",
        "Produktionsdatum",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ],
    "Type 21": [
        "ID_Fahrzeug",
        "Produktionsdatum_Origin_01011970",
        "origin",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ],
}

for vehicle_type, data in vehicle_eda_data.items():
    available_columns = [
        column for column in preview_columns[vehicle_type]
        if column in data.columns
    ]
    print(f"{vehicle_type}: first three raw rows")
    display(data[available_columns].head(3))

,Vehicle type,Rows inspected,Columns,Raw data types
0,Type 11,50000,9,string: 9 columns
1,Type 21,50000,10,string: 10 columns


Type 11: first three raw rows


,ID_Fahrzeug,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,11-1-11-1,2008-11-18,1,11,0,NA,0
1,11-1-11-2,2008-11-18,1,11,0,NA,0
2,11-1-11-3,2008-11-19,1,11,0,NA,0


Type 21: first three raw rows


,ID_Fahrzeug,Produktionsdatum_Origin_01011970,origin,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
0,21-2-21-1,14202,01-01-1970,2,21,0,NA,0
1,21-2-21-2,14202,01-01-1970,2,21,0,NA,0
2,21-2-21-3,15353,01-01-1970,2,21,0,NA,0


## 5.4 Basic Statistics and Data Integrity

For this task, the useful questions are:

- Are the vehicle IDs present and how many unique vehicles are observed?
- Which OEM plants occur in each vehicle file?
- Which raw values occur in the defect flag?
- How much defect information is missing?
- What is the observed range of the numeric defect mileage?

The values are only summarised here. Missing markers are counted, but they are not replaced. Numeric conversion is used temporarily for the mileage summary and does not change the raw table.

In [16]:
def count_raw_missing(series):
    """Count the missing-value markers without changing the raw data."""
    text = series.astype("string").str.strip()
    return int(text.isin(RAW_MISSING_MARKERS).sum())


basic_summary_rows = []
missing_value_rows = []

for vehicle_type, data in vehicle_eda_data.items():
    defect_values = data["Fehlerhaft"].astype("string").str.strip()
    observed_defect_values = sorted(
        defect_values[~defect_values.isin(RAW_MISSING_MARKERS)].unique()
    )
    defect_mileage = pd.to_numeric(
        data["Fehlerhaft_Fahrleistung"],
        errors="coerce",
    )

    basic_summary_rows.append(
        {
            "Vehicle type": vehicle_type,
            "Rows inspected": len(data),
            "Unique vehicle IDs": data["ID_Fahrzeug"].nunique(),
            "OEM plants": data["Werksnummer"].nunique(),
            "Observed defect values": ", ".join(observed_defect_values),
            "Mileage minimum": defect_mileage.min(),
            "Mileage median": defect_mileage.median(),
            "Mileage maximum": defect_mileage.max(),
        }
    )

    relevant_columns = [
        "ID_Fahrzeug",
        "Herstellernummer",
        "Werksnummer",
        "Fehlerhaft",
        "Fehlerhaft_Datum",
        "Fehlerhaft_Fahrleistung",
    ]
    date_column = (
        "Produktionsdatum"
        if "Produktionsdatum" in data.columns
        else "Produktionsdatum_Origin_01011970"
    )
    relevant_columns.append(date_column)

    for column in relevant_columns:
        missing_count = count_raw_missing(data[column])
        missing_value_rows.append(
            {
                "Vehicle type": vehicle_type,
                "Variable": column,
                "Missing values": missing_count,
                "Missing (%)": round(100 * missing_count / len(data), 2),
            }
        )

display(pd.DataFrame(basic_summary_rows))
display(pd.DataFrame(missing_value_rows))

,Vehicle type,Rows inspected,Unique vehicle IDs,OEM plants,Observed defect values,Mileage minimum,Mileage median,Mileage maximum
0,Type 11,50000,50000,2,"0, 1",0,0.0,45463
1,Type 21,50000,50000,1,"0, 1",0,0.0,227523


,Vehicle type,Variable,Missing values,Missing (%)
0,Type 11,ID_Fahrzeug,0,0.00
1,Type 11,Herstellernummer,0,0.00
2,Type 11,Werksnummer,0,0.00
3,Type 11,Fehlerhaft,0,0.00
4,Type 11,Fehlerhaft_Datum,45097,90.19
5,Type 11,Fehlerhaft_Fahrleistung,0,0.00
6,Type 11,Produktionsdatum,0,0.00
7,Type 21,ID_Fahrzeug,0,0.00
8,Type 21,Herstellernummer,0,0.00
9,Type 21,Werksnummer,0,0.00


## 5.5 Initial Visualisations

Three simple charts give enough orientation before cleaning:

1. the raw number of defective, non-defective, and missing or other defect entries by vehicle type, and
2. the number of vehicle observations by OEM plant, and
3. the distribution of the recorded defect mileage for defective vehicles.

A production-year plot is deliberately postponed. The two vehicle types use different raw date formats, so a time comparison is only possible after the dates have been standardised in Section 6.

In [17]:
defect_plot_rows = []
plant_plot_rows = []

for vehicle_type, data in vehicle_eda_data.items():
    defect_values = data["Fehlerhaft"].astype("string").str.strip()
    defect_labels = defect_values.map(
        {
            "0": "Not defective",
            "1": "Defective",
        }
    ).fillna("Missing or other")

    for defect_status, count in defect_labels.value_counts().items():
        defect_plot_rows.append(
            {
                "Vehicle type": vehicle_type,
                "Defect status": defect_status,
                "Vehicles": int(count),
            }
        )

    for plant_number, count in data["Werksnummer"].value_counts().items():
        plant_plot_rows.append(
            {
                "Vehicle type": vehicle_type,
                "OEM plant number": plant_number,
                "Vehicles": int(count),
            }
        )

defect_plot_data = pd.DataFrame(defect_plot_rows)
fig = px.bar(
    defect_plot_data,
    x="Vehicle type",
    y="Vehicles",
    color="Defect status",
    barmode="group",
    title="Raw vehicle defect entries in the EDA sample",
)
fig.show()

plant_plot_data = pd.DataFrame(plant_plot_rows)
fig = px.bar(
    plant_plot_data,
    x="OEM plant number",
    y="Vehicles",
    color="Vehicle type",
    barmode="group",
    title="Vehicle observations by OEM plant in the EDA sample",
)
fig.show()

# The source columns stay unchanged. Numeric conversion is only used
# to visualise the continuous mileage variable during the EDA.
mileage_plot_frames = []
for vehicle_type, data in vehicle_eda_data.items():
    defective_rows = data["Fehlerhaft"].astype("string").str.strip().eq("1")
    mileage_values = pd.to_numeric(
        data.loc[defective_rows, "Fehlerhaft_Fahrleistung"]
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )
    mileage_plot_frames.append(
        pd.DataFrame(
            {
                "Vehicle type": vehicle_type,
                "Defect mileage": mileage_values,
            }
        ).dropna()
    )

mileage_plot_data = pd.concat(mileage_plot_frames, ignore_index=True)
fig = px.histogram(
    mileage_plot_data,
    x="Defect mileage",
    facet_col="Vehicle type",
    nbins=50,
    marginal="box",
    title="Raw defect-mileage distribution in the EDA sample",
    labels={"Defect mileage": "Defect mileage [km]"},
)
fig.update_xaxes(matches=None)
fig.update_yaxes(title="Vehicles")
fig.show()

### Result

The first two charts confirm the structure already seen in the tables. Both vehicle samples contain direct defective and non-defective records. Type 11 contains the OEM plants O11 and O12, while Type 21 contains O21. The mileage chart only uses rows marked as defective because non-defective vehicle rows contain a mileage value of zero. Type 11 defect mileage is concentrated between about 42,000 and 45,000 km. Most Type 21 values are around 9,000 to 13,000 km, but one much larger value reaches 227,523 km.

## 5.6 Main Findings for Data Cleaning

The initial exploration gives the following starting points:

- The compact profile covers all 69 required files: 49 CSV files and 20 TXT files across vehicle, mapping, component, part, and plant data.
- High missing-cell shares in some component and part files are partly caused by sparse repeated `.x` and `.y` column blocks. The columns are compared before they are combined during cleaning.
- The two vehicle files contain the IDs, plant numbers, and defect variables required for the audit comparison.
- The vehicle types use different raw production-date formats. These dates should not be compared before they are standardised.
- Missing defect dates or mileage do not automatically mean that a row is wrong. Their meaning has to be considered together with the defect flag during cleaning.
- The raw previews in Section 4 also show technical export columns, repeated `.x` and `.y` blocks, and special TXT separators.
- The charts are only first views of the raw vehicle data. They are not the final defect rates because component and part defects have not yet been added.

In the inspected 50,000 rows, both vehicle types have 50,000 unique vehicle IDs and no missing values in the main ID, plant, or defect columns. Type 11 contains 4,903 direct defect entries (9.81%) and Type 21 contains 4,889 (9.78%). The missing defect dates match the number of non-defective rows in both samples, so they do not indicate a separate problem. The plant chart shows that the Type 11 sample contains plants 11 and 12, while the Type 21 sample contains plant 21. These are still sample results and are not used as the final plant comparison.

No corrections or joins have been made in this section. The detailed problems and the corresponding cleaning actions are documented at the start of Section 6.

<a id="section-6"></a>

# 6. Data Cleaning and Preparation

This section corrects the problems found during the initial inspection and EDA. It also combines files that describe the same kind of object. The merging across vehicles, components, and parts is kept for Section 7.

The cleaning rules follow the meaning of the variables. Missing defect information stays unknown. Technically impossible values, such as negative mileage, are converted to missing and recorded. After each file group, a compact cleaning summary separates the raw file, the fully cleaned and validated intermediate result, and the smaller table retained for integration. A short preview of the combined cleaned table is also displayed so that its actual values and column structure remain visible. To keep memory use manageable, full source columns are validated per file and only the fields needed for the later analysis are retained.

## 6.1 Detailed Data Issues and Required Actions

The first table contains only issues that were actually found during the raw-data inspection or the cleaning runs. The file counts use the 69 relevant source files: 2 vehicle files, 2 vehicle-component mapping files, 14 component files, 14 component-part mapping files, 34 part files, and 3 plant files. If an issue can only occur in one file group, the denominator refers to that group instead of all 69 files. Counts from the raw-data inspection are based on the same sample of up to 50,000 rows per file used in the EDA.

### 6.1.1 Observed Structural and Content Issues

| Observed Issue | Affected Files | Number of Affected Files | Cleaning Action |
|---|---|---:|---|
| Technical export columns | All relevant source files except the OEM and Tier-1 plant files; examples are `X`, `X1`, `_Zeilenindex`, empty column names, and `Unnamed:*` | **67/69** | Remove these columns after importing each file. |
| Different missing-value markers | Files with blank cells or the markers `NA`, `N/A`, `NULL`, and `null` in the EDA sample | **52/69** | Strip text values and replace the defined markers with `pd.NA`. |
| Spaces in plant column names | OEM, Tier-1, and Tier-2 plant files | **3/3 plant files** | Strip the column names. |
| Repeated column blocks with `.x` and `.y` suffixes | **Components:** `K4`, `K3AG1`, `K3SG1`, `K2LE1`, `K1DI1`<br><br>**Parts:** `T01`, `T02`, `T05`, `T09`, `T12`, `T15`, `T16`, `T17`, `T22`, `T23`, `T24`, `T30`, `T32`, `T35` | **5/14 component files**<br>**14/34 part files** | Compare the normal, `.x`, and `.y` versions and combine the first non-missing value. |
| Production dates stored as days since an origin date | **Vehicle:** `Fahrzeuge_OEM2_Typ21.csv`<br><br>**Components:** `K1BE1`, `K6`, `K3AG2`, `K3SG2`, `K2LE2`, `K2ST2`, `K1BE2`, `K1DI2`<br><br>**Parts:** `T03`, `T04`, `T06`, `T07`, `T08`, `T10`, `T11`, `T13`, `T14`, `T18`, `T19`, `T20`, `T21`, `T25`, `T26`, `T27`, `T31`, `T34`, `T36`, `T37` | **1/2 vehicle files**<br>**8/14 component files**<br>**20/34 part files**<br>**29/50 production files in total** | Add the stored number of days to the date in `origin`. Convert normal production dates directly and keep the cleaned column as a datetime. |
| Manufacturer or plant numbers do not match the structured object ID | **Plant number:** `T05`, `T07`, `T08`, `T10`, `T13`, `T18`, `K1DI1`, `K2ST1`, `K6`<br><br>**Manufacturer number:** `T06`, `T13`, `K2ST1` | **10/50 production files** | Extract the expected manufacturer and plant number from the object ID. Use it when the separate value is missing or different, and record the correction. |
| Empty or misplaced plant rows and short postal codes | Tier-1 plant file: 22 completely empty rows and one row containing only plant number `2193`<br><br>Tier-2 plant file: the complete record for plant `2193` and 4 four-digit postal codes | **2/3 plant files** | Remove the 22 empty Tier-1 rows and the incomplete `2193` row. Plant `2193` belongs to the Tier-2 file, where its complete Borgsdorf record is available. Add leading zeros to the four affected Tier-2 postal codes. |
| OEM plant numbers have an additional prefix | OEM plant file; values such as `O11` | **1/1 OEM plant file** | Remove the leading `O` so that the plant number matches the format used in the vehicle data. |
| Short part-type names in component-part mappings | `K1BE1`, `K1BE2`, `K1DI1`, `K1DI2`; for example `ID_T1` instead of `ID_T01` | **4/14 component-part mapping files** | Check both possible column names and rename the selected part-ID column to `ID_Einzelteil` during processing. |
| Decimal commas in defect-mileage values | Parts `T04`, `T10`, `T12`, `T13`, `T14`, `T15`, `T17`, `T18`, `T21`, `T23`, `T26`, `T32` | **12/34 part files** | Replace the decimal comma with a decimal point before numeric conversion. Record values that still cannot be converted. |
| Negative defect mileage | Component `K3SG1` and parts `T04`, `T07`, `T09`, `T19`, `T20`, `T31` | **1/14 component files**<br>**6/34 part files** | Replace negative mileage with `NA` and record the number of affected rows. Keep the remaining production record. |

### 6.1.2 Data Preparation and Validation Rules

The following points are preparation decisions and validation rules. They are important for a reproducible result, but they should not be read as additional errors that were found in every file. Most checks apply to complete processing stages.

| Rule | Scope | Required Preparation or Validation |
|---|---|---|
| Preserve identifiers and keys | ID, manufacturer, plant, and postal-code columns | Import these columns as text so that leading zeros and exact identifiers are not changed automatically. |
| Convert imported dates explicitly | Production and defect-date columns | `read_source_file` intentionally imports all columns as strings. Convert normal date values to datetimes during cleaning and use `YYYY-MM-DD` during export. |
| Keep missing defect information unknown | Vehicle, component, and part data | Use three defect states: `1` for defective, `0` for not defective, and `NA` for unknown. No missing defect flags were found in the current cleaning run, but the rule prevents missing information from being treated as `0`. |
| Use the correct part types for every component | Component-part mapping files | Use the configured part-type list, connect each mapping to the matching part-quality table, and aggregate the result directly by component ID. |
| Add plant information at the required level | Vehicle, component, part, and plant data | Add OEM plant data to vehicles and the required Tier-1 fields to components. Validate Tier-2 part keys against the plant reference without copying plant columns to every part row. |
| Propagate defects from the bottom up | Cleaned vehicle, component, part, and mapping data | Mark a vehicle as defective if the vehicle itself, one of its components, or one of its parts is defective. Keep the result unknown if there is no confirmed defect but some information is missing. |
| IDs must be unique after cleaning | Cleaned vehicle, component, and part tables | Check uniqueness after cleaning. |
| Each vehicle should have four components | Cleaned wide vehicle-component mapping | Check that every vehicle has one mapping row and that all four component ID columns are filled. |
| Table joins must have the expected structure | Plant joins and bottom-up integration | Use `validate` during merges to check whether the join is `many-to-one` or `one-to-one`. Check the row count after every merge. |
| The final dataset must stay at vehicle level | `final_vehicle_data` | Check the expected number of Type 11 and Type 21 vehicles and require one unique row per `Vehicle_ID`. |
| Detail IDs are temporary merge keys | Component and part integration | Use component and part IDs during bottom-up aggregation, but do not include them in the final vehicle-level dataset. |
| Cleaning changes must be documented | Conflicts, corrected keys, invalid values, and defensive duplicate handling | Keep the source file, type of correction, and number of affected rows in the cleaning log. Also show a before/after cleaning summary for every source file. |

## 6.2 Helper Functions for Cleaning

The functions below solve problems that occur in several files:

- `standardize_raw_values` strips whitespace and replaces missing-value markers,
- `remove_export_columns` drops technical row counters,
- `combine_duplicate_columns` combines regular, `.x`, and `.y` columns,
- `convert_numeric_values` standardises decimal commas, converts numeric columns, and records invalid values,
- `standardize_dates` converts the different date formats while keeping datetime columns,
- `correct_keys_from_id` corrects manufacturer and plant numbers from the structured IDs,
- `consolidate_duplicate_ids` combines repeated component or part IDs and records conflicts,
- `record_cleaning_summary` separates the raw, validated, and retained table structures,
- `record_post_cleaning_quality` checks converted fields before unused source columns are discarded, and
- `three_state_flag` keeps the distinction between confirmed defect, confirmed non-defect, and unknown.

No statistical imputation is used. Missing defect information remains unknown because replacing it with a typical value would change the later defect rates.

In [18]:
# Detailed corrections are collected for the quality checks at the end of the section.
cleaning_log = []

# One compact before/after row is stored for every source file.
cleaning_summary_rows = []


def dtype_summary(data):
    """Return all column data types as one readable string."""
    return ", ".join(
        f"{column}: {dtype}" for column, dtype in data.dtypes.items()
    )


def record_cleaning_summary(
    source_file,
    data_level,
    raw_data,
    clean_data,
    retained_data=None,
):
    """Record the raw, validated, and retained structure of one source file."""
    if retained_data is None:
        retained_data = clean_data

    raw_columns = list(map(str, raw_data.columns))
    clean_columns = list(map(str, clean_data.columns))
    retained_columns = list(map(str, retained_data.columns))
    prepared_columns = list(dict.fromkeys(clean_columns + retained_columns))

    removed_columns = [column for column in raw_columns if column not in clean_columns]
    added_columns = [column for column in prepared_columns if column not in raw_columns]
    dropped_columns = [column for column in clean_columns if column not in retained_columns]

    cleaning_summary_rows.append(
        {
            "Data level": data_level,
            "Source file": source_file,
            "Raw shape": f"{raw_data.shape[0]:,} x {raw_data.shape[1]}",
            "Cleaned / validated shape": (
                f"{clean_data.shape[0]:,} x {clean_data.shape[1]}"
            ),
            "Retained shape": (
                f"{retained_data.shape[0]:,} x {retained_data.shape[1]}"
            ),
            "Columns before": ", ".join(raw_columns),
            "Columns after cleaning": ", ".join(clean_columns),
            "Columns retained": ", ".join(retained_columns),
            "Removed / reshaped during cleaning": (
                ", ".join(removed_columns) or "None"
            ),
            "Added during preparation": ", ".join(added_columns) or "None",
            "Dropped after validation": ", ".join(dropped_columns) or "None",
            "Dtypes before": dtype_summary(raw_data),
            "Dtypes after cleaning": dtype_summary(clean_data),
            "Dtypes retained": dtype_summary(retained_data),
        }
    )


def display_cleaning_summary(data_level):
    """Display the before/after rows for one cleaning subsection."""
    summary = pd.DataFrame(cleaning_summary_rows)
    summary = summary.loc[summary["Data level"] == data_level]
    with pd.option_context("display.max_colwidth", 120):
        display(summary.reset_index(drop=True))


def standardize_raw_values(data):
    """Trim text and replace the missing-value markers used in the source files."""
    cleaned = data.copy()
    cleaned.columns = [str(column).strip() for column in cleaned.columns]

    for column in cleaned.columns:
        cleaned[column] = cleaned[column].astype("string").str.strip()

    missing_markers = {
        "": pd.NA,
        "NA": pd.NA,
        "N/A": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
    }
    return cleaned.replace(missing_markers)


def remove_export_columns(data):
    """Drop technical row counters created by earlier exports."""
    technical_columns = [
        column
        for column in data.columns
        if column in {"X", "X1", "_Zeilenindex"}
        or column.startswith("Unnamed:")
        or column == ""
    ]
    return data.drop(columns=technical_columns, errors="ignore")


def combine_duplicate_columns(data, source_file):
    """Combine name, name.x, and name.y columns and record disagreements."""
    base_columns = []
    for column in data.columns:
        base_column = re.sub(r"\.[xy]$", "", column)
        if base_column not in base_columns:
            base_columns.append(base_column)

    combined = pd.DataFrame(index=data.index)
    conflict_count = 0

    for base_column in base_columns:
        candidates = [
            column
            for column in [base_column, f"{base_column}.x", f"{base_column}.y"]
            if column in data.columns
        ]

        combined_values = pd.Series(pd.NA, index=data.index, dtype="string")
        for candidate in candidates:
            candidate_values = data[candidate].astype("string")
            conflicts = (
                combined_values.notna()
                & candidate_values.notna()
                & combined_values.ne(candidate_values)
            )
            conflict_count += int(conflicts.fillna(False).sum())
            combined_values = combined_values.combine_first(candidate_values)

        combined[base_column] = combined_values

    if conflict_count > 0:
        cleaning_log.append(
            {
                "Quelldatei": source_file,
                "Prüfung": "Conflicting .x/.y values",
                "Anzahl": conflict_count,
            }
        )

    return combined


def convert_numeric_values(
    series,
    source_file,
    column_name,
    integer=False,
    non_negative=False,
    allowed_values=None,
):
    """Convert one numeric column and document invalid values."""
    # Some source files use a decimal comma, for example "1002,86".
    standardized_values = (
        series.astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
    )
    numeric_values = pd.to_numeric(standardized_values, errors="coerce")

    conversion_failures = series.notna() & numeric_values.isna()
    failure_count = int(conversion_failures.sum())
    if failure_count > 0:
        cleaning_log.append(
            {
                "Quelldatei": source_file,
                "Prüfung": f"Invalid {column_name} converted to missing",
                "Anzahl": failure_count,
            }
        )

    if allowed_values is not None:
        unexpected_values = numeric_values.notna() & ~numeric_values.isin(allowed_values)
        unexpected_count = int(unexpected_values.sum())
        if unexpected_count > 0:
            cleaning_log.append(
                {
                    "Quelldatei": source_file,
                    "Prüfung": f"Unexpected {column_name} converted to missing",
                    "Anzahl": unexpected_count,
                }
            )
            numeric_values = numeric_values.mask(unexpected_values)

    if non_negative:
        negative_values = numeric_values < 0
        negative_count = int(negative_values.sum())
        if negative_count > 0:
            cleaning_log.append(
                {
                    "Quelldatei": source_file,
                    "Prüfung": f"Negative {column_name} converted to missing",
                    "Anzahl": negative_count,
                }
            )
            numeric_values = numeric_values.mask(negative_values)

    if integer:
        return numeric_values.astype("Int64")
    return numeric_values


def consolidate_duplicate_ids(data, id_column, source_file):
    """Combine repeated IDs while retaining a confirmed defect from any copy."""
    duplicate_rows = data[id_column].notna() & data.duplicated(id_column, keep=False)
    duplicate_count = int(duplicate_rows.sum())

    if duplicate_count == 0:
        return data

    duplicated_data = data.loc[duplicate_rows].copy()
    detail_columns = [
        column for column in data.columns if column not in {id_column, "Fehlerhaft"}
    ]
    conflicting_ids = (
        duplicated_data.groupby(id_column)[detail_columns]
        .nunique(dropna=True)
        .gt(1)
        .any(axis=1)
    )

    cleaning_log.append(
        {
            "Quelldatei": source_file,
            "Prüfung": "Rows in duplicate ID groups consolidated",
            "Anzahl": duplicate_count,
        }
    )
    if int(conflicting_ids.sum()) > 0:
        cleaning_log.append(
            {
                "Quelldatei": source_file,
                "Prüfung": "Duplicate IDs with conflicting details",
                "Anzahl": int(conflicting_ids.sum()),
            }
        )

    aggregation = {
        column: "max" if column == "Fehlerhaft" else "first"
        for column in data.columns
        if column != id_column
    }
    consolidated_rows = duplicated_data.groupby(id_column, as_index=False).agg(aggregation)

    return pd.concat(
        [data.loc[~duplicate_rows], consolidated_rows],
        ignore_index=True,
    )


# Quality checks are stored per file before wide source columns are discarded.
post_cleaning_quality_rows = []


def record_post_cleaning_quality(table_name, source_file, data, valid_plant_numbers):
    """Record the converted fields and plant-key quality for one cleaned file."""
    defect_values = data["Fehlerhaft"]
    mileage_values = data["Fehlerhaft_Fahrleistung"]
    missing_plant_matches = (
        data["Werksnummer"].notna()
        & ~data["Werksnummer"].isin(valid_plant_numbers)
    )

    post_cleaning_quality_rows.append(
        {
            "Table": table_name,
            "Source file": source_file,
            "Production date is datetime": pd.api.types.is_datetime64_any_dtype(
                data["Produktionsdatum"]
            ),
            "Defect date is datetime": pd.api.types.is_datetime64_any_dtype(
                data["Fehlerhaft_Datum"]
            ),
            "Defect flag is Int64": str(defect_values.dtype) == "Int64",
            "Missing defect flags": int(defect_values.isna().sum()),
            "Invalid defect flags": int(
                (defect_values.notna() & ~defect_values.isin([0, 1])).sum()
            ),
            "Negative mileage": int((mileage_values < 0).sum()),
            "Missing plant matches": int(missing_plant_matches.sum()),
        }
    )


def three_state_flag(has_defect, has_unknown):
    """Return 1 for a defect, 0 for known good, and NA for unknown."""
    has_defect = pd.Series(has_defect, dtype="boolean")
    has_unknown = pd.Series(has_unknown, dtype="boolean")
    result = pd.Series(0, index=has_defect.index, dtype="Int64")
    result = result.mask(
        ~has_defect.fillna(False) & has_unknown.fillna(False),
        pd.NA,
    )
    return result.mask(has_defect.fillna(False), 1)

In [19]:
def parse_date_values(
    series,
    source_file,
    column_name,
    date_format="mixed",
):
    """Convert one date column and record values that cannot be parsed."""
    parsed_dates = pd.to_datetime(
        series,
        format=date_format,
        errors="coerce",
    )
    conversion_failures = series.notna() & parsed_dates.isna()
    failure_count = int(conversion_failures.sum())

    if failure_count > 0:
        cleaning_log.append(
            {
                "Quelldatei": source_file,
                "Prüfung": f"Invalid {column_name} converted to missing",
                "Anzahl": failure_count,
            }
        )

    return parsed_dates


def standardize_dates(data, source_file):
    """Convert production and defect dates and keep them as datetimes."""
    cleaned = data.copy()
    origin_value_column = "Produktionsdatum_Origin_01011970"

    if origin_value_column in cleaned.columns:
        if "origin" in cleaned.columns:
            origin_dates = parse_date_values(
                cleaned["origin"],
                source_file,
                "origin",
                date_format="%d-%m-%Y",
            )
        else:
            origin_dates = pd.Series(
                pd.Timestamp("1970-01-01"),
                index=cleaned.index,
            )

        day_values = convert_numeric_values(
            cleaned[origin_value_column],
            source_file,
            origin_value_column,
        )
        converted_dates = origin_dates + pd.to_timedelta(
            day_values,
            unit="D",
        )

        if "Produktionsdatum" in cleaned.columns:
            existing_dates = parse_date_values(
                cleaned["Produktionsdatum"],
                source_file,
                "Produktionsdatum",
            )
        else:
            existing_dates = pd.Series(pd.NaT, index=cleaned.index)

        cleaned["Produktionsdatum"] = existing_dates.fillna(converted_dates)
        cleaned = cleaned.drop(
            columns=[origin_value_column, "origin"],
            errors="ignore",
        )

    for date_column in ["Produktionsdatum", "Fehlerhaft_Datum"]:
        if date_column in cleaned.columns:
            cleaned[date_column] = parse_date_values(
                cleaned[date_column],
                source_file,
                date_column,
            )

    return cleaned


def correct_keys_from_id(data, id_column, source_file):
    """If Herstellernummer / Werksnummer differs from ID, take the ID.

    We join plants later on Werksnummer, so a wrong plant number would lead to errors in merging. 
    Every overwrite is written to cleaning_log.
    """
    cleaned = data.copy()
    # Group 1 = manufacturer, group 2 = plant. Everything is separated by '-'.
    id_keys = cleaned[id_column].astype("string").str.extract(
        r"^[^-]+-([^-]+)-([^-]+)-"
    )
    id_keys.columns = ["Herstellernummer_ID", "Werksnummer_ID"]

    key_pairs = [
        ("Herstellernummer", "Herstellernummer_ID"),
        ("Werksnummer", "Werksnummer_ID"),
    ]

    for target_column, id_key_column in key_pairs:
        if target_column not in cleaned.columns:
            cleaned[target_column] = pd.NA

        # Remove ".0" leftover in Werk IDs.
        current_values = (
            cleaned[target_column]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
        )
        id_values = id_keys[id_key_column].astype("string")
        corrections = (
            id_values.notna()
            & current_values.fillna("").ne(id_values)
        )
        correction_count = int(corrections.sum())

        if correction_count > 0:
            cleaning_log.append(
                {
                    "Quelldatei": source_file,
                    "Prüfung": f"Corrected {target_column} from ID",
                    "Anzahl": correction_count,
                }
            )

        # Overwrite mismatches with the ID segment, then fill remaining gaps.
        cleaned[target_column] = current_values.mask(corrections, id_values)
        cleaned[target_column] = cleaned[target_column].fillna(id_values)

    return cleaned

## 6.3 Plant Data

The OEM, Tier-1, and Tier-2 plant files are cleaned and combined into one reference table. The plant number is used to add locations to vehicles and components and to validate the plant keys in the part files.

For every file, missing-value markers and text formats are standardised first. Rows without the plant name, city, or postal code cannot be used as reference data and are removed. Plant numbers and postal codes are kept as text, while coordinates are converted to numbers. Duplicate plant numbers are removed because the later merge expects one reference row per plant. The source-file label is used for the checks but is not retained in the combined reference table.

In [20]:
plant_source_files = {
    "OEM": plant_paths["OEM plants"],
    "Tier 1": plant_paths["Tier 1 plants"],
    "Tier 2": plant_paths["Tier 2 plants"],
}

plant_frames = []

for plant_level, file_path in plant_source_files.items():
    raw_plant_data = read_source_file(file_path)
    cleaned_plant_data = standardize_raw_values(raw_plant_data)
    cleaned_plant_data = cleaned_plant_data.dropna(axis=1, how="all")

    # One file contains a damaged umlaut in the longitude column name.
    cleaned_plant_data.columns = [
        "Längengrad" if "ngengrad" in column else column
        for column in cleaned_plant_data.columns
    ]

    required_plant_columns = ["Werk", "ORT", "PLZ"]
    incomplete_rows = cleaned_plant_data[required_plant_columns].isna().any(axis=1)
    if int(incomplete_rows.sum()) > 0:
        cleaning_log.append(
            {
                "Quelldatei": file_path.name,
                "Prüfung": "Incomplete plant rows removed",
                "Anzahl": int(incomplete_rows.sum()),
            }
        )
    cleaned_plant_data = cleaned_plant_data.loc[~incomplete_rows].copy()

    cleaned_plant_data["Werksebene"] = plant_level
    cleaned_plant_data["Werksnummer"] = (
        cleaned_plant_data["Werk"].astype("string").str.replace(r"^O", "", regex=True)
    )
    cleaned_plant_data["PLZ"] = (
        cleaned_plant_data["PLZ"].str.replace(r"\.0$", "", regex=True).str.zfill(5)
    )
    for coordinate in ["Breitengrad", "Längengrad"]:
        cleaned_plant_data[coordinate] = convert_numeric_values(
            cleaned_plant_data[coordinate], file_path.name, coordinate
        )

    duplicate_plants = cleaned_plant_data.duplicated("Werksnummer", keep=False)
    if int(duplicate_plants.sum()) > 0:
        cleaning_log.append(
            {
                "Quelldatei": file_path.name,
                "Prüfung": "Duplicate plant numbers removed",
                "Anzahl": int(duplicate_plants.sum()),
            }
        )
        cleaned_plant_data = cleaned_plant_data.drop_duplicates("Werksnummer")

    cleaned_plant_data["Quelldatei"] = file_path.name
    retained_plant_data = cleaned_plant_data[
        [
            "Werksnummer", "Werk", "PLZ", "ORT", "Breitengrad",
            "Längengrad", "Werksebene",
        ]
    ].copy()
    record_cleaning_summary(
        file_path.name,
        "Plant data",
        raw_plant_data,
        cleaned_plant_data,
        retained_plant_data,
    )
    plant_frames.append(retained_plant_data)
    del raw_plant_data, cleaned_plant_data, retained_plant_data

plants_clean = pd.concat(plant_frames, ignore_index=True)

PLANT_COLUMNS = ["Werksnummer", "Werk", "PLZ", "ORT", "Breitengrad", "Längengrad"]


def plants_by_level(level):
    """Return one unique plant row for the requested supply-chain level."""
    return (
        plants_clean.loc[plants_clean["Werksebene"] == level, PLANT_COLUMNS]
        .drop_duplicates("Werksnummer")
    )


display_cleaning_summary("Plant data")
display(plants_clean.head())
display(plants_clean.groupby("Werksebene").size().rename("Number of plants").to_frame())

,Data level,Source file,Raw shape,Cleaned / validated shape,Retained shape,Columns before,Columns after cleaning,Columns retained,Removed / reshaped during cleaning,Added during preparation,Dropped after validation,Dtypes before,Dtypes after cleaning,Dtypes retained
0,Plant data,OEM_Werke_2017-07-04_TrR.csv,5 x 5,5 x 8,5 x 7,"PLZ , ORT, Werk, Breitengrad, L„ngengrad","PLZ, ORT, Werk, Breitengrad, Längengrad, Werksebene, Werksnummer, Quelldatei","Werksnummer, Werk, PLZ, ORT, Breitengrad, Längengrad, Werksebene","PLZ , L„ngengrad","PLZ, Längengrad, Werksebene, Werksnummer, Quelldatei",Quelldatei,"PLZ : string, ORT: string, Werk: string, Breitengrad: string, L„ngengrad: string","PLZ: string, ORT: string, Werk: string, Breitengrad: Float64, Längengrad: Float64, Werksebene: str, Werksnummer: str...","Werksnummer: string, Werk: string, PLZ: string, ORT: string, Breitengrad: Float64, Längengrad: Float64, Werksebene: str"
1,Plant data,Tier1_Werke_2017-07-11_v1.2_TrR.csv,45 x 5,22 x 8,22 x 7,"PLZ , ORT, Werk, Breitengrad, L„ngengrad","PLZ, ORT, Werk, Breitengrad, Längengrad, Werksebene, Werksnummer, Quelldatei","Werksnummer, Werk, PLZ, ORT, Breitengrad, Längengrad, Werksebene","PLZ , L„ngengrad","PLZ, Längengrad, Werksebene, Werksnummer, Quelldatei",Quelldatei,"PLZ : string, ORT: string, Werk: string, Breitengrad: string, L„ngengrad: string","PLZ: string, ORT: string, Werk: string, Breitengrad: Float64, Längengrad: Float64, Werksebene: str, Werksnummer: str...","Werksnummer: string, Werk: string, PLZ: string, ORT: string, Breitengrad: Float64, Längengrad: Float64, Werksebene: str"
2,Plant data,Tier2_Werke_2017-07-11_v1.2_TrR.csv,45 x 18,45 x 8,45 x 7,"PLZ, ORT , Werk, Breitengrad, L„ngengrad, Unnamed: 5, Unnamed: 6, Unnamed: 7, Unnamed: 8, Unnamed: 9, Unnamed: 10, U...","PLZ, ORT, Werk, Breitengrad, Längengrad, Werksebene, Werksnummer, Quelldatei","Werksnummer, Werk, PLZ, ORT, Breitengrad, Längengrad, Werksebene","ORT , L„ngengrad, Unnamed: 5, Unnamed: 6, Unnamed: 7, Unnamed: 8, Unnamed: 9, Unnamed: 10, Unnamed: 11, Unnamed: 12,...","ORT, Längengrad, Werksebene, Werksnummer, Quelldatei",Quelldatei,"PLZ: string, ORT : string, Werk: string, Breitengrad: string, L„ngengrad: string, Unnamed: 5: string, Unnamed: 6: st...","PLZ: string, ORT: string, Werk: string, Breitengrad: Float64, Längengrad: Float64, Werksebene: str, Werksnummer: str...","Werksnummer: string, Werk: string, PLZ: string, ORT: string, Breitengrad: Float64, Längengrad: Float64, Werksebene: str"


,Werksnummer,Werk,PLZ,ORT,Breitengrad,Längengrad,Werksebene
0,11,O11,90491,NUERNBERG,49.46699,11.107353,OEM
1,12,O12,53225,BONN,50.742015,7.120073,OEM
2,13,O13,33607,BIELEFELD,52.022568,8.540961,OEM
3,21,O21,37073,GOETTINGEN,51.535703,9.932804,OEM
4,22,O22,93051,REGENSBURG,49.008761,12.080696,OEM


,Number of plants
Werksebene,
OEM,5
Tier 1,22
Tier 2,45


## 6.4 Vehicle Data

The Type 11 and Type 21 vehicle files are cleaned with the same rules and combined. Each file is read and processed separately so that both raw vehicle files are not held in memory at the same time. Technical export columns and duplicate `.x` / `.y` fields are removed before dates, keys, defect flags, and defect mileage are converted.

The OEM plant table is merged on `Werksnummer`. This is a many-to-one merge: many vehicles can be produced at one plant, while each plant number occurs once in the reference table. The cleaned and merged file is documented before the next vehicle file is read. The source-file label is only needed for this documentation and is dropped from the retained vehicle table.

In [21]:
oem_plants = plants_by_level("OEM")
oem_plant_numbers = set(oem_plants["Werksnummer"].dropna())
vehicle_frames = []

for vehicle_type, file_path in vehicle_data_paths.items():
    print(f"Cleaning {file_path.name} ...")
    raw_vehicle_data = read_source_file(file_path)
    cleaned_vehicle_data = standardize_raw_values(raw_vehicle_data)
    cleaned_vehicle_data = remove_export_columns(cleaned_vehicle_data)
    cleaned_vehicle_data = combine_duplicate_columns(
        cleaned_vehicle_data, file_path.name
    )
    cleaned_vehicle_data = standardize_dates(
        cleaned_vehicle_data, file_path.name
    )
    cleaned_vehicle_data = correct_keys_from_id(
        cleaned_vehicle_data, "ID_Fahrzeug", file_path.name
    )
    cleaned_vehicle_data["Fehlerhaft"] = convert_numeric_values(
        cleaned_vehicle_data["Fehlerhaft"],
        file_path.name,
        "Fehlerhaft",
        integer=True,
        allowed_values={0, 1},
    )
    cleaned_vehicle_data["Fehlerhaft_Fahrleistung"] = convert_numeric_values(
        cleaned_vehicle_data["Fehlerhaft_Fahrleistung"],
        file_path.name,
        "Fehlerhaft_Fahrleistung",
        non_negative=True,
    )
    cleaned_vehicle_data["Fahrzeugtyp"] = vehicle_type
    cleaned_vehicle_data["OEM"] = "OEM1" if vehicle_type == "Type 11" else "OEM2"
    cleaned_vehicle_data["Quelldatei"] = file_path.name

    cleaned_vehicle_data = cleaned_vehicle_data.merge(
        oem_plants,
        on="Werksnummer",
        how="left",
        validate="many_to_one",
    )
    record_post_cleaning_quality(
        "vehicles_clean",
        file_path.name,
        cleaned_vehicle_data,
        oem_plant_numbers,
    )
    retained_vehicle_data = cleaned_vehicle_data[
        [
            "ID_Fahrzeug", "Fahrzeugtyp", "OEM", "Produktionsdatum",
            "Herstellernummer", "Werksnummer", "Fehlerhaft", "Fehlerhaft_Datum",
            "Fehlerhaft_Fahrleistung", "Werk", "PLZ", "ORT", "Breitengrad",
            "Längengrad",
        ]
    ].copy()
    record_cleaning_summary(
        file_path.name,
        "Vehicle data",
        raw_vehicle_data,
        cleaned_vehicle_data,
        retained_vehicle_data,
    )
    vehicle_frames.append(retained_vehicle_data)
    del raw_vehicle_data, cleaned_vehicle_data, retained_vehicle_data

vehicles_clean = pd.concat(vehicle_frames, ignore_index=True)
del vehicle_frames

display_cleaning_summary("Vehicle data")
display(vehicles_clean.head())
display(
    vehicles_clean.groupby("Fahrzeugtyp").size()
    .rename("Number of vehicles").to_frame()
)

Cleaning Fahrzeuge_OEM1_Typ11.csv ...
Cleaning Fahrzeuge_OEM2_Typ21.csv ...


,Data level,Source file,Raw shape,Cleaned / validated shape,Retained shape,Columns before,Columns after cleaning,Columns retained,Removed / reshaped during cleaning,Added during preparation,Dropped after validation,Dtypes before,Dtypes after cleaning,Dtypes retained
0,Vehicle data,Fahrzeuge_OEM1_Typ11.csv,"1,977,164 x 9","1,977,164 x 15","1,977,164 x 14","Unnamed: 0, X1, ID_Fahrzeug, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerha...","ID_Fahrzeug, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung,...","ID_Fahrzeug, Fahrzeugtyp, OEM, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehler...","Unnamed: 0, X1","Fahrzeugtyp, OEM, Quelldatei, Werk, PLZ, ORT, Breitengrad, Längengrad",Quelldatei,"Unnamed: 0: string, X1: string, ID_Fahrzeug: string, Produktionsdatum: string, Herstellernummer: string, Werksnummer...","ID_Fahrzeug: string, Produktionsdatum: datetime64[us], Herstellernummer: string, Werksnummer: string, Fehlerhaft: In...","ID_Fahrzeug: string, Fahrzeugtyp: str, OEM: str, Produktionsdatum: datetime64[us], Herstellernummer: string, Werksnu..."
1,Vehicle data,Fahrzeuge_OEM2_Typ21.csv,"512,354 x 10","512,354 x 15","512,354 x 14","Unnamed: 0, X1, ID_Fahrzeug, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, P...","ID_Fahrzeug, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Produktionsdatum,...","ID_Fahrzeug, Fahrzeugtyp, OEM, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehler...","Unnamed: 0, X1, Produktionsdatum_Origin_01011970, origin","Produktionsdatum, Fahrzeugtyp, OEM, Quelldatei, Werk, PLZ, ORT, Breitengrad, Längengrad",Quelldatei,"Unnamed: 0: string, X1: string, ID_Fahrzeug: string, Herstellernummer: string, Werksnummer: string, Fehlerhaft: stri...","ID_Fahrzeug: string, Herstellernummer: string, Werksnummer: string, Fehlerhaft: Int64, Fehlerhaft_Datum: datetime64[...","ID_Fahrzeug: string, Fahrzeugtyp: str, OEM: str, Produktionsdatum: datetime64[ns], Herstellernummer: string, Werksnu..."


,ID_Fahrzeug,Fahrzeugtyp,OEM,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung,Werk,PLZ,ORT,Breitengrad,Längengrad
0,11-1-11-1,Type 11,OEM1,2008-11-18,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353
1,11-1-11-2,Type 11,OEM1,2008-11-18,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353
2,11-1-11-3,Type 11,OEM1,2008-11-19,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353
3,11-1-11-4,Type 11,OEM1,2008-11-19,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353
4,11-1-11-5,Type 11,OEM1,2008-11-19,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353


,Number of vehicles
Fahrzeugtyp,
Type 11,1977164
Type 21,512354


## 6.5 Vehicle-Component Mapping

Each mapping file already contains one row per vehicle and four component ID columns. This structure matches the planned final dataset, so it is kept wide. We thought about converting it into a long table but that would create four rows per vehicle and would later have to be reversed.

The cleaning therefore removes technical columns, checks that all four role columns are present, and combines both vehicle types into `vehicle_components_wide_clean`. Component IDs remain temporary merge keys and are removed after the integration in Section 7. Vehicle type and source-file labels are used for the per-file summary only; the retained mapping contains just the vehicle ID and four component IDs.

In [22]:
vehicle_component_frames = []
vehicle_mapping_counts = []
component_id_columns = [
    "ID_Karosserie",
    "ID_Schaltung",
    "ID_Sitze",
    "ID_Motor",
]

for vehicle_type, file_path in vehicle_mapping_paths.items():
    print(f"Cleaning {file_path.name} ...")
    raw_mapping_data = read_source_file(file_path)
    cleaned_mapping_data = standardize_raw_values(raw_mapping_data)
    cleaned_mapping_data = remove_export_columns(cleaned_mapping_data)

    missing_columns = [
        column for column in component_id_columns
        if column not in cleaned_mapping_data.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Missing component columns in {file_path.name}: {missing_columns}"
        )

    cleaned_mapping_data["Fahrzeugtyp"] = vehicle_type
    cleaned_mapping_data["Quelldatei"] = file_path.name
    cleaned_mapping_data = cleaned_mapping_data[
        ["ID_Fahrzeug", "Fahrzeugtyp", *component_id_columns, "Quelldatei"]
    ]
    retained_mapping_data = cleaned_mapping_data[
        ["ID_Fahrzeug", *component_id_columns]
    ].copy()

    record_cleaning_summary(
        file_path.name,
        "Vehicle-component mapping",
        raw_mapping_data,
        cleaned_mapping_data,
        retained_mapping_data,
    )
    vehicle_component_frames.append(retained_mapping_data)
    vehicle_mapping_counts.append(
        {
            "Vehicle type": vehicle_type,
            "Vehicles with component mapping": len(retained_mapping_data),
        }
    )
    del raw_mapping_data, cleaned_mapping_data, retained_mapping_data

vehicle_components_wide_clean = pd.concat(
    vehicle_component_frames,
    ignore_index=True,
)
del vehicle_component_frames

display_cleaning_summary("Vehicle-component mapping")
display(vehicle_components_wide_clean.head())
display(pd.DataFrame(vehicle_mapping_counts))


Cleaning Bestandteile_Fahrzeuge_OEM1_Typ11.csv ...
Cleaning Bestandteile_Fahrzeuge_OEM2_Typ21.csv ...


,Data level,Source file,Raw shape,Cleaned / validated shape,Retained shape,Columns before,Columns after cleaning,Columns retained,Removed / reshaped during cleaning,Added during preparation,Dropped after validation,Dtypes before,Dtypes after cleaning,Dtypes retained
0,Vehicle-component mapping,Bestandteile_Fahrzeuge_OEM1_Typ11.csv,"1,977,164 x 6","1,977,164 x 7","1,977,164 x 5","Unnamed: 0, ID_Karosserie, ID_Schaltung, ID_Sitze, ID_Motor, ID_Fahrzeug","ID_Fahrzeug, Fahrzeugtyp, ID_Karosserie, ID_Schaltung, ID_Sitze, ID_Motor, Quelldatei","ID_Fahrzeug, ID_Karosserie, ID_Schaltung, ID_Sitze, ID_Motor",Unnamed: 0,"Fahrzeugtyp, Quelldatei","Fahrzeugtyp, Quelldatei","Unnamed: 0: string, ID_Karosserie: string, ID_Schaltung: string, ID_Sitze: string, ID_Motor: string, ID_Fahrzeug: st...","ID_Fahrzeug: string, Fahrzeugtyp: str, ID_Karosserie: string, ID_Schaltung: string, ID_Sitze: string, ID_Motor: stri...","ID_Fahrzeug: string, ID_Karosserie: string, ID_Schaltung: string, ID_Sitze: string, ID_Motor: string"
1,Vehicle-component mapping,Bestandteile_Fahrzeuge_OEM2_Typ21.csv,"512,354 x 6","512,354 x 7","512,354 x 5","Unnamed: 0, ID_Karosserie, ID_Schaltung, ID_Sitze, ID_Motor, ID_Fahrzeug","ID_Fahrzeug, Fahrzeugtyp, ID_Karosserie, ID_Schaltung, ID_Sitze, ID_Motor, Quelldatei","ID_Fahrzeug, ID_Karosserie, ID_Schaltung, ID_Sitze, ID_Motor",Unnamed: 0,"Fahrzeugtyp, Quelldatei","Fahrzeugtyp, Quelldatei","Unnamed: 0: string, ID_Karosserie: string, ID_Schaltung: string, ID_Sitze: string, ID_Motor: string, ID_Fahrzeug: st...","ID_Fahrzeug: string, Fahrzeugtyp: str, ID_Karosserie: string, ID_Schaltung: string, ID_Sitze: string, ID_Motor: stri...","ID_Fahrzeug: string, ID_Karosserie: string, ID_Schaltung: string, ID_Sitze: string, ID_Motor: string"


,ID_Fahrzeug,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor
0,11-1-11-1,K4-112-1121-3,K3SG1-105-1051-32,K2LE1-109-1091-2,K1BE1-101-1011-7
1,11-1-11-2,K4-112-1121-4,K3SG1-105-1051-141,K2ST1-109-1092-5,K1BE1-101-1011-12
2,11-1-11-3,K4-112-1121-7,K3SG1-105-1051-106,K2ST1-109-1092-57,K1BE1-101-1011-38
3,11-1-11-4,K4-112-1121-9,K3SG1-105-1051-21,K2ST1-109-1092-91,K1BE1-101-1011-97
4,11-1-11-5,K4-112-1121-11,K3SG1-105-1051-59,K2ST1-109-1092-4,K1BE1-101-1011-65


,Vehicle type,Vehicles with component mapping
0,Type 11,1977164
1,Type 21,512354


## 6.6 Component Data

The 14 component files are cleaned one after another. Dates, mileage, IDs, duplicates, and defect flags are checked while the complete file is available. The Cleaning Summary documents this full result.

Only the fields required for the final analysis are retained afterwards: component ID, type, role, Tier-1 plant, and direct defect status. The Tier-1 merge uses `Werksnummer` and is validated as many-to-one. Postal codes and coordinates are not copied to millions of component rows because the final component analysis only requires the supplier plant and city.

In [23]:
tier_1_plants = plants_by_level("Tier 1")[
    ["Werksnummer", "Werk", "ORT"]
]
tier_1_plant_numbers = set(tier_1_plants["Werksnummer"].dropna())
component_frames = []

for component_type, component_role in component_configuration.items():
    source_file = component_paths[f"{component_type} production"]
    print(f"Cleaning {source_file.name} ...")
    raw_component_data = read_source_file(source_file)
    cleaned_component_data = standardize_raw_values(raw_component_data)
    cleaned_component_data = remove_export_columns(cleaned_component_data)
    cleaned_component_data = combine_duplicate_columns(
        cleaned_component_data,
        source_file.name,
    )

    component_id_columns_found = [
        column
        for column in cleaned_component_data.columns
        if column.startswith("ID_")
    ]
    if len(component_id_columns_found) != 1:
        raise ValueError(f"Expected one component ID column in {source_file.name}.")

    component_id_column = component_id_columns_found[0]
    cleaned_component_data = standardize_dates(
        cleaned_component_data,
        source_file.name,
    )
    cleaned_component_data = correct_keys_from_id(
        cleaned_component_data,
        component_id_column,
        source_file.name,
    )
    cleaned_component_data["Fehlerhaft"] = convert_numeric_values(
        cleaned_component_data["Fehlerhaft"],
        source_file.name,
        "Fehlerhaft",
        integer=True,
        allowed_values={0, 1},
    )
    cleaned_component_data["Fehlerhaft_Fahrleistung"] = convert_numeric_values(
        cleaned_component_data["Fehlerhaft_Fahrleistung"],
        source_file.name,
        "Fehlerhaft_Fahrleistung",
        non_negative=True,
    )
    cleaned_component_data = consolidate_duplicate_ids(
        cleaned_component_data,
        component_id_column,
        source_file.name,
    )

    cleaned_component_data = cleaned_component_data.rename(
        columns={component_id_column: "ID_Komponente"}
    )
    cleaned_component_data["Komponententyp"] = component_type
    cleaned_component_data["Komponentenrolle"] = component_role
    cleaned_component_data["Quelldatei"] = source_file.name
    cleaned_component_data = cleaned_component_data.merge(
        tier_1_plants,
        on="Werksnummer",
        how="left",
        validate="many_to_one",
    )

    record_post_cleaning_quality(
        "components_clean",
        source_file.name,
        cleaned_component_data,
        tier_1_plant_numbers,
    )

    # The converted dates and mileage were checked above. They are not needed
    # for the final Vehicle-Level analysis and are not retained in memory.
    retained_component_data = cleaned_component_data[
        [
            "ID_Komponente", "Komponententyp", "Komponentenrolle",
            "Werksnummer", "Fehlerhaft", "Werk", "ORT",
        ]
    ].copy()
    record_cleaning_summary(
        source_file.name,
        "Component data",
        raw_component_data,
        cleaned_component_data,
        retained_component_data,
    )
    component_frames.append(retained_component_data)
    del raw_component_data, cleaned_component_data, retained_component_data

components_clean = pd.concat(component_frames, ignore_index=True)
del component_frames

display_cleaning_summary("Component data")
display(components_clean.head())
display(
    components_clean.groupby("Komponententyp").size()
    .rename("Number of components").to_frame()
)

Cleaning Komponente_K4.csv ...
Cleaning Komponente_K3AG1.csv ...
Cleaning Komponente_K3SG1.csv ...
Cleaning Komponente_K2LE1.txt ...
Cleaning Komponente_K2ST1.txt ...
Cleaning Komponente_K1BE1.csv ...
Cleaning Komponente_K1DI1.csv ...
Cleaning Komponente_K6.csv ...
Cleaning Komponente_K3AG2.txt ...
Cleaning Komponente_K3SG2.csv ...
Cleaning Komponente_K2LE2.txt ...
Cleaning Komponente_K2ST2.csv ...
Cleaning Komponente_K1BE2.csv ...
Cleaning Komponente_K1DI2.txt ...


,Data level,Source file,Raw shape,Cleaned / validated shape,Retained shape,Columns before,Columns after cleaning,Columns retained,Removed / reshaped during cleaning,Added during preparation,Dropped after validation,Dtypes before,Dtypes after cleaning,Dtypes retained
0,Component data,Komponente_K4.csv,"1,977,164 x 16","1,977,164 x 12","1,977,164 x 7","Unnamed: 0, X1, ID_Karosserie.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Dat...","ID_Komponente, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","ID_Komponente, Komponententyp, Komponentenrolle, Werksnummer, Fehlerhaft, Werk, ORT","Unnamed: 0, X1, ID_Karosserie.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Dat...","ID_Komponente, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","Produktionsdatum, Herstellernummer, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Quelldatei","Unnamed: 0: string, X1: string, ID_Karosserie.x: string, Produktionsdatum.x: string, Herstellernummer.x: string, Wer...","ID_Komponente: string, Produktionsdatum: datetime64[us], Herstellernummer: string, Werksnummer: string, Fehlerhaft: ...","ID_Komponente: string, Komponententyp: str, Komponentenrolle: str, Werksnummer: string, Fehlerhaft: Int64, Werk: str..."
1,Component data,Komponente_K3AG1.csv,"477,052 x 23","477,052 x 12","477,052 x 7","Unnamed: 0, X1, ID_Schaltung.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datu...","ID_Komponente, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","ID_Komponente, Komponententyp, Komponentenrolle, Werksnummer, Fehlerhaft, Werk, ORT","Unnamed: 0, X1, ID_Schaltung.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datu...","ID_Komponente, Komponententyp, Komponentenrolle, Quelldatei, Werk, ORT","Produktionsdatum, Herstellernummer, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Quelldatei","Unnamed: 0: string, X1: string, ID_Schaltung.x: string, Produktionsdatum.x: string, Herstellernummer.x: string, Werk...","ID_Komponente: string, Produktionsdatum: datetime64[us], Herstellernummer: string, Werksnummer: string, Fehlerhaft: ...","ID_Komponente: string, Komponententyp: str, Komponentenrolle: str, Werksnummer: string, Fehlerhaft: Int64, Werk: str..."
2,Component data,Komponente_K3SG1.csv,"1,908,208 x 16","1,908,208 x 12","1,908,208 x 7","Unnamed: 0, X1, ID_Schaltung.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datu...","ID_Komponente, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","ID_Komponente, Komponententyp, Komponentenrolle, Werksnummer, Fehlerhaft, Werk, ORT","Unnamed: 0, X1, ID_Schaltung.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datu...","ID_Komponente, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","Produktionsdatum, Herstellernummer, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Quelldatei","Unnamed: 0: string, X1: string, ID_Schaltung.x: string, Produktionsdatum.x: string, Herstellernummer.x: string, Werk...","ID_Komponente: string, Produktionsdatum: datetime64[us], Herstellernummer: string, Werksnummer: string, Fehlerhaft: ...","ID_Komponente: string, Komponententyp: str, Komponentenrolle: str, Werksnummer: string, Fehlerhaft: Int64, Werk: str..."
3,Component data,Komponente_K2LE1.txt,"477,052 x 16","477,052 x 12","477,052 x 7","_Zeilenindex, X1, ID_Sitze.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datum....","ID_Komponente, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","ID_Komponente, Komponententyp, Komponentenrolle, Werksnummer, Fehlerhaft, Werk, ORT","_Zeilenindex, X1, 

,ID_Komponente,Komponententyp,Komponentenrolle,Werksnummer,Fehlerhaft,Werk,ORT
0,K4-112-1121-3,K4,Karosserie,1121,0,1121,BRAUNSCHWEIG
1,K4-112-1121-4,K4,Karosserie,1121,0,1121,BRAUNSCHWEIG
2,K4-112-1121-7,K4,Karosserie,1121,0,1121,BRAUNSCHWEIG
3,K4-112-1121-9,K4,Karosserie,1121,0,1121,BRAUNSCHWEIG
4,K4-112-1121-11,K4,Karosserie,1121,0,1121,BRAUNSCHWEIG


,Number of components
Komponententyp,
K1BE1,1192630
K1BE2,409422
K1DI1,1192630
K1DI2,409422
K2LE1,477052
K2LE2,163769
K2ST1,1908208
K2ST2,655075
K3AG1,477052


## 6.7 Part Data

The part files are cleaned before the component-part mappings are processed. This order allows the mappings to use a ready part-quality lookup and prevents the complete assignment table from being stored.

Each part file is fully cleaned and validated first. The file-specific ID is renamed to `ID_Einzelteil`, duplicates are consolidated, and dates, mileage, plant keys, and defect flags are checked. Tier-2 plant numbers are validated with `isin` instead of copying plant names and coordinates to every part row. The Cleaning Summary shows both the fully validated file and the retained two-column lookup. Only `ID_Einzelteil` and `Fehlerhaft` are kept because these are the only part fields required for the Bottom-up aggregation.

In [24]:
tier_2_plant_numbers = set(
    plants_by_level("Tier 2")["Werksnummer"].dropna()
)
part_frames = []
part_counts = []

for part_type, source_file in part_paths.items():
    print(f"Cleaning {source_file.name} ...")
    raw_part_data = read_source_file(source_file)
    cleaned_part_data = standardize_raw_values(raw_part_data)
    cleaned_part_data = remove_export_columns(cleaned_part_data)
    cleaned_part_data = combine_duplicate_columns(
        cleaned_part_data,
        source_file.name,
    )

    part_id_columns = [
        column
        for column in cleaned_part_data.columns
        if column.startswith("ID_")
    ]
    if len(part_id_columns) != 1:
        raise ValueError(f"Expected one part ID column in {source_file.name}.")

    part_id_column = part_id_columns[0]
    cleaned_part_data = standardize_dates(
        cleaned_part_data,
        source_file.name,
    )
    cleaned_part_data = correct_keys_from_id(
        cleaned_part_data,
        part_id_column,
        source_file.name,
    )
    cleaned_part_data["Fehlerhaft"] = convert_numeric_values(
        cleaned_part_data["Fehlerhaft"],
        source_file.name,
        "Fehlerhaft",
        integer=True,
        allowed_values={0, 1},
    )
    cleaned_part_data["Fehlerhaft_Fahrleistung"] = convert_numeric_values(
        cleaned_part_data["Fehlerhaft_Fahrleistung"],
        source_file.name,
        "Fehlerhaft_Fahrleistung",
        non_negative=True,
    )
    cleaned_part_data = consolidate_duplicate_ids(
        cleaned_part_data,
        part_id_column,
        source_file.name,
    )
    cleaned_part_data = cleaned_part_data.rename(
        columns={part_id_column: "ID_Einzelteil"}
    )
    cleaned_part_data["Einzelteiltyp"] = part_type
    cleaned_part_data["Quelldatei"] = source_file.name

    record_post_cleaning_quality(
        "parts_clean",
        source_file.name,
        cleaned_part_data,
        tier_2_plant_numbers,
    )
    retained_part_data = cleaned_part_data[
        ["ID_Einzelteil", "Fehlerhaft"]
    ].copy()
    record_cleaning_summary(
        source_file.name,
        "Part data",
        raw_part_data,
        cleaned_part_data,
        retained_part_data,
    )

    part_frames.append(retained_part_data)
    part_counts.append(
        {"Part type": part_type, "Number of parts": len(retained_part_data)}
    )
    del raw_part_data, cleaned_part_data, retained_part_data

parts_clean = pd.concat(part_frames, ignore_index=True)
del part_frames

parts_clean_row_count = len(parts_clean)
parts_clean_missing_ids = int(parts_clean["ID_Einzelteil"].isna().sum())
parts_clean_ids_unique = not parts_clean["ID_Einzelteil"].duplicated().any()
assert parts_clean_missing_ids == 0
assert parts_clean_ids_unique

display_cleaning_summary("Part data")
display(parts_clean.head())
display(pd.DataFrame(part_counts))

Cleaning Einzelteil_T01.txt ...
Cleaning Einzelteil_T02.txt ...
Cleaning Einzelteil_T03.txt ...
Cleaning Einzelteil_T04.csv ...
Cleaning Einzelteil_T05.csv ...
Cleaning Einzelteil_T06.csv ...
Cleaning Einzelteil_T07.txt ...
Cleaning Einzelteil_T08.csv ...
Cleaning Einzelteil_T09.txt ...
Cleaning Einzelteil_T10.csv ...
Cleaning Einzelteil_T11.txt ...
Cleaning Einzelteil_T12.csv ...
Cleaning Einzelteil_T13.csv ...
Cleaning Einzelteil_T14.csv ...
Cleaning Einzelteil_T15.csv ...
Cleaning Einzelteil_T16.txt ...
Cleaning Einzelteil_T17.csv ...
Cleaning Einzelteil_T18.csv ...
Cleaning Einzelteil_T19.csv ...
Cleaning Einzelteil_T20.txt ...
Cleaning Einzelteil_T21.csv ...
Cleaning Einzelteil_T22.txt ...
Cleaning Einzelteil_T23.csv ...
Cleaning Einzelteil_T24.txt ...
Cleaning Einzelteil_T25.csv ...
Cleaning Einzelteil_T26.csv ...
Cleaning Einzelteil_T27.txt ...
Cleaning Einzelteil_T30.csv ...
Cleaning Einzelteil_T31.txt ...
Cleaning Einzelteil_T32.csv ...
Cleaning Einzelteil_T34.txt ...
Cleaning

,Data level,Source file,Raw shape,Cleaned / validated shape,Retained shape,Columns before,Columns after cleaning,Columns retained,Removed / reshaped during cleaning,Added during preparation,Dropped after validation,Dtypes before,Dtypes after cleaning,Dtypes retained
0,Part data,Einzelteil_T01.txt,"3,204,104 x 23","3,204,104 x 9","3,204,104 x 2","_Zeilenindex, X1, ID_T01.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datum.x,...","ID_Einzelteil, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","ID_Einzelteil, Fehlerhaft","_Zeilenindex, X1, ID_T01.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datum.x,...","ID_Einzelteil, Einzelteiltyp, Quelldatei","Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Einzelteiltyp, Quelldatei","_Zeilenindex: string, X1: string, ID_T01.x: string, Produktionsdatum.x: string, Herstellernummer.x: string, Werksnum...","ID_Einzelteil: string, Produktionsdatum: datetime64[us], Herstellernummer: string, Werksnummer: string, Fehlerhaft: ...","ID_Einzelteil: string, Fehlerhaft: Int64"
1,Part data,Einzelteil_T02.txt,"3,204,104 x 16","3,204,104 x 9","3,204,104 x 2","_Zeilenindex, X1, ID_T02.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datum.x,...","ID_Einzelteil, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","ID_Einzelteil, Fehlerhaft","_Zeilenindex, X1, ID_T02.x, Produktionsdatum.x, Herstellernummer.x, Werksnummer.x, Fehlerhaft.x, Fehlerhaft_Datum.x,...","ID_Einzelteil, Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistun...","Produktionsdatum, Herstellernummer, Werksnummer, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Einzelteiltyp, Quelldatei","_Zeilenindex: string, X1: string, ID_T02.x: string, Produktionsdatum.x: string, Herstellernummer.x: string, Werksnum...","ID_Einzelteil: string, Produktionsdatum: datetime64[us], Herstellernummer: string, Werksnummer: string, Fehlerhaft: ...","ID_Einzelteil: string, Fehlerhaft: Int64"
2,Part data,Einzelteil_T03.txt,"1,192,630 x 10","1,192,630 x 9","1,192,630 x 2","_Zeilenindex, X1, ID_T03, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Prod...","ID_Einzelteil, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Produktionsdatu...","ID_Einzelteil, Fehlerhaft","_Zeilenindex, X1, ID_T03, Produktionsdatum_Origin_01011970, origin","ID_Einzelteil, Produktionsdatum, Einzelteiltyp, Quelldatei","Herstellernummer, Werksnummer, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Produktionsdatum, Einzelteiltyp, Quelldatei","_Zeilenindex: string, X1: string, ID_T03: string, Herstellernummer: string, Werksnummer: string, Fehlerhaft: string,...","ID_Einzelteil: string, Herstellernummer: string, Werksnummer: string, Fehlerhaft: Int64, Fehlerhaft_Datum: datetime6...","ID_Einzelteil: string, Fehlerhaft: Int64"
3,Part data,Einzelteil_T04.csv,"1,192,630 x 10","1,192,630 x 9","1,192,630 x 2","Unnamed: 0, X1, ID_T04, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Produk...","ID_Einzelteil, Herstellernummer, Werksnummer, Fehlerhaft, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Produktionsdatu...","ID_Einzelteil, Fehlerhaft","Unnamed: 0, X1, ID_T04, Produktionsdatum_Origin_01011970, origin","ID_Einzelteil, Produktionsdatum, Einzelteiltyp, Quelldatei","Herstellernummer, Werksnummer, Fehlerhaft_Datum, Fehlerhaft_Fahrleistung, Produktionsdatum, Einzelteiltyp, Quelldatei","Unnamed: 0: string, X1: string, ID_T04: string, Herstellernummer: string, Werksnummer: string, Fehlerhaft: string, F...","ID_Einzelteil: string, Herstellernummer: string, Werksnummer: string, Fehlerhaft: Int64, Fehlerhaft_Datum: datetime6...","ID_Einzelteil: string, Fehlerhaft: Int64"
4,Part data,Einzelteil_T05.csv

,ID_Einzelteil,Fehlerhaft
0,1-201-2011-247,0
1,1-201-2011-429,0
2,1-201-2011-363,1
3,1-201-2011-30,0
4,1-201-2011-72,1


,Part type,Number of parts
0,T01,3204104
1,T02,3204104
2,T03,1192630
3,T04,1192630
4,T05,1192630
5,T06,1192630
6,T07,409422
7,T08,409422
8,T09,409422
9,T10,409422


## 6.8 Component-Part Mapping and Aggregation

Each component-part mapping file is cleaned and reshaped separately. Its part IDs are matched to the narrow `parts_clean` lookup on `ID_Einzelteil`. This is a many-to-one relationship because every cleaned part ID is unique.

The mapping is then grouped immediately by `ID_Komponente`. Only one aggregated result per component is retained: the combined part-defect status and the number of installed parts. The count is kept temporarily to confirm that every component received a mapping and is dropped after that check. The temporary long mapping for the current file is deleted before the next file is processed. This avoids a combined table of about 40 million assignment rows.

In [25]:
part_defect_lookup = (
    parts_clean.drop_duplicates("ID_Einzelteil")
    .set_index("ID_Einzelteil")["Fehlerhaft"]
)

component_part_aggregate_frames = []
component_part_mapping_counts = []
component_part_mapping_preview = None
unmatched_part_ids = 0

for component_type, expected_part_types in component_part_types.items():
    source_file = component_paths[f"{component_type} mapping"]
    print(f"Cleaning and aggregating {source_file.name} ...")
    raw_component_part_data = read_source_file(source_file)
    cleaned_component_part_data = standardize_raw_values(raw_component_part_data)
    cleaned_component_part_data = remove_export_columns(cleaned_component_part_data)
    component_id_column = f"ID_{component_type}"

    if component_id_column not in cleaned_component_part_data.columns:
        raise ValueError(f"Missing {component_id_column} in {source_file.name}.")

    file_mapping_frames = []
    for part_type in expected_part_types:
        part_number = int(part_type[1:])
        possible_names = [f"ID_T{part_number:02d}", f"ID_T{part_number}"]
        part_columns = [
            column
            for column in dict.fromkeys(possible_names)
            if column in cleaned_component_part_data.columns
        ]
        if len(part_columns) != 1:
            raise ValueError(f"Could not identify {part_type} in {source_file.name}.")

        one_part_mapping = cleaned_component_part_data[
            [component_id_column, part_columns[0]]
        ].rename(
            columns={
                component_id_column: "ID_Komponente",
                part_columns[0]: "ID_Einzelteil",
            }
        )
        one_part_mapping["Komponententyp"] = component_type
        one_part_mapping["Einzelteiltyp"] = part_type
        one_part_mapping["Quelldatei"] = source_file.name
        file_mapping_frames.append(one_part_mapping)

    cleaned_file_mapping = pd.concat(file_mapping_frames, ignore_index=True)
    del file_mapping_frames

    cleaned_file_mapping["Part_Direct_Defect"] = (
        cleaned_file_mapping["ID_Einzelteil"]
        .map(part_defect_lookup)
        .astype("Int64")
    )
    unmatched_part_ids += int(
        (
            cleaned_file_mapping["ID_Einzelteil"].notna()
            & ~cleaned_file_mapping["ID_Einzelteil"].isin(part_defect_lookup.index)
        ).sum()
    )
    cleaned_file_mapping["Part_Is_Defective"] = cleaned_file_mapping[
        "Part_Direct_Defect"
    ].eq(1)
    cleaned_file_mapping["Part_Status_Unknown"] = cleaned_file_mapping[
        "Part_Direct_Defect"
    ].isna()

    if component_part_mapping_preview is None:
        component_part_mapping_preview = cleaned_file_mapping.head().copy()

    component_part_aggregate = cleaned_file_mapping.groupby(
        "ID_Komponente",
        as_index=False,
    ).agg(
        Confirmed_Part_Defect=("Part_Is_Defective", "max"),
        Unknown_Part_Status=("Part_Status_Unknown", "max"),
        Installed_Part_Count=("ID_Einzelteil", "size"),
    )
    component_part_aggregate["Part_Defect"] = three_state_flag(
        component_part_aggregate["Confirmed_Part_Defect"],
        component_part_aggregate["Unknown_Part_Status"],
    )
    retained_component_part_data = component_part_aggregate[
        ["ID_Komponente", "Part_Defect", "Installed_Part_Count"]
    ].copy()

    record_cleaning_summary(
        source_file.name,
        "Component-part mapping",
        raw_component_part_data,
        cleaned_file_mapping,
        retained_component_part_data,
    )
    component_part_aggregate_frames.append(retained_component_part_data)
    component_part_mapping_counts.append(
        {
            "Component type": component_type,
            "Assignment rows processed": len(cleaned_file_mapping),
            "Components aggregated": len(retained_component_part_data),
        }
    )
    del (
        raw_component_part_data,
        cleaned_component_part_data,
        cleaned_file_mapping,
        component_part_aggregate,
        retained_component_part_data,
    )

assert unmatched_part_ids == 0

part_quality_by_component = pd.concat(
    component_part_aggregate_frames,
    ignore_index=True,
)
del component_part_aggregate_frames

component_part_mapping_total_rows = sum(
    row["Assignment rows processed"] for row in component_part_mapping_counts
)
assert not part_quality_by_component["ID_Komponente"].duplicated().any()

# The large part lookup is no longer needed after all mapping files are processed.
del parts_clean, part_defect_lookup

display_cleaning_summary("Component-part mapping")
display(component_part_mapping_preview)
display(part_quality_by_component.head())
display(pd.DataFrame(component_part_mapping_counts))


Cleaning and aggregating Bestandteile_Komponente_K4.csv ...
Cleaning and aggregating Bestandteile_Komponente_K3AG1.csv ...
Cleaning and aggregating Bestandteile_Komponente_K3SG1.csv ...
Cleaning and aggregating Bestandteile_Komponente_K2LE1.csv ...
Cleaning and aggregating Bestandteile_Komponente_K2ST1.csv ...
Cleaning and aggregating Bestandteile_Komponente_K1BE1.csv ...
Cleaning and aggregating Bestandteile_Komponente_K1DI1.csv ...
Cleaning and aggregating Bestandteile_Komponente_K6.csv ...
Cleaning and aggregating Bestandteile_Komponente_K3AG2.csv ...
Cleaning and aggregating Bestandteile_Komponente_K3SG2.csv ...
Cleaning and aggregating Bestandteile_Komponente_K2LE2.csv ...
Cleaning and aggregating Bestandteile_Komponente_K2ST2.csv ...
Cleaning and aggregating Bestandteile_Komponente_K1BE2.csv ...
Cleaning and aggregating Bestandteile_Komponente_K1DI2.csv ...


,Data level,Source file,Raw shape,Cleaned / validated shape,Retained shape,Columns before,Columns after cleaning,Columns retained,Removed / reshaped during cleaning,Added during preparation,Dropped after validation,Dtypes before,Dtypes after cleaning,Dtypes retained
0,Component-part mapping,Bestandteile_Komponente_K4.csv,"1,977,164 x 5","5,931,492 x 8","1,977,164 x 3","X1, ID_T30, ID_T31, ID_T32, ID_K4","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Komponente, Part_Defect, Installed_Part_Count","X1, ID_T30, ID_T31, ID_T32, ID_K4","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part_Status_Unknown","X1: string, ID_T30: string, ID_T31: string, ID_T32: string, ID_K4: string","ID_Komponente: string, ID_Einzelteil: string, Komponententyp: str, Einzelteiltyp: str, Quelldatei: str, Part_Direct_...","ID_Komponente: string, Part_Defect: Int64, Installed_Part_Count: int64"
1,Component-part mapping,Bestandteile_Komponente_K3AG1.csv,"477,052 x 5","1,431,156 x 8","477,052 x 3","X1, ID_T21, ID_T24, ID_T25, ID_K3AG1","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Komponente, Part_Defect, Installed_Part_Count","X1, ID_T21, ID_T24, ID_T25, ID_K3AG1","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part_Status_Unknown","X1: string, ID_T21: string, ID_T24: string, ID_T25: string, ID_K3AG1: string","ID_Komponente: string, ID_Einzelteil: string, Komponententyp: str, Einzelteiltyp: str, Quelldatei: str, Part_Direct_...","ID_Komponente: string, Part_Defect: Int64, Installed_Part_Count: int64"
2,Component-part mapping,Bestandteile_Komponente_K3SG1.csv,"1,908,208 x 5","5,724,624 x 8","1,908,208 x 3","X1, ID_T21, ID_T22, ID_T23, ID_K3SG1","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Komponente, Part_Defect, Installed_Part_Count","X1, ID_T21, ID_T22, ID_T23, ID_K3SG1","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part_Status_Unknown","X1: string, ID_T21: string, ID_T22: string, ID_T23: string, ID_K3SG1: string","ID_Komponente: string, ID_Einzelteil: string, Komponententyp: str, Einzelteiltyp: str, Quelldatei: str, Part_Direct_...","ID_Komponente: string, Part_Defect: Int64, Installed_Part_Count: int64"
3,Component-part mapping,Bestandteile_Komponente_K2LE1.csv,"477,052 x 5","1,431,156 x 8","477,052 x 3","Unnamed: 0, ID_T11, ID_T14, ID_T15, ID_K2LE1","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Komponente, Part_Defect, Installed_Part_Count","Unnamed: 0, ID_T11, ID_T14, ID_T15, ID_K2LE1","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part...","ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_Defect, Part_Is_Defective, Part_Status_Unknown","Unnamed: 0: string, ID_T11: string, ID_T14: string, ID_T15: string, ID_K2LE1: string","ID_Komponente: string, ID_Einzelteil: string, Komponententyp: str, Einzelteiltyp: str, Quelldatei: str, Part_Direct_...","ID_Komponente: string, Part_Defect: Int64, Installed_Part_Count: int64"
4,Component-part mapping,Bestandteile_Komponente_K2ST1.csv,"1,908,208 x 6","5,724,624 x 8","1,908,208 x 3","X1, X, ID_T11, ID_T12, ID_T13, ID_K2ST1","ID_Komponente, ID_Einzelteil, Komponententyp, Einzelteiltyp, Quelldatei, Part_Direct_De

,ID_Komponente,ID_Einzelteil,Komponententyp,Einzelteiltyp,Quelldatei,Part_Direct_Defect,Part_Is_Defective,Part_Status_Unknown
0,K4-112-1121-1,30-216-2161-242,K4,T30,Bestandteile_Komponente_K4.csv,1,True,False
1,K4-112-1121-2,30-216-2161-87,K4,T30,Bestandteile_Komponente_K4.csv,0,False,False
2,K4-112-1121-3,30-217-2171-214,K4,T30,Bestandteile_Komponente_K4.csv,0,False,False
3,K4-112-1121-4,30-216-2161-144,K4,T30,Bestandteile_Komponente_K4.csv,0,False,False
4,K4-112-1121-5,30-216-2161-71,K4,T30,Bestandteile_Komponente_K4.csv,0,False,False


,ID_Komponente,Part_Defect,Installed_Part_Count
0,K4-112-1121-1,1,3
1,K4-112-1121-10,1,3
2,K4-112-1121-100,0,3
3,K4-112-1121-1000,0,3
4,K4-112-1121-10000,0,3


,Component type,Assignment rows processed,Components aggregated
0,K4,5931492,1977164
1,K3AG1,1431156,477052
2,K3SG1,5724624,1908208
3,K2LE1,1431156,477052
4,K2ST1,5724624,1908208
5,K1BE1,4770520,1192630
6,K1DI1,4770520,1192630
7,K6,2049416,512354
8,K3AG2,491307,163769
9,K3SG2,1965225,655075


## 6.9 Cleaning and Validation

The validation is performed after all file groups have been cleaned. Wide source fields that are not required later were already checked per file before they were discarded. The first table reconciles row counts and keys for the retained working tables. The second table combines the per-file checks for dates, defect flags, mileage, and plant keys.

The validation covers:

- missing and duplicate main IDs,
- one wide component-mapping row and four filled component IDs per vehicle,
- one aggregated part-quality row per component,
- datetime and nullable-integer conversions before column reduction,
- invalid defect flags and negative mileage, and
- missing plant-key matches.

Missing defect flags are reported but remain valid because they represent an unknown quality status.

In [26]:
vehicle_component_id_columns = [
    "ID_Karosserie", "ID_Schaltung", "ID_Sitze", "ID_Motor"
]

intermediate_table_summary = pd.DataFrame(
    [
        {
            "Table": "plants_clean",
            "Rows": len(plants_clean),
            "Main key": "Werksebene + Werksnummer",
            "Missing main key": int(plants_clean["Werksnummer"].isna().sum()),
        },
        {
            "Table": "vehicles_clean",
            "Rows": len(vehicles_clean),
            "Main key": "ID_Fahrzeug",
            "Missing main key": int(vehicles_clean["ID_Fahrzeug"].isna().sum()),
        },
        {
            "Table": "vehicle_components_wide_clean",
            "Rows": len(vehicle_components_wide_clean),
            "Main key": "ID_Fahrzeug",
            "Missing main key": int(
                vehicle_components_wide_clean[
                    ["ID_Fahrzeug", *vehicle_component_id_columns]
                ].isna().any(axis=1).sum()
            ),
        },
        {
            "Table": "components_clean",
            "Rows": len(components_clean),
            "Main key": "ID_Komponente",
            "Missing main key": int(components_clean["ID_Komponente"].isna().sum()),
        },
        {
            "Table": "parts_clean (narrow lookup)",
            "Rows": parts_clean_row_count,
            "Main key": "ID_Einzelteil",
            "Missing main key": parts_clean_missing_ids,
        },
        {
            "Table": "part_quality_by_component",
            "Rows": len(part_quality_by_component),
            "Main key": "ID_Komponente",
            "Missing main key": int(
                part_quality_by_component["ID_Komponente"].isna().sum()
            ),
        },
    ]
)

assert not vehicles_clean["ID_Fahrzeug"].duplicated().any()
assert not components_clean["ID_Komponente"].duplicated().any()
assert parts_clean_ids_unique
assert vehicle_components_wide_clean["ID_Fahrzeug"].is_unique
assert len(vehicle_components_wide_clean) == len(vehicles_clean)
assert not vehicle_components_wide_clean[vehicle_component_id_columns].isna().any().any()
assert not part_quality_by_component["ID_Komponente"].duplicated().any()

display(intermediate_table_summary)

quality_detail = pd.DataFrame(post_cleaning_quality_rows)
post_cleaning_quality_summary = (
    quality_detail.groupby("Table", as_index=False)
    .agg(
        Source_files=("Source file", "nunique"),
        Production_dates_valid=("Production date is datetime", "all"),
        Defect_dates_valid=("Defect date is datetime", "all"),
        Defect_dtypes_valid=("Defect flag is Int64", "all"),
        Missing_defect_flags=("Missing defect flags", "sum"),
        Invalid_defect_flags=("Invalid defect flags", "sum"),
        Negative_mileage=("Negative mileage", "sum"),
        Missing_plant_matches=("Missing plant matches", "sum"),
    )
)
display(post_cleaning_quality_summary)

assert post_cleaning_quality_summary[
    ["Production_dates_valid", "Defect_dates_valid", "Defect_dtypes_valid"]
].all().all()
assert post_cleaning_quality_summary["Invalid_defect_flags"].eq(0).all()
assert post_cleaning_quality_summary["Negative_mileage"].eq(0).all()
assert post_cleaning_quality_summary["Missing_plant_matches"].eq(0).all()

if cleaning_log:
    cleaning_log_summary = (
        pd.DataFrame(cleaning_log)
        .groupby("Prüfung", as_index=False)["Anzahl"]
        .sum()
        .sort_values("Anzahl", ascending=False)
    )
    display(cleaning_log_summary)

,Table,Rows,Main key,Missing main key
0,plants_clean,72,Werksebene + Werksnummer,0
1,vehicles_clean,2489518,ID_Fahrzeug,0
2,vehicle_components_wide_clean,2489518,ID_Fahrzeug,0
3,components_clean,12101830,ID_Komponente,0
4,parts_clean (narrow lookup),41451120,ID_Einzelteil,0
5,part_quality_by_component,12101830,ID_Komponente,0


,Table,Source_files,Production_dates_valid,Defect_dates_valid,Defect_dtypes_valid,Missing_defect_flags,Invalid_defect_flags,Negative_mileage,Missing_plant_matches
0,components_clean,14,True,True,True,0,0,0,0
1,parts_clean,34,True,True,True,0,0,0,0
2,vehicles_clean,2,True,True,True,0,0,0,0


,Prüfung,Anzahl
1,Corrected Werksnummer from ID,4209692
0,Corrected Herstellernummer from ID,2194440
3,Negative Fehlerhaft_Fahrleistung converted to missing,17375
2,Incomplete plant rows removed,23


### 6.9.1 Cleaning Result

The retained tables contain 2,489,518 vehicles and the same number of vehicle-component mapping rows. All vehicle IDs are unique, and every vehicle has four component IDs. The 12,101,830 cleaned components also have 12,101,830 aggregated part-quality records. In addition, the narrow part lookup contains 41,451,120 unique part records. No main keys, defect flags, or plant matches are missing after cleaning.

The cleaning log keeps the corrections visible instead of silently changing the data. In the Tier-1 plant file, 22 completely empty rows and one incomplete row containing the Tier-2 plant number `2193` are removed. The complete `2193` record remains available in the Tier-2 plant file. In addition, 17,375 negative mileage values are converted to missing. The structured object IDs are used to correct 4,209,692 plant numbers and 2,194,440 manufacturer numbers, as defined in Section 3.5. Decimal-comma mileage values are standardised before conversion and therefore remain valid numeric observations. Overall, the retained tables are ready for the integration in Section 7.

<a id="section-7"></a>

# 7. Data Integration

The retained working tables are now connected from component level to vehicle level. Part quality has already been aggregated during Section 6.8, so the large component-part assignments do not have to be merged again.

The remaining integration has three stages:

1. Check the aggregated part result.
2. Combine it with the direct component defect.
3. Connect each of the four component roles to the wide vehicle mapping.

Every merge uses only the required columns and validates its expected relationship.

## 7.1 Aggregated Part Quality Input

`part_quality_by_component` contains one row per component ID. `Part_Defect` is 1 if at least one installed part is defective, 0 if all part statuses are known and non-defective, and missing if no defect is confirmed but at least one part status is unknown.

In [27]:
part_integration_summary = pd.DataFrame(
    [
        {
            "Integration check": "Component-part assignment rows processed",
            "Result": component_part_mapping_total_rows,
        },
        {
            "Integration check": "Part IDs without a matching part record",
            "Result": unmatched_part_ids,
        },
        {
            "Integration check": "Components with an aggregated part result",
            "Result": len(part_quality_by_component),
        },
    ]
)

assert unmatched_part_ids == 0
assert part_quality_by_component["ID_Komponente"].is_unique

display(part_quality_by_component.head())
part_integration_summary

,ID_Komponente,Part_Defect,Installed_Part_Count
0,K4-112-1121-1,1,3
1,K4-112-1121-10,1,3
2,K4-112-1121-100,0,3
3,K4-112-1121-1000,0,3
4,K4-112-1121-10000,0,3


,Integration check,Result
0,Component-part assignment rows processed,40021948
1,Part IDs without a matching part record,0
2,Components with an aggregated part result,12101830


## 7.2 Create an Effective Quality Status for Each Component

The aggregated part result is merged with the narrow `components_clean` table on `ID_Komponente`. Both tables contain at most one row per component ID, so this is a one-to-one merge.

The direct component flag and the aggregated part flag remain separate. The effective component flag becomes 1 if either source confirms a defect. It becomes 0 only when both sources are known and equal to 0, otherwise it stays missing. The installed-part count is used once to verify that every component received an aggregated mapping. It is not an analysis variable and is removed together with the separate component and part-aggregate tables after this check.

In [28]:
clean_component_count = len(components_clean)
component_quality_lookup = components_clean.merge(
    part_quality_by_component,
    on="ID_Komponente",
    how="left",
    validate="one_to_one",
).rename(
    columns={
        "Fehlerhaft": "Component_Direct_Defect",
        "Werksnummer": "Supplier_Plant_Number",
        "Werk": "Supplier_Plant",
        "ORT": "Supplier_City",
    }
)

components_without_part_mapping = int(
    component_quality_lookup["Installed_Part_Count"].isna().sum()
)
assert components_without_part_mapping == 0

confirmed_component_defect = (
    component_quality_lookup["Component_Direct_Defect"].eq(1)
    | component_quality_lookup["Part_Defect"].eq(1)
)
unknown_component_status = (
    component_quality_lookup["Component_Direct_Defect"].isna()
    | component_quality_lookup["Part_Defect"].isna()
)
component_quality_lookup["Component_Effective_Defect"] = three_state_flag(
    confirmed_component_defect,
    unknown_component_status,
)

component_quality_lookup = component_quality_lookup[
    [
        "ID_Komponente", "Komponententyp", "Komponentenrolle",
        "Supplier_Plant_Number", "Supplier_Plant", "Supplier_City",
        "Component_Direct_Defect", "Part_Defect",
        "Component_Effective_Defect",
    ]
]

component_integration_summary = pd.DataFrame(
    [
        {
            "Integration check": "Clean component records",
            "Result": clean_component_count,
        },
        {
            "Integration check": "Component records after part aggregation",
            "Result": len(component_quality_lookup),
        },
        {
            "Integration check": "Components without part aggregation",
            "Result": components_without_part_mapping,
        },
    ]
)

del components_clean, part_quality_by_component

display(component_quality_lookup.head())
component_integration_summary

,ID_Komponente,Komponententyp,Komponentenrolle,Supplier_Plant_Number,Supplier_Plant,Supplier_City,Component_Direct_Defect,Part_Defect,Component_Effective_Defect
0,K4-112-1121-3,K4,Karosserie,1121,1121,BRAUNSCHWEIG,0,0,0
1,K4-112-1121-4,K4,Karosserie,1121,1121,BRAUNSCHWEIG,0,0,0
2,K4-112-1121-7,K4,Karosserie,1121,1121,BRAUNSCHWEIG,0,0,0
3,K4-112-1121-9,K4,Karosserie,1121,1121,BRAUNSCHWEIG,0,0,0
4,K4-112-1121-11,K4,Karosserie,1121,1121,BRAUNSCHWEIG,0,0,0


,Integration check,Result
0,Clean component records,12101830
1,Component records after part aggregation,12101830
2,Components without part aggregation,0


## 7.3 Build the Wide Vehicle-Component Quality Table

Each of the four component ID columns in `vehicle_components_wide_clean` is connected separately to the component-quality lookup. The component merge is many-to-one because every component ID is unique in the lookup. Each role result still has one row per vehicle.

The four role results are first added to a component-only wide table. This table is then merged with the vehicle master only once in Section 8. Component IDs are removed after their merge; only component type, supplier plant, and the three quality flags remain.

In [29]:
ROLE_COMPONENT_COLUMNS = {
    "ID_Karosserie": ("Karosserie", "Body"),
    "ID_Schaltung": ("Schaltung", "Transmission"),
    "ID_Sitze": ("Sitze", "Seats"),
    "ID_Motor": ("Motor", "Engine"),
}
ROLE_PREFIXES = {
    component_role: prefix
    for component_role, prefix in ROLE_COMPONENT_COLUMNS.values()
}


def build_vehicle_role_table(component_id_column, component_role, prefix):
    """Connect one wide component ID column to quality and supplier fields."""
    role_assignments = vehicle_components_wide_clean[
        ["ID_Fahrzeug", component_id_column]
    ].rename(columns={component_id_column: "ID_Komponente"})
    role_assignments["Mapped_Component_Type"] = (
        role_assignments["ID_Komponente"]
        .astype("string")
        .str.extract(r"^([A-Za-z0-9]+)-", expand=False)
    )

    role_data = role_assignments.merge(
        component_quality_lookup,
        on="ID_Komponente",
        how="left",
        validate="many_to_one",
    )
    if role_data["ID_Fahrzeug"].duplicated().any():
        raise ValueError(f"More than one {component_role} assignment found per vehicle.")

    type_mismatch = (
        role_data["Mapped_Component_Type"].notna()
        & role_data["Komponententyp"].notna()
        & role_data["Mapped_Component_Type"].ne(role_data["Komponententyp"])
    )
    role_mismatch = (
        role_data["Komponentenrolle"].notna()
        & role_data["Komponentenrolle"].ne(component_role)
    )
    if type_mismatch.any() or role_mismatch.any():
        raise ValueError(f"Component type or role mismatch found for {component_role}.")

    role_data = role_data[
        [
            "ID_Fahrzeug", "Komponententyp", "Supplier_Plant_Number",
            "Supplier_Plant", "Supplier_City", "Component_Direct_Defect",
            "Part_Defect", "Component_Effective_Defect",
        ]
    ]
    return role_data.rename(
        columns={
            "ID_Fahrzeug": "Vehicle_ID",
            "Komponententyp": f"{prefix}_Type",
            "Supplier_Plant_Number": f"{prefix}_Supplier_Plant_Number",
            "Supplier_Plant": f"{prefix}_Supplier_Plant",
            "Supplier_City": f"{prefix}_Supplier_City",
            "Component_Direct_Defect": f"{prefix}_Direct_Defect",
            "Part_Defect": f"{prefix}_Part_Defect",
            "Component_Effective_Defect": f"{prefix}_Effective_Defect",
        }
    )


vehicle_component_quality_wide = vehicle_components_wide_clean[
    ["ID_Fahrzeug"]
].rename(columns={"ID_Fahrzeug": "Vehicle_ID"})

role_merge_checks = []
for component_id_column, (component_role, prefix) in ROLE_COMPONENT_COLUMNS.items():
    role_data = build_vehicle_role_table(
        component_id_column,
        component_role,
        prefix,
    )
    rows_before = len(vehicle_component_quality_wide)
    vehicle_component_quality_wide = vehicle_component_quality_wide.merge(
        role_data,
        on="Vehicle_ID",
        how="left",
        validate="one_to_one",
    )
    role_merge_checks.append(
        {
            "Merge": f"Vehicle mapping + {prefix}",
            "Key": "Vehicle_ID",
            "Rows before": rows_before,
            "Rows after": len(vehicle_component_quality_wide),
            "Vehicles without component match": int(
                vehicle_component_quality_wide[f"{prefix}_Type"].isna().sum()
            ),
        }
    )

    preview_columns = [
        "Vehicle_ID", f"{prefix}_Type", f"{prefix}_Supplier_Plant",
        f"{prefix}_Direct_Defect", f"{prefix}_Part_Defect",
        f"{prefix}_Effective_Defect",
    ]
    print(f"{prefix} merge preview")
    display(vehicle_component_quality_wide[preview_columns].head(3))
    del role_data

del vehicle_components_wide_clean, component_quality_lookup

display(pd.DataFrame(role_merge_checks))

Body merge preview


,Vehicle_ID,Body_Type,Body_Supplier_Plant,Body_Direct_Defect,Body_Part_Defect,Body_Effective_Defect
0,11-1-11-1,K4,1121,0,0,0
1,11-1-11-2,K4,1121,0,0,0
2,11-1-11-3,K4,1121,0,0,0


Transmission merge preview


,Vehicle_ID,Transmission_Type,Transmission_Supplier_Plant,Transmission_Direct_Defect,Transmission_Part_Defect,Transmission_Effective_Defect
0,11-1-11-1,K3SG1,1051,0,1,1
1,11-1-11-2,K3SG1,1051,0,1,1
2,11-1-11-3,K3SG1,1051,0,1,1


Seats merge preview


,Vehicle_ID,Seats_Type,Seats_Supplier_Plant,Seats_Direct_Defect,Seats_Part_Defect,Seats_Effective_Defect
0,11-1-11-1,K2LE1,1091,1,0,1
1,11-1-11-2,K2ST1,1092,1,0,1
2,11-1-11-3,K2ST1,1092,0,0,0


Engine merge preview


,Vehicle_ID,Engine_Type,Engine_Supplier_Plant,Engine_Direct_Defect,Engine_Part_Defect,Engine_Effective_Defect
0,11-1-11-1,K1BE1,1011,0,1,1
1,11-1-11-2,K1BE1,1011,0,1,1
2,11-1-11-3,K1BE1,1011,0,0,0


,Merge,Key,Rows before,Rows after,Vehicles without component match
0,Vehicle mapping + Body,Vehicle_ID,2489518,2489518,0
1,Vehicle mapping + Transmission,Vehicle_ID,2489518,2489518,0
2,Vehicle mapping + Seats,Vehicle_ID,2489518,2489518,0
3,Vehicle mapping + Engine,Vehicle_ID,2489518,2489518,0


## 7.4 Integration Result

In total, 40,021,948 component-part assignment rows were processed. Every part ID had a matching part record, and the result contains one aggregated part status for each of the 12,101,830 components. The component merge kept the same number of records and did not leave a component without a part result. The four role merges also kept all 2,489,518 vehicle rows, and no vehicle is missing a Body, Transmission, Seats, or Engine match. The integration therefore preserves the expected component and vehicle levels.

<a id="section-8"></a>

# 8. Creation and Validation of the Final Dataset

## 8.1 Plan for the Final Vehicle-Level Dataset

The final unit of analysis is the vehicle. Therefore, the dataset contains exactly one row per `Vehicle_ID`. This is detailed enough for the requested plant and component analysis.

The table is planned in four blocks:

1. **Vehicle information:** vehicle type, OEM, production date and year, direct defect flag, defect date, and defect mileage.
2. **OEM plant information:** plant number, name, postal code, city, and coordinates of the plant that produced the vehicle.
3. **Installed component information:** one column group for Body, Transmission, Seats, and Engine. Each group contains the component type, Tier-1 supplier plant, direct component defect, aggregated part defect, and effective component defect.
4. **Vehicle-level quality result:** any direct component defect, any part defect, any effective component defect, and the final effective vehicle defect.

Part IDs and component IDs are not included. They have already served their purpose as merge keys. A missing defect value is not changed to 0: it remains unknown unless another level confirms a defect.

## 8.2 Add the Four Component Roles to Each Vehicle

The cleaned vehicle table is first renamed to clear final column names. The already wide component-quality table from Section 7 is then merged once on `Vehicle_ID`. This is a one-to-one merge and must keep the original number of vehicle rows.

In [30]:
clean_vehicle_row_count = len(vehicles_clean)
final_vehicle_data = vehicles_clean.rename(
    columns={
        "ID_Fahrzeug": "Vehicle_ID",
        "Fahrzeugtyp": "Vehicle_Type",
        "Produktionsdatum": "Vehicle_Production_Date",
        "Herstellernummer": "Vehicle_Manufacturer_Number",
        "Werksnummer": "OEM_Plant_Number",
        "Fehlerhaft": "Vehicle_Direct_Defect",
        "Fehlerhaft_Datum": "Vehicle_Defect_Date",
        "Fehlerhaft_Fahrleistung": "Vehicle_Defect_Mileage",
        "Werk": "OEM_Plant",
        "PLZ": "OEM_Postal_Code",
        "ORT": "OEM_City",
        "Breitengrad": "OEM_Latitude",
        "Längengrad": "OEM_Longitude",
    }
)[
    [
        "Vehicle_ID", "Vehicle_Type", "OEM", "Vehicle_Production_Date",
        "Vehicle_Manufacturer_Number", "OEM_Plant_Number",
        "Vehicle_Direct_Defect", "Vehicle_Defect_Date", "Vehicle_Defect_Mileage",
        "OEM_Plant", "OEM_Postal_Code", "OEM_City", "OEM_Latitude",
        "OEM_Longitude",
    ]
].copy()
final_vehicle_data["Vehicle_Production_Year"] = final_vehicle_data[
    "Vehicle_Production_Date"
].dt.year.astype("Int64")

rows_before = len(final_vehicle_data)
final_vehicle_data = final_vehicle_data.merge(
    vehicle_component_quality_wide,
    on="Vehicle_ID",
    how="left",
    validate="one_to_one",
)

final_merge_check = pd.DataFrame(
    [
        {
            "Merge": "Vehicles + all component roles",
            "Key": "Vehicle_ID",
            "Expected relationship": "one-to-one",
            "Rows before": rows_before,
            "Rows after": len(final_vehicle_data),
            "Vehicles without complete component match": int(
                final_vehicle_data[
                    [f"{prefix}_Type" for prefix in ROLE_PREFIXES.values()]
                ].isna().any(axis=1).sum()
            ),
        }
    ]
)

del vehicles_clean, vehicle_component_quality_wide

display(final_merge_check)
display(final_vehicle_data.head(3))

,Merge,Key,Expected relationship,Rows before,Rows after,Vehicles without complete component match
0,Vehicles + all component roles,Vehicle_ID,one-to-one,2489518,2489518,0


,Vehicle_ID,Vehicle_Type,OEM,Vehicle_Production_Date,Vehicle_Manufacturer_Number,OEM_Plant_Number,Vehicle_Direct_Defect,Vehicle_Defect_Date,Vehicle_Defect_Mileage,OEM_Plant,OEM_Postal_Code,OEM_City,OEM_Latitude,OEM_Longitude,Vehicle_Production_Year,...,Transmission_Effective_Defect,Seats_Type,Seats_Supplier_Plant_Number,Seats_Supplier_Plant,Seats_Supplier_City,Seats_Direct_Defect,Seats_Part_Defect,Seats_Effective_Defect,Engine_Type,Engine_Supplier_Plant_Number,Engine_Supplier_Plant,Engine_Supplier_City,Engine_Direct_Defect,Engine_Part_Defect,Engine_Effective_Defect
0,11-1-11-1,Type 11,OEM1,2008-11-18,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353,2008,...,1,K2LE1,1091,1091,INGOLSTADT,1,0,1,K1BE1,1011,1011,MUENCHEN,0,1,1
1,11-1-11-2,Type 11,OEM1,2008-11-18,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353,2008,...,1,K2ST1,1092,1092,WUERZBURG,1,0,1,K1BE1,1011,1011,MUENCHEN,0,1,1
2,11-1-11-3,Type 11,OEM1,2008-11-19,1,11,0,NaT,0,O11,90491,NUERNBERG,49.46699,11.107353,2008,...,1,K2ST1,1092,1092,WUERZBURG,0,0,0,K1BE1,1011,1011,MUENCHEN,0,0,0


## 8.3 Calculate the Vehicle-Level Defect Flags

The role columns already contain the effective status of each component. The final flags are calculated across the four roles:

- `Any_Component_Direct_Defect` is 1 if at least one component itself is defective.
- `Any_Part_Defect` is 1 if at least one part in any component is defective.
- `Any_Component_Effective_Defect` combines the two previous sources.
- `Vehicle_Effective_Defect` is 1 if the vehicle itself, an installed component, or an installed part is defective.

The helper keeps the same three-state rule at every level: confirmed defect has priority, fully known non-defect becomes 0, and incomplete information without a confirmed defect remains missing.

In [31]:
def combine_defect_columns(data, columns):
    """Combine several 0/1/NA columns with the three-state defect rule."""
    confirmed = data[columns].eq(1).any(axis=1)
    unknown = data[columns].isna().any(axis=1)
    return three_state_flag(confirmed, unknown)


role_prefixes = list(ROLE_PREFIXES.values())
direct_component_columns = [f"{prefix}_Direct_Defect" for prefix in role_prefixes]
part_defect_columns = [f"{prefix}_Part_Defect" for prefix in role_prefixes]
effective_component_columns = [f"{prefix}_Effective_Defect" for prefix in role_prefixes]

final_vehicle_data["Any_Component_Direct_Defect"] = combine_defect_columns(
    final_vehicle_data, direct_component_columns
)
final_vehicle_data["Any_Part_Defect"] = combine_defect_columns(
    final_vehicle_data, part_defect_columns
)
final_vehicle_data["Any_Component_Effective_Defect"] = combine_defect_columns(
    final_vehicle_data, effective_component_columns
)
final_vehicle_data["Vehicle_Effective_Defect"] = combine_defect_columns(
    final_vehicle_data,
    ["Vehicle_Direct_Defect", "Any_Component_Effective_Defect"],
)

all_quality_columns = [
    "Vehicle_Direct_Defect",
    *direct_component_columns,
    *part_defect_columns,
]
final_vehicle_data["Data_Status"] = np.where(
    final_vehicle_data[all_quality_columns].isna().any(axis=1),
    "Partly unknown",
    "Complete",
)

# Put the summary fields directly after the vehicle and OEM information.
base_columns = [
    "Vehicle_ID", "Vehicle_Type", "OEM", "Vehicle_Production_Date",
    "Vehicle_Production_Year", "Vehicle_Manufacturer_Number", "OEM_Plant_Number",
    "OEM_Plant", "OEM_Postal_Code", "OEM_City", "OEM_Latitude", "OEM_Longitude",
    "Vehicle_Direct_Defect", "Vehicle_Defect_Date", "Vehicle_Defect_Mileage",
    "Any_Component_Direct_Defect", "Any_Part_Defect",
    "Any_Component_Effective_Defect", "Vehicle_Effective_Defect", "Data_Status",
]
role_columns = [
    column
    for prefix in role_prefixes
    for column in [
        f"{prefix}_Type", f"{prefix}_Supplier_Plant_Number",
        f"{prefix}_Supplier_Plant", f"{prefix}_Supplier_City",
        f"{prefix}_Direct_Defect", f"{prefix}_Part_Defect",
        f"{prefix}_Effective_Defect",
    ]
]
final_vehicle_data = final_vehicle_data[base_columns + role_columns]

display(final_vehicle_data.head())

,Vehicle_ID,Vehicle_Type,OEM,Vehicle_Production_Date,Vehicle_Production_Year,Vehicle_Manufacturer_Number,OEM_Plant_Number,OEM_Plant,OEM_Postal_Code,OEM_City,OEM_Latitude,OEM_Longitude,Vehicle_Direct_Defect,Vehicle_Defect_Date,Vehicle_Defect_Mileage,...,Transmission_Effective_Defect,Seats_Type,Seats_Supplier_Plant_Number,Seats_Supplier_Plant,Seats_Supplier_City,Seats_Direct_Defect,Seats_Part_Defect,Seats_Effective_Defect,Engine_Type,Engine_Supplier_Plant_Number,Engine_Supplier_Plant,Engine_Supplier_City,Engine_Direct_Defect,Engine_Part_Defect,Engine_Effective_Defect
0,11-1-11-1,Type 11,OEM1,2008-11-18,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,1,K2LE1,1091,1091,INGOLSTADT,1,0,1,K1BE1,1011,1011,MUENCHEN,0,1,1
1,11-1-11-2,Type 11,OEM1,2008-11-18,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,1,K2ST1,1092,1092,WUERZBURG,1,0,1,K1BE1,1011,1011,MUENCHEN,0,1,1
2,11-1-11-3,Type 11,OEM1,2008-11-19,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,1,K2ST1,1092,1092,WUERZBURG,0,0,0,K1BE1,1011,1011,MUENCHEN,0,0,0
3,11-1-11-4,Type 11,OEM1,2008-11-19,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,1,K2ST1,1092,1092,WUERZBURG,0,0,0,K1BE1,1011,1011,MUENCHEN,0,1,1
4,11-1-11-5,Type 11,OEM1,2008-11-19,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,1,K2ST1,1092,1092,WUERZBURG,0,1,1,K1BE1,1011,1011,MUENCHEN,1,0,1


## 8.4 Final Dataset Validation

The completed table is checked before export. The most important condition is that the integration has not changed the unit of analysis: every cleaned vehicle must appear exactly once. The checks also cover the expected row and column counts, duplicate column names, the four component roles, missing or invalid defect flags, non-negative mileage, the data status, and the final Bottom-up defect logic. Every statement in Section 8.6 is based on the validation table produced below.

In [32]:
expected_vehicle_counts = {
    "Type 11": 1_977_164,
    "Type 21": 512_354,
}
actual_vehicle_counts = final_vehicle_data.groupby("Vehicle_Type").size().to_dict()

defect_columns = [
    column for column in final_vehicle_data.columns if column.endswith("_Defect")
]
duplicate_columns = final_vehicle_data.columns[
    final_vehicle_data.columns.duplicated()
].tolist()
missing_defect_values = int(
    final_vehicle_data[defect_columns].isna().sum().sum()
)
invalid_defect_values = int(
    sum(
        (
            final_vehicle_data[column].notna()
            & ~final_vehicle_data[column].isin([0, 1])
        ).sum()
        for column in defect_columns
    )
)

recalculated_effective_defect = combine_defect_columns(
    final_vehicle_data,
    ["Vehicle_Direct_Defect", "Any_Component_Effective_Defect"],
)
effective_logic_matches = recalculated_effective_defect.equals(
    final_vehicle_data["Vehicle_Effective_Defect"]
)

final_validation = pd.DataFrame(
    [
        {
            "Check": "Final dataset columns",
            "Expected": 48,
            "Actual": final_vehicle_data.shape[1],
        },
        {
            "Check": "Rows equal cleaned vehicles",
            "Expected": clean_vehicle_row_count,
            "Actual": len(final_vehicle_data),
        },
        {
            "Check": "Unique Vehicle IDs",
            "Expected": len(final_vehicle_data),
            "Actual": final_vehicle_data["Vehicle_ID"].nunique(),
        },
        {
            "Check": "Vehicle Type 11 rows",
            "Expected": expected_vehicle_counts["Type 11"],
            "Actual": actual_vehicle_counts.get("Type 11", 0),
        },
        {
            "Check": "Vehicle Type 21 rows",
            "Expected": expected_vehicle_counts["Type 21"],
            "Actual": actual_vehicle_counts.get("Type 21", 0),
        },
        {
            "Check": "Duplicate column names",
            "Expected": 0,
            "Actual": len(duplicate_columns),
        },
        {
            "Check": "Missing defect values",
            "Expected": 0,
            "Actual": missing_defect_values,
        },
        {
            "Check": "Invalid defect values",
            "Expected": 0,
            "Actual": invalid_defect_values,
        },
        {
            "Check": "Negative defect mileage",
            "Expected": 0,
            "Actual": int((final_vehicle_data["Vehicle_Defect_Mileage"] < 0).sum()),
        },
        {
            "Check": "Missing component types",
            "Expected": 0,
            "Actual": int(
                final_vehicle_data[[f"{p}_Type" for p in role_prefixes]]
                .isna().sum().sum()
            ),
        },
        {
            "Check": "Vehicles with incomplete data status",
            "Expected": 0,
            "Actual": int(final_vehicle_data["Data_Status"].ne("Complete").sum()),
        },
        {
            "Check": "Effective defect logic matches",
            "Expected": True,
            "Actual": effective_logic_matches,
        },
    ]
)
final_validation["Passed"] = final_validation["Expected"] == final_validation["Actual"]

assert actual_vehicle_counts == expected_vehicle_counts
assert not final_vehicle_data["Vehicle_ID"].duplicated().any()
assert not duplicate_columns
assert final_validation["Passed"].all()
assert not any("Component_ID" in column for column in final_vehicle_data.columns)
assert not any("Part_ID" in column for column in final_vehicle_data.columns)

display(final_validation)
display(final_vehicle_data["Data_Status"].value_counts(dropna=False).to_frame("Vehicles"))

,Check,Expected,Actual,Passed
0,Final dataset columns,48,48,True
1,Rows equal cleaned vehicles,2489518,2489518,True
2,Unique Vehicle IDs,2489518,2489518,True
3,Vehicle Type 11 rows,1977164,1977164,True
4,Vehicle Type 21 rows,512354,512354,True
5,Duplicate column names,0,0,True
6,Missing defect values,0,0,True
7,Invalid defect values,0,0,True
8,Negative defect mileage,0,0,True
9,Missing component types,0,0,True


,Vehicles
Data_Status,
Complete,2489518


## 8.5 Export of the Final Dataset

The only analytical dataset exported for the submission is `SoSe26_Case_Study_finalData_Group_44.csv`. The cleaned source tables remain intermediate objects inside the notebook and are not written as additional analysis files. This avoids several large exports.

The detailed cleaning log is exported to the `Additional_files` folder as `SoSe26_Case_Study_cleaningLog_Group_44.csv`. It is a documentation file rather than a second analytical dataset and records the source file, cleaning check, and number of affected rows. Dates are exported in `YYYY-MM-DD` format and both files use UTF-8 encoding.

In [33]:
FINAL_DATA_PATH = PROJECT_ROOT / "SoSe26_Case_Study_finalData_Group_44.csv"
ADDITIONAL_FILES_DIRECTORY = PROJECT_ROOT / "Additional_files"
ADDITIONAL_FILES_DIRECTORY.mkdir(exist_ok=True)
CLEANING_LOG_PATH = (
    ADDITIONAL_FILES_DIRECTORY / "SoSe26_Case_Study_cleaningLog_Group_44.csv"
)

final_vehicle_data.to_csv(
    FINAL_DATA_PATH,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d",
    chunksize=100_000,
)

cleaning_log_table = (
    pd.DataFrame(cleaning_log, columns=["Quelldatei", "Prüfung", "Anzahl"])
    .sort_values(["Quelldatei", "Prüfung"])
    .reset_index(drop=True)
)
cleaning_log_table.to_csv(
    CLEANING_LOG_PATH,
    index=False,
    encoding="utf-8",
)
display(cleaning_log_table.head())

export_summary = pd.DataFrame(
    [
        {
            "Dataset": "Final vehicle-level analysis data",
            "Rows": len(final_vehicle_data),
            "Columns": final_vehicle_data.shape[1],
            "Output file": FINAL_DATA_PATH.name,
        },
        {
            "Dataset": "Detailed cleaning log",
            "Rows": len(cleaning_log_table),
            "Columns": cleaning_log_table.shape[1],
            "Output file": CLEANING_LOG_PATH.name,
        },
    ]
)
export_summary

,Quelldatei,Prüfung,Anzahl
0,Einzelteil_T04.csv,Negative Fehlerhaft_Fahrleistung converted to missing,1390
1,Einzelteil_T05.csv,Corrected Werksnummer from ID,596315
2,Einzelteil_T06.csv,Corrected Herstellernummer from ID,477052
3,Einzelteil_T07.txt,Corrected Werksnummer from ID,409422
4,Einzelteil_T07.txt,Negative Fehlerhaft_Fahrleistung converted to missing,52


,Dataset,Rows,Columns,Output file
0,Final vehicle-level analysis data,2489518,48,SoSe26_Case_Study_finalData_Group_44.csv
1,Detailed cleaning log,20,3,SoSe26_Case_Study_cleaningLog_Group_44.csv


## 8.6 Final Dataset Result

The validation table confirms 2,489,518 vehicle rows and 48 columns. Every `Vehicle_ID` is unique, and the expected 1,977,164 Type 11 and 512,354 Type 21 vehicles are present. It also reports zero duplicate column names, zero missing or invalid defect values, zero negative vehicle defect-mileage values, and zero missing component types. The recalculated effective defect flag matches the stored result, and no vehicle has an incomplete data status. The final vehicle-level dataset therefore passes all documented checks before export.

<a id="section-9"></a>

# 9. Decision Criteria

The audit question is answered at the **OEM vehicle plant**. The analytical question is:

> In which OEM plant is the relative share of effectively defective vehicles highest, where a vehicle is defective if the vehicle itself, an installed component, or an installed part is defective?

The ranking rules are fixed before looking at the result:

1. **Primary criterion:** effective vehicle defect rate (`Vehicle_Effective_Defect == 1`) among all produced vehicles.
2. **Secondary criterion:** absolute number of effectively defective vehicles, which represents customer impact.
3. **Definition check:** compare the direct vehicle flag with the effective Bottom-up definition. The effective definition is used for the audit decision.
4. **Fair comparison:** O11 and O12 are compared within Type 11. O21 produces Type 21, so a higher rate indicates an audit priority but does not prove a causal plant effect.
5. **Small differences:** rate differences are reported in percentage points..

Missing quality information stays unknown. It is not counted as a confirmed defect. Component types and Tier-1 supplier plants are analysed after the OEM ranking to indicate where the audit should look more closely.

The comparison is descriptive and uses all production records available in the supplied data. The results are therefore reported as vehicle counts, defect rates, and differences in percentage points.

<a id="section-10"></a>

# 10. OEM Plant Comparison

The three audit candidates were introduced in Section 3.2. O11 in Nürnberg and O12 in Bonn produce Vehicle Type 11, while O21 in Göttingen produces Vehicle Type 21. This difference must remain visible when the plant rates are interpreted.

All results in this section are calculated from the final vehicle-level dataset created in Section 8. If the notebook has just been run, the table is reused from memory. Otherwise, the single submission CSV is loaded.

In [34]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [35]:
CITY_LABELS = {
    "NUERNBERG": "Nürnberg",
    "NURNBERG": "Nürnberg",
    "BONN": "Bonn",
    "GOETTINGEN": "Göttingen",
    "GOTTINGEN": "Göttingen",
    "GÖTTINGEN": "Göttingen",
    "MUENCHEN": "München",
    "WUERZBURG": "Würzburg",
    "FUERTH": "Fürth",
}

PLANT_COLORS = {
    "O11 - Nürnberg": "#137CBD",
    "O12 - Bonn": "#F29D49",
    "O21 - Göttingen": "#1F9D8A",
}

PLANT_LINESTYLES = {
    "O11 - Nürnberg": "solid",
    "O12 - Bonn": "dash",
    "O21 - Göttingen": "dot",
}

ROLE_LABELS = {
    "Motor": "Engine",
    "Sitze": "Seats",
    "Schaltung": "Transmission",
    "Karosserie": "Body",
}
ROLE_COLORS = {
    "Engine": "#E74C3C",
    "Seats": "#7D3C98",
    "Transmission": "#1E8449",
    "Body": "#2E86AB",
}
ROLE_ORDER = ["Engine", "Seats", "Transmission", "Body"]

SOURCE_ORDER = [
    "Nur Fahrzeug", "Nur Komponente", "Nur Einzelteil",
    "Fahrzeug + Komponente", "Fahrzeug + Einzelteil",
    "Komponente + Einzelteil", "Fahrzeug + Komponente + Einzelteil",
]


def plant_label(werk, city):
    """Build a stable label such as 'O11 - Nürnberg'."""
    city_key = "" if pd.isna(city) else str(city).strip().upper()
    city_name = CITY_LABELS.get(city_key, str(city).title() if pd.notna(city) else "")
    return f"{werk} - {city_name}".strip(" -")


def is_confirmed_defect(series):
    """Return True only for confirmed defects; unknown values stay False."""
    return pd.to_numeric(series, errors="coerce").eq(1)


def defect_source(vehicle_flag, component_flag, part_flag):
    """Describe which of the three defect levels are confirmed."""
    has_vehicle = is_confirmed_defect(vehicle_flag)
    has_component = is_confirmed_defect(component_flag)
    has_part = is_confirmed_defect(part_flag)
    source = np.full(len(vehicle_flag), "Kein Fehler", dtype=object)
    source = np.where(has_vehicle & ~has_component & ~has_part, "Nur Fahrzeug", source)
    source = np.where(~has_vehicle & has_component & ~has_part, "Nur Komponente", source)
    source = np.where(~has_vehicle & ~has_component & has_part, "Nur Einzelteil", source)
    source = np.where(has_vehicle & has_component & ~has_part, "Fahrzeug + Komponente", source)
    source = np.where(has_vehicle & ~has_component & has_part, "Fahrzeug + Einzelteil", source)
    source = np.where(~has_vehicle & has_component & has_part, "Komponente + Einzelteil", source)
    source = np.where(
        has_vehicle & has_component & has_part,
        "Fahrzeug + Komponente + Einzelteil",
        source,
    )
    return pd.Series(source, index=vehicle_flag.index)


def load_final_vehicle_data():
    """Load the single submission CSV when the table is not already in memory."""
    final_path = Path.cwd().resolve() / "SoSe26_Case_Study_finalData_Group_44.csv"
    if not final_path.exists():
        raise FileNotFoundError("Run Section 8.5 to create the final dataset first.")
    return pd.read_csv(final_path, low_memory=False)


def style_figure(figure, height=430):
    """Apply a compact layout that stays readable in the notebook."""
    figure.update_layout(
        height=height,
        margin=dict(l=50, r=30, t=70, b=50),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend_title_text="",
    )
    figure.update_xaxes(showgrid=False)
    figure.update_yaxes(gridcolor="#EAF0F4", zeroline=False)
    return figure


def rate_axis(values, percent_points=False):
    """Choose a readable y-axis range for close defect rates."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    lo, hi = float(np.min(vals)), float(np.max(vals))
    span = hi - lo
    if percent_points:
        min_span, pad = 0.5, max(span * 0.2, 0.08)
        steps, tickformat = [0.1, 0.2, 0.5, 1.0, 2.0], ".1f"
    else:
        min_span, pad = 0.005, max(span * 0.2, 0.0008)
        steps, tickformat = [0.001, 0.002, 0.005, 0.01, 0.02], ".1%"
    if span < min_span:
        middle = (lo + hi) / 2
        lo, hi = middle - min_span / 2, middle + min_span / 2
    else:
        lo, hi = max(0, lo - pad), hi + pad
    dtick = next((step for step in steps if (hi - lo) / step <= 8), steps[-1])
    return dict(range=[lo, hi], dtick=dtick, tickformat=tickformat, ticks="outside")

## 10.1 Audit Candidates and Vehicle Quality Table

The first output confirms the three plants, their vehicle types, and the number of available vehicles. The analysis table itself contains one row per vehicle. The direct vehicle, component, and part flags remain visible next to the effective result. Confirmed defects are counted as `1` while unknown values are not changed into confirmed defects.

In [36]:
# Reuse the table without making a second full copy in memory.
if "final_vehicle_data" in globals():
    analysis_vehicles = final_vehicle_data
    del final_vehicle_data
elif "analysis_vehicles" not in globals():
    analysis_vehicles = load_final_vehicle_data()

# Restore the two date columns when the table was loaded from CSV.
for date_column in ["Vehicle_Production_Date", "Vehicle_Defect_Date"]:
    analysis_vehicles[date_column] = pd.to_datetime(
        analysis_vehicles[date_column], errors="coerce"
    )

analysis_vehicles["Werk_Label"] = [
    plant_label(plant, city)
    for plant, city in zip(analysis_vehicles["OEM_Plant"], analysis_vehicles["OEM_City"])
]
analysis_vehicles["Effektiver_Fehler"] = is_confirmed_defect(
    analysis_vehicles["Vehicle_Effective_Defect"]
)
analysis_vehicles["Direkter_Fahrzeugfehler"] = is_confirmed_defect(
    analysis_vehicles["Vehicle_Direct_Defect"]
)
analysis_vehicles["Fehlerquelle"] = defect_source(
    analysis_vehicles["Vehicle_Direct_Defect"],
    analysis_vehicles["Any_Component_Direct_Defect"],
    analysis_vehicles["Any_Part_Defect"],
)

candidate_overview = (
    analysis_vehicles.groupby(
        ["Werk_Label", "Vehicle_Type"],
        dropna=False,
    )
    .size()
    .rename("Vehicles")
    .reset_index()
    .rename(
        columns={
            "Werk_Label": "OEM plant",
            "Vehicle_Type": "Vehicle type",
        }
    )
    .sort_values("OEM plant")
    .reset_index(drop=True)
)
display(candidate_overview)

print(
    f"{len(analysis_vehicles):,} vehicles in the analysis table "
    f"across {analysis_vehicles['Werk_Label'].nunique()} OEM plants."
)
analysis_vehicles.head()

,OEM plant,Vehicle type,Vehicles
0,O11 - Nürnberg,Type 11,1186298
1,O12 - Bonn,Type 11,790866
2,O21 - Göttingen,Type 21,512354


2,489,518 vehicles in the analysis table across 3 OEM plants.


,Vehicle_ID,Vehicle_Type,OEM,Vehicle_Production_Date,Vehicle_Production_Year,Vehicle_Manufacturer_Number,OEM_Plant_Number,OEM_Plant,OEM_Postal_Code,OEM_City,OEM_Latitude,OEM_Longitude,Vehicle_Direct_Defect,Vehicle_Defect_Date,Vehicle_Defect_Mileage,...,Seats_Supplier_City,Seats_Direct_Defect,Seats_Part_Defect,Seats_Effective_Defect,Engine_Type,Engine_Supplier_Plant_Number,Engine_Supplier_Plant,Engine_Supplier_City,Engine_Direct_Defect,Engine_Part_Defect,Engine_Effective_Defect,Werk_Label,Effektiver_Fehler,Direkter_Fahrzeugfehler,Fehlerquelle
0,11-1-11-1,Type 11,OEM1,2008-11-18,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,INGOLSTADT,1,0,1,K1BE1,1011,1011,MUENCHEN,0,1,1,O11 - Nürnberg,True,False,Komponente + Einzelteil
1,11-1-11-2,Type 11,OEM1,2008-11-18,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,WUERZBURG,1,0,1,K1BE1,1011,1011,MUENCHEN,0,1,1,O11 - Nürnberg,True,False,Komponente + Einzelteil
2,11-1-11-3,Type 11,OEM1,2008-11-19,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,WUERZBURG,0,0,0,K1BE1,1011,1011,MUENCHEN,0,0,0,O11 - Nürnberg,True,False,Nur Einzelteil
3,11-1-11-4,Type 11,OEM1,2008-11-19,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,WUERZBURG,0,0,0,K1BE1,1011,1011,MUENCHEN,0,1,1,O11 - Nürnberg,True,False,Nur Einzelteil
4,11-1-11-5,Type 11,OEM1,2008-11-19,2008,1,11,O11,90491,NUERNBERG,49.46699,11.107353,0,NaT,0,...,WUERZBURG,0,1,1,K1BE1,1011,1011,MUENCHEN,1,0,1,O11 - Nürnberg,True,False,Komponente + Einzelteil


## 10.2 Absolute and Relative Rates by OEM Plant

The primary ranking uses the effective defect rate.


In [37]:
oem_plant_quality = (
    analysis_vehicles.groupby(
        ["Werk_Label", "OEM_Plant", "Vehicle_Type"], dropna=False
    )
    .agg(
        Fahrzeuge=("Vehicle_ID", "size"),
        Effektive_Fehler=("Effektiver_Fehler", "sum"),
        Direkte_Fahrzeugfehler=("Direkter_Fahrzeugfehler", "sum"),
        Breitengrad=("OEM_Latitude", "first"),
        Längengrad=("OEM_Longitude", "first"),
        ORT=("OEM_City", "first"),
    )
    .reset_index()
    .rename(columns={"OEM_Plant": "Werk", "Vehicle_Type": "Fahrzeugtyp"})
)

oem_plant_quality["Effektive_Quote"] = (
    oem_plant_quality["Effektive_Fehler"] / oem_plant_quality["Fahrzeuge"]
)
oem_plant_quality["Direkte_Quote"] = (
    oem_plant_quality["Direkte_Fahrzeugfehler"] / oem_plant_quality["Fahrzeuge"]
)
oem_plant_quality = oem_plant_quality.sort_values(
    ["Effektive_Quote", "Effektive_Fehler"], ascending=False
).reset_index(drop=True)

oem_plant_quality

,Werk_Label,Werk,Fahrzeugtyp,Fahrzeuge,Effektive_Fehler,Direkte_Fahrzeugfehler,Breitengrad,Längengrad,ORT,Effektive_Quote,Direkte_Quote
0,O21 - Göttingen,O21,Type 21,512354,455877,50796,51.535703,9.932804,GOETTINGEN,0.88977,0.099142
1,O11 - Nürnberg,O11,Type 11,1186298,1041102,118762,49.46699,11.107353,NUERNBERG,0.877606,0.100111
2,O12 - Bonn,O12,Type 11,790866,693991,79307,50.742015,7.120073,BONN,0.877508,0.100279


In [38]:
first_plant = oem_plant_quality.iloc[0]
second_plant = oem_plant_quality.iloc[1]
type11_plants = oem_plant_quality.loc[
    oem_plant_quality["Fahrzeugtyp"] == "Type 11"
].sort_values("Effektive_Quote", ascending=False)

effective_gap_pp = (
    first_plant["Effektive_Quote"] - second_plant["Effektive_Quote"]
) * 100

direct_ranking = oem_plant_quality.sort_values(
    "Direkte_Quote", ascending=False
).reset_index(drop=True)

if len(type11_plants) >= 2:
    type11_gap_pp = (
        type11_plants.iloc[0]["Effektive_Quote"]
        - type11_plants.iloc[1]["Effektive_Quote"]
    ) * 100
else:
    type11_gap_pp = np.nan

comparison_checks = pd.DataFrame(
    [
        {
            "Check": "Highest effective rate",
            "Result": first_plant["Werk_Label"],
        },
        {
            "Check": "Highest direct vehicle rate",
            "Result": direct_ranking.iloc[0]["Werk_Label"],
        },
        {
            "Check": "Definition test changes the ranking",
            "Result": first_plant["Werk_Label"] != direct_ranking.iloc[0]["Werk_Label"],
        },
        {
            "Check": f"Effective gap {first_plant['Werk_Label']} vs {second_plant['Werk_Label']}",
            "Result": f"{effective_gap_pp:.3f} percentage points",
        },
        {
            "Check": "Type 11 effective gap O11 vs O12",
            "Result": f"{type11_gap_pp:.3f} percentage points",
        },
    ]
)
comparison_checks


,Check,Result
0,Highest effective rate,O21 - Göttingen
1,Highest direct vehicle rate,O12 - Bonn
2,Definition test changes the ranking,True
3,Effective gap O21 - Göttingen vs O11 - Nürnberg,1.216 percentage points
4,Type 11 effective gap O11 vs O12,0.010 percentage points


The first chart places the two decision criteria next to each other. The left side shows the absolute number of effectively defective vehicles and therefore the customer volume. The right side shows the effective defect rate and therefore the relative process result used for the audit ranking.

In [39]:
plant_colors = [
    PLANT_COLORS.get(label, "#137CBD") for label in oem_plant_quality["Werk_Label"]
]

overview_figure = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Defective vehicles", "Effective defect rate"),
    horizontal_spacing=0.16,
)
overview_figure.add_trace(
    go.Bar(
        x=oem_plant_quality["Werk_Label"],
        y=oem_plant_quality["Effektive_Fehler"],
        marker_color=plant_colors,
        text=oem_plant_quality["Effektive_Fehler"].map(lambda value: f"{value:,.0f}"),
        textposition="outside",
        customdata=oem_plant_quality[["Fahrzeuge", "Fahrzeugtyp"]],
        hovertemplate=(
            "<b>%{x}</b><br>Defective: %{y:,.0f}"
            "<br>Vehicles: %{customdata[0]:,.0f}"
            "<br>Type: %{customdata[1]}<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)
overview_figure.add_trace(
    go.Bar(
        x=oem_plant_quality["Werk_Label"],
        y=oem_plant_quality["Effektive_Quote"] * 100,
        marker_color=plant_colors,
        text=(oem_plant_quality["Effektive_Quote"] * 100).map(
            lambda value: f"{value:.2f} %"
        ),
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>Effective rate: %{y:.3f} %<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=2,
)
overview_figure.update_yaxes(title_text="Vehicles", row=1, col=1)
overview_figure.update_yaxes(
    title_text="Defect rate [%]",
    row=1,
    col=2,
    **rate_axis(oem_plant_quality["Effektive_Quote"] * 100, percent_points=True),
)
overview_figure.update_layout(title="Defective vehicles and effective rate by OEM plant")
style_figure(overview_figure)


### Result

O21 Göttingen has the highest effective rate at 88.98%, followed by O11 Nürnberg at 87.76% and O12 Bonn at 87.75%. The gap between O21 and O11 is 1.216 percentage points. O11 and O12 differ by only 0.010 percentage points and are practically very similar. Nürnberg still has the largest absolute number with 1,041,102 effectively defective vehicles because it also has the highest production volume. The relative rate points to O21 as the first audit location, while the absolute result shows that O11 still affects more vehicles.

### Comparison with the Direct Vehicle Flag

The next chart compares the direct vehicle flag with the complete Bottom-up definition. Light bars use only the direct vehicle flag. Dark bars classify a vehicle as defective when the vehicle itself, an installed component, or an installed part is defective.

In [40]:
definition_long = oem_plant_quality.melt(
    id_vars=["Werk_Label"],
    value_vars=["Direkte_Quote", "Effektive_Quote"],
    var_name="Definition",
    value_name="Quote",
)
definition_long["Definition"] = definition_long["Definition"].map(
    {
        "Direkte_Quote": "Vehicle flag only",
        "Effektive_Quote": "Vehicle, component, or part",
    }
)
definition_long["Quote_pct"] = definition_long["Quote"] * 100

definition_figure = px.bar(
    definition_long,
    x="Werk_Label",
    y="Quote_pct",
    color="Definition",
    barmode="group",
    color_discrete_sequence=["#A8D5F2", "#137CBD"],
    title="Defect rate by definition",
    labels={"Werk_Label": "OEM plant", "Quote_pct": "Defect rate [%]"},
)
definition_figure.update_traces(
    hovertemplate="<b>%{x}</b><br>%{fullData.name}: %{y:.3f} %<extra></extra>"
)
style_figure(definition_figure)


### Result

The direct vehicle rate is highest at O12 with 10.03%, followed by O11 with 10.01% and O21 with 9.91%. After component and part defects are included, O21 moves to the first position. The chosen defect definition therefore changes the plant ranking. Looking only at the vehicle file would lead to a different audit recommendation.

## 10.3 Defect Source at Vehicle Level

The following chart shows which of the three quality levels (vehicle, component, part), contribute to vehicles classified as effectively defective. The categories also include combinations because more than one defect source can occur in the same vehicle.


In [41]:
source_counts = (
    analysis_vehicles.loc[analysis_vehicles["Fehlerquelle"] != "Kein Fehler"]
    .groupby(["Werk_Label", "Fehlerquelle"], dropna=False)
    .size()
    .rename("Fahrzeuge")
    .reset_index()
)
source_counts["Anteil"] = source_counts["Fahrzeuge"] / source_counts.groupby(
    "Werk_Label"
)["Fahrzeuge"].transform("sum")

source_figure = px.bar(
    source_counts,
    x="Werk_Label",
    y="Anteil",
    color="Fehlerquelle",
    category_orders={"Fehlerquelle": SOURCE_ORDER},
    title="Defect source of defective vehicles",
    labels={
        "Werk_Label": "OEM plant",
        "Anteil": "Share of defective vehicles",
        "Fehlerquelle": "Defect source",
    },
    color_discrete_sequence=[
        "#A8D5F2",
        "#67B4DF",
        "#137CBD",
        "#8FD5CA",
        "#50B8A6",
        "#F6BE83",
        "#D95C59",
    ],
)
source_figure.update_traces(
    hovertemplate="%{fullData.name}<br>Share: %{y:.1%}<extra></extra>"
)
source_figure.update_yaxes(tickformat=".0%")
style_figure(source_figure, height=470)


### Result

Part defects explain a large share of the effective vehicle results. A part defect without a direct vehicle or component defect accounts for about 49% of defective vehicles at O11 and O12 and about 50% at O21. A further 31% contain both a component and a part defect. In comparison, a direct vehicle defect without a component or part defect accounts for only about 1% to 2%. This supports including the component and supplier chain in the audit instead of examining final assembly alone.

## 10.4 Defect Rates over Time

The dropdown compares the effective vehicle rate with the effective defect rate of Engine, Seats, Transmission, and Body by OEM plant and production year. The component rates are calculated directly from the four role-specific columns in the final dataset.

This view checks whether the result is driven by a single production year and shows which component role contributes repeatedly over time.

In [42]:
yearly_quality = (
    analysis_vehicles.dropna(subset=["Vehicle_Production_Year"])
    .groupby(["Vehicle_Production_Year", "Werk_Label", "Vehicle_Type"], dropna=False)
    .agg(
        Fahrzeuge=("Vehicle_ID", "size"),
        Effektive_Fehler=("Effektiver_Fehler", "sum"),
    )
    .reset_index()
    .rename(
        columns={
            "Vehicle_Production_Year": "Produktionsjahr",
            "Vehicle_Type": "Fahrzeugtyp",
        }
    )
)
yearly_quality["Effektive_Quote"] = (
    yearly_quality["Effektive_Fehler"] / yearly_quality["Fahrzeuge"]
)

# Each vehicle has one installed component per role, so the same final table
# can be grouped directly for the component time series.
yearly_role_frames = []
for role in ROLE_ORDER:
    role_data = analysis_vehicles.dropna(subset=["Vehicle_Production_Year"])[
        ["Vehicle_Production_Year", "Werk_Label", f"{role}_Effective_Defect"]
    ].copy()
    role_data["Ist_Fehlerhaft"] = is_confirmed_defect(
        role_data[f"{role}_Effective_Defect"]
    )
    role_summary = (
        role_data.groupby(["Vehicle_Production_Year", "Werk_Label"], dropna=False)
        .agg(Anzahl=("Ist_Fehlerhaft", "size"), Fehler=("Ist_Fehlerhaft", "sum"))
        .reset_index()
        .rename(columns={"Vehicle_Production_Year": "Produktionsjahr"})
    )
    role_summary["Role"] = role
    role_summary["Quote"] = role_summary["Fehler"] / role_summary["Anzahl"]
    yearly_role_frames.append(role_summary)

yearly_role = pd.concat(yearly_role_frames, ignore_index=True)
plant_order = [plant for plant in PLANT_COLORS if plant in set(yearly_quality["Werk_Label"])]

trend_figure = go.Figure()
trace_views = []

for plant in plant_order:
    sub = yearly_quality.loc[yearly_quality["Werk_Label"] == plant].sort_values(
        "Produktionsjahr"
    )
    trend_figure.add_trace(
        go.Scatter(
            x=sub["Produktionsjahr"],
            y=sub["Effektive_Quote"],
            mode="lines+markers",
            name=f"{plant} – Overall",
            line=dict(color=PLANT_COLORS[plant], width=3),
            marker=dict(size=8, color=PLANT_COLORS[plant]),
            customdata=np.stack(
                [sub["Fahrzeuge"], sub["Effektive_Fehler"], sub["Fahrzeugtyp"]], axis=1
            ),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>Year: %{x:.0f}<br>Rate: %{y:.1%}"
                "<br>Defective: %{customdata[1]:,.0f}"
                "<br>Vehicles: %{customdata[0]:,.0f}<extra></extra>"
            ),
            visible=True,
        )
    )
    trace_views.append("Overall")

for role in ROLE_ORDER:
    for plant in plant_order:
        sub = yearly_role.loc[
            (yearly_role["Role"] == role) & (yearly_role["Werk_Label"] == plant)
        ].sort_values("Produktionsjahr")
        if sub.empty:
            continue
        trend_figure.add_trace(
            go.Scatter(
                x=sub["Produktionsjahr"],
                y=sub["Quote"],
                mode="lines+markers",
                name=f"{plant} – {role}",
                line=dict(
                    color=ROLE_COLORS[role],
                    width=2,
                    dash=PLANT_LINESTYLES.get(plant, "solid"),
                ),
                marker=dict(size=7, color=ROLE_COLORS[role]),
                customdata=np.stack([sub["Anzahl"], sub["Fehler"]], axis=1),
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>Year: %{x:.0f}<br>Rate: %{y:.1%}"
                    "<br>Defective: %{customdata[1]:,.0f}"
                    "<br>Components: %{customdata[0]:,.0f}<extra></extra>"
                ),
                visible=False,
            )
        )
        trace_views.append(role)

view_titles = {
    "Overall": "Effective vehicle defect rate by production year",
    "Engine": "Engine defect rate by production year",
    "Seats": "Seat defect rate by production year",
    "Transmission": "Transmission defect rate by production year",
    "Body": "Body defect rate by production year",
    "All": "Vehicle and component-role defect rates by production year",
}


def values_for_view(view):
    if view == "Overall":
        return yearly_quality["Effektive_Quote"]
    if view == "All":
        return pd.concat(
            [yearly_quality["Effektive_Quote"], yearly_role["Quote"]], ignore_index=True
        )
    return yearly_role.loc[yearly_role["Role"] == view, "Quote"]


buttons = []
for view in ["Overall", "Engine", "Seats", "Transmission", "Body", "All"]:
    visible = [True] * len(trace_views) if view == "All" else [
        tag == view for tag in trace_views
    ]
    yaxis = {"title": {"text": "Defect rate"}}
    yaxis.update(rate_axis(values_for_view(view)))
    buttons.append(
        dict(
            label=view,
            method="update",
            args=[{"visible": visible}, {"title": {"text": view_titles[view]}, "yaxis": yaxis}],
        )
    )

overall_yaxis = {"title": {"text": "Defect rate"}}
overall_yaxis.update(rate_axis(values_for_view("Overall")))
trend_figure.update_layout(
    title=view_titles["Overall"],
    yaxis=overall_yaxis,
    updatemenus=[
        dict(type="dropdown", buttons=buttons, x=1, xanchor="right", y=1.16, yanchor="top")
    ],
)
trend_figure.update_xaxes(title="Production year", dtick=1)
style_figure(trend_figure, height=500)
trend_figure.update_layout(margin=dict(l=50, r=30, t=100, b=50))
trend_figure

### Result

O21 has the highest effective vehicle rate in every observed production year. Its yearly rate stays between 88.67% and 89.38%, while the rates of O11 and O12 remain around 87.4% to 88.1%. The ranking is therefore not caused by one unusual production year. The O21 Engine rate remains close to 51%, while its Body rate stays near 41%. The Seats rate decreases from 37.5% in 2008 to about 34% in the later years.

The year 2008 contains considerably fewer vehicles than the later years, especially for O21. Small changes in that year should therefore not be interpreted as a separate long-term trend.

<a id="section-11"></a>

# 11. Component Defects at Supplier Level

The OEM ranking identifies where to start the audit. The role-specific columns in the final dataset are then used to examine which installed component types have high defect rates and at which Tier-1 plants they were produced.

The effective component flag is used because it follows the task definition: a component is effectively defective if the component itself or at least one of its installed parts is defective. Individual part IDs and part types are outside the final reporting level.

In [43]:
component_quality_frames = []

for role in ROLE_ORDER:
    role_data = analysis_vehicles[
        [
            f"{role}_Type", f"{role}_Supplier_Plant", f"{role}_Supplier_City",
            f"{role}_Direct_Defect", f"{role}_Part_Defect",
            f"{role}_Effective_Defect",
        ]
    ].copy()
    role_data["Direct_Is_Defective"] = is_confirmed_defect(
        role_data[f"{role}_Direct_Defect"]
    )
    role_data["Part_Is_Defective"] = is_confirmed_defect(
        role_data[f"{role}_Part_Defect"]
    )
    role_data["Effective_Is_Defective"] = is_confirmed_defect(
        role_data[f"{role}_Effective_Defect"]
    )

    role_summary = (
        role_data.groupby(
            [f"{role}_Type", f"{role}_Supplier_Plant", f"{role}_Supplier_City"],
            dropna=False,
        )
        .agg(
            Anzahl=("Effective_Is_Defective", "size"),
            Direkte_Fehler=("Direct_Is_Defective", "sum"),
            Part_Fehler=("Part_Is_Defective", "sum"),
            Effektive_Fehler=("Effective_Is_Defective", "sum"),
        )
        .reset_index()
        .rename(
            columns={
                f"{role}_Type": "Komponententyp",
                f"{role}_Supplier_Plant": "Werk",
                f"{role}_Supplier_City": "ORT",
            }
        )
    )
    role_summary["Komponentenrolle"] = role
    component_quality_frames.append(role_summary)

component_quality = pd.concat(component_quality_frames, ignore_index=True)
component_quality["Fehler"] = component_quality["Effektive_Fehler"]
component_quality["Fehlerquote"] = component_quality["Fehler"] / component_quality["Anzahl"]
component_quality["Quote_pct"] = component_quality["Fehlerquote"] * 100
component_quality["Werk_Label"] = [
    plant_label(plant, city)
    for plant, city in zip(component_quality["Werk"], component_quality["ORT"])
]

print(f"{len(component_quality)} component plant-type rows from the final dataset.")
component_quality

32 component plant-type rows from the final dataset.


,Komponententyp,Werk,ORT,Anzahl,Direkte_Fehler,Part_Fehler,Effektive_Fehler,Komponentenrolle,Fehler,Fehlerquote,Quote_pct,Werk_Label
0,K1BE1,1011,MUENCHEN,296624,52237,127378,157060,Engine,157060,0.529492,52.949188,1011 - München
1,K1BE1,1021,LEIPZIG,98902,17355,42632,52501,Engine,52501,0.530839,53.083861,1021 - Leipzig
2,K1BE1,1041,BERLIN,592894,103720,255729,314711,Engine,314711,0.530805,53.080483,1041 - Berlin
3,K1BE2,1011,MUENCHEN,53575,9588,23098,28511,Engine,28511,0.53217,53.216986,1011 - München
4,K1BE2,1041,BERLIN,215701,37727,92811,114443,Engine,114443,0.530563,53.056314,1041 - Berlin
5,K1DI1,1021,LEIPZIG,198257,19970,85240,96679,Engine,96679,0.487645,48.764482,1021 - Leipzig
6,K1DI1,1031,DRESDEN,395280,39626,170004,192423,Engine,192423,0.486802,48.680176,1031 - Dresden
7,K1DI1,1041,BERLIN,395207,39718,170584,192992,Engine,192992,0.488331,48.833143,1041 - Berlin
8,K1DI2,1021,LEIPZIG,143639,14343,62157,70235,Engine,70235,0.488969,48.896887,1021 - Leipzig
9,K1DI2,1031,DRESDEN,99439,10010,42809,48471,Engine,48471,0.487445,48.744456,1031 - Dresden


## 11.1 Component Types

The first chart pools supplier plants and compares the effective defect rate of each component type. Colour identifies the role. This shows whether Engine, Seats, Transmission, or Body variants contribute differently to the final vehicle result.

In [44]:
component_by_type = (
    component_quality.groupby(["Komponententyp", "Komponentenrolle"], dropna=False)
    .agg(Anzahl=("Anzahl", "sum"), Fehler=("Fehler", "sum"))
    .reset_index()
)
component_by_type["Fehlerquote"] = (
    component_by_type["Fehler"] / component_by_type["Anzahl"]
)
component_by_type["Quote_pct"] = component_by_type["Fehlerquote"] * 100
# The role names are already stored in English in component_quality.
component_by_type["Role"] = component_by_type["Komponentenrolle"]
component_by_type = component_by_type.sort_values(
    "Fehlerquote", ascending=False
).reset_index(drop=True)

component_type_figure = px.bar(
    component_by_type,
    x="Komponententyp",
    y="Quote_pct",
    color="Role",
    color_discrete_map=ROLE_COLORS,
    category_orders={"Role": ROLE_ORDER},
    custom_data=["Anzahl", "Fehler"],
    title="Component defect rate by type",
    labels={
        "Komponententyp": "Component type",
        "Quote_pct": "Defect rate [%]",
        "Role": "Role",
    },
)
component_type_figure.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>Rate: %{y:.2f} %"
        "<br>Items: %{customdata[0]:,.0f}"
        "<br>Defects: %{customdata[1]:,.0f}<extra></extra>"
    )
)
style_figure(component_type_figure)
component_type_figure.update_yaxes(
    title="Defect rate [%]",
    **rate_axis(component_by_type["Quote_pct"], percent_points=True),
)
component_type_figure.show()
component_by_type


,Komponententyp,Komponentenrolle,Anzahl,Fehler,Fehlerquote,Quote_pct,Role
0,K1BE2,Engine,269276,142954,0.530883,53.088281,Engine
1,K1BE1,Engine,988420,524272,0.530414,53.04142,Engine
2,K1DI2,Engine,243078,118706,0.488345,48.83453,Engine
3,K1DI1,Engine,988744,482094,0.487582,48.758223,Engine
4,K6,Body,512354,209961,0.409797,40.979674,Body
5,K2LE1,Seats,395897,152173,0.384375,38.437523,Seats
6,K2LE2,Seats,102811,39515,0.384346,38.434603,Seats
7,K3AG1,Transmission,395440,144910,0.366453,36.645256,Transmission
8,K2ST2,Seats,409543,141293,0.345002,34.500162,Seats
9,K4,Body,1977164,679721,0.343786,34.378585,Body


### Result

The engine variants have the highest effective component rates. `K1BE1` and `K1BE2` are both close to 53%, while `K1DI1` and `K1DI2` are close to 49%. The Type-21 body `K6` follows at about 41%. Most seat, transmission, and `K4` body variants are between 34% and 38%. This makes the engine variants the first component-level topic for the audit.

## 11.2 Component Rates by Plant and Type

The next chart splits the same effective component rates by component type and producing Tier-1 plant. This makes the supplier follow-up more specific than a general average by plant.

In [45]:
component_plant_figure = px.bar(
    component_quality.sort_values(["Komponententyp", "Werk_Label"]),
    x="Komponententyp",
    y="Quote_pct",
    color="Werk_Label",
    barmode="group",
    custom_data=["Anzahl", "Fehler"],
    title="Component defect rate by type and producing plant",
    labels={
        "Komponententyp": "Component type",
        "Quote_pct": "Defect rate [%]",
        "Werk_Label": "Tier-1 plant",
    },
)
component_plant_figure.update_traces(
    hovertemplate=(
        "<b>%{fullData.name}</b><br>%{x}: %{y:.2f} %"
        "<br>Items: %{customdata[0]:,.0f}"
        "<br>Defects: %{customdata[1]:,.0f}<extra></extra>"
    )
)
style_figure(component_plant_figure, height=500)
component_plant_figure.update_yaxes(
    title="Defect rate [%]",
    **rate_axis(component_quality["Quote_pct"], percent_points=True),
)
component_plant_figure.show()

component_quality.sort_values("Fehlerquote", ascending=False)[
    ["Werk_Label", "Komponententyp", "Komponentenrolle", "Anzahl", "Fehler", "Fehlerquote"]
].reset_index(drop=True)


,Werk_Label,Komponententyp,Komponentenrolle,Anzahl,Fehler,Fehlerquote
0,1011 - München,K1BE2,Engine,53575,28511,0.53217
1,1021 - Leipzig,K1BE1,Engine,98902,52501,0.530839
2,1041 - Berlin,K1BE1,Engine,592894,314711,0.530805
3,1041 - Berlin,K1BE2,Engine,215701,114443,0.530563
4,1011 - München,K1BE1,Engine,296624,157060,0.529492
5,1021 - Leipzig,K1DI2,Engine,143639,70235,0.488969
6,1041 - Berlin,K1DI1,Engine,395207,192992,0.488331
7,1021 - Leipzig,K1DI1,Engine,198257,96679,0.487645
8,1031 - Dresden,K1DI2,Engine,99439,48471,0.487445
9,1031 - Dresden,K1DI1,Engine,395280,192423,0.486802


### Result

Most plants producing the same component type have similar rates, but two differences stand out. For `K2LE1` and `K2LE2`, Dortmund reaches about 41.6% to 41.7%, compared with about 33.5% to 33.7% in Ingolstadt. For `K3AG1`, Schweinfurt has 41.92%, while Hannover and Trunkelsberg are close to 34.4%.

For the O21 follow-up, `K1BE2` is highest in München at 53.22% and Berlin at 53.06%. `K1DI2` reaches 48.90% in Leipzig and 48.74% in Dresden. The Type-21 body `K6` is similar in Freital and Landshut at about 41%. The largest plant difference within a Type-21 component is therefore the `K2LE2` seat from Dortmund compared with Ingolstadt.

## 11.3 Direct and Part-Related Component Defects

The final comparison separates direct component defects from cases where at least one installed part is defective. The two rates can overlap for the same component, so they should be read as defect sources rather than as parts of a total. The effective rate is shown as the combined result.

In [46]:
component_source_by_type = (
    component_quality.groupby(["Komponententyp", "Komponentenrolle"], dropna=False)
    .agg(
        Anzahl=("Anzahl", "sum"),
        Direkte_Fehler=("Direkte_Fehler", "sum"),
        Part_Fehler=("Part_Fehler", "sum"),
        Effektive_Fehler=("Effektive_Fehler", "sum"),
    )
    .reset_index()
)

for count_column, rate_column in [
    ("Direkte_Fehler", "Direct component"),
    ("Part_Fehler", "Installed part"),
    ("Effektive_Fehler", "Effective component"),
]:
    component_source_by_type[rate_column] = (
        component_source_by_type[count_column] / component_source_by_type["Anzahl"] * 100
    )

component_source_long = component_source_by_type.melt(
    id_vars=["Komponententyp", "Komponentenrolle", "Anzahl"],
    value_vars=["Direct component", "Installed part", "Effective component"],
    var_name="Defect source",
    value_name="Defect rate [%]",
)

component_source_figure = px.bar(
    component_source_long,
    x="Komponententyp",
    y="Defect rate [%]",
    color="Defect source",
    barmode="group",
    title="Direct, part-related, and effective defect rates by component type",
    labels={"Komponententyp": "Component type"},
    color_discrete_sequence=["#A8D5F2", "#F6BE83", "#137CBD"],
)
component_source_figure.update_traces(
    hovertemplate="<b>%{x}</b><br>%{fullData.name}: %{y:.2f} %<extra></extra>"
)
style_figure(component_source_figure, height=500)
component_source_figure.show()
component_source_by_type

,Komponententyp,Komponentenrolle,Anzahl,Direkte_Fehler,Part_Fehler,Effektive_Fehler,Direct component,Installed part,Effective component
0,K1BE1,Engine,988420,173312,425739,524272,17.534247,43.072682,53.04142
1,K1BE2,Engine,269276,47315,115909,142954,17.571191,43.044683,53.088281
2,K1DI1,Engine,988744,99314,425828,482094,10.04446,43.067569,48.758223
3,K1DI2,Engine,243078,24353,104966,118706,10.018595,43.182024,48.83453
4,K2LE1,Seats,395897,61521,107363,152173,15.539648,27.118922,38.437523
5,K2LE2,Seats,102811,16052,27776,39515,15.613115,27.016564,38.434603
6,K2ST1,Seats,1581267,158106,428244,543562,9.998691,27.082333,34.375093
7,K2ST2,Seats,409543,41199,111265,141293,10.05975,27.168087,34.500162
8,K3AG1,Transmission,395440,51280,107444,144910,12.967833,27.170747,36.645256
9,K3AG2,Transmission,102654,10166,27646,35098,9.90317,26.931245,34.190582


### Result

Part-related defects make up an important part of the effective component rates. About 43% of the engine observations have at least one defective installed part. The direct engine rate is about 17.5% for the petrol variants and about 10% for the diesel variants. For `K6`, the part-related rate is 34.4%, compared with a direct component rate of 10.0%. Because both flags can occur for the same component, these percentages are not added together. The result still shows that the audit should include the supplied parts and not only the direct component process.

<a id="section-12"></a>

# 12. Audit Recommendation

The next process audit should take place at **O21 Göttingen**. O21 has the highest effective vehicle defect rate at 88.98%, compared with 87.76% at O11 Nürnberg and 87.75% at O12 Bonn. O11 has more defective vehicles in absolute terms, but the relative process rate is the primary criterion defined in Section 9. The O21 result is also visible in every production year and is not caused by one unusual year.

The recommendation depends on the complete Bottom-up defect definition. If only the direct vehicle flag were used, O12 would have the highest rate. In the complete result, around half of the defective vehicles are classified through a part defect without a direct vehicle or component defect. This is why the O21 audit should include the component and supplier chain.

For Type 21, the first technical focus should be the engine variants `K1BE2` and `K1DI2`, with effective rates of 53.09% and 48.83%. Their part-related rates are close to 43%. The `K2LE2` seat from Dortmund is another relevant finding because its rate is 41.71%, compared with 33.52% for the same type from Ingolstadt. The `K6` body should also be considered, although Freital and Landshut are very similar at about 41%. These results suggest starting with the engine supply chain and then examining the Dortmund seat supplier and the K6 body process.

## 12.1 Limitations

- O21 produces only Type 21. The data therefore cannot separate a vehicle-type effect from an OEM plant effect..
- The Bottom-up rule classifies a vehicle as defective when any lower-level defect is confirmed. This explains the high effective rates and makes the result dependent on the chosen defect definition.
- Part information is aggregated before the final dataset is created. The analysis can identify a part-related component defect, but it cannot identify one specific part or Tier-2 plant as the root cause.

The results therefore identify O21 as the best place to start the audit, but they do not prove that assembly in Göttingen caused the defects. A technical root-cause analysis is still required during the audit.